# Sparse Poisson Deconvolution with Block-Coordinate DCA: Block Size and Geometry

> **Relation to the paper.** This notebook is the main-text protocol of Section 5.2 and Appendix A.4 of *Non-Euclidean
> Coordinate DC Programming*: it produces Figure 2, Table 1 and the "Inner solver, main text" rows of Table 3 (the
> `tikz_figures/` cell writes the pgfplots sources and the support table). All seven DCA methods run Algorithm 1 with the
> backtracked inner steps (19)–(20) and the $\varepsilon_k$ rule (21) with $\rho=0.1$: a block solve stops as soon as a
> computable upper bound $U(z)\ge Q_k(z)-\min Q_k$ on its error is at most $\varepsilon_k=\rho\max\{\Delta_k,\,Q_k(x^k)-Q_k(z)\}$,
> where $\Delta_k$ is the Burg gap contribution of the selected block at $x^k$ and $Q_k(x^k)-Q_k(z)$ the decrease achieved so
> far, so that $\varphi(x^k)-\varphi(x^{k+1})\ge(1-\rho)\Delta_k$ (Lemma A.1). The bound is a convex duality certificate
> formed from the gradient of the new point (Poisson box dual; derivative bracket for a coordinate), so condition (2) of the
> paper is certified by the method itself and no reference solver takes part in any decision. An independent reference
> minimiser (`brentq` for a coordinate, L-BFGS-B for the full vector) is run afterwards for reporting only (uncharged
> ledger category `reference`, excluded from the clock) to show the true error next to the certified one. The safe-step protocol of
> Figure 4 and Table 2 (global constants $L_i$, no backtracking, $t_k=30+\lfloor k/N\rfloor$ steps per subproblem, 30 seeds) is
> the same code with `backtracking_geometries=()`, `inner_relative_tolerance=None` and `seeds=range(105, 135)`.
>
> **Notation.** The notebook writes $F$, $Q_k$, $\mu$, $\operatorname{top}_Q$, $L_j$, $D_B$, $\Delta_k$ and "Burg gap" for
> the paper's $\varphi$ in (7) (the KL form differs from the Poisson negative log-likelihood by the constant
> $\sum_jy_j\log y_j-y_j$), the block surrogate $\hat\varphi_{i}(\cdot,x^k)$ restricted to the selected block (up to a
> constant), the weight $\lambda$ of the trimmed penalty ($\mu=\lambda$ throughout, as in (7)), $\Vert\cdot\Vert_{(Q)}$, the
> constants $L_i$ of Lemma A.2, $D_{\psi_i}$, the block contribution $\Delta^k_{i_k}$ of Theorem 4.2 and $\operatorname{Gap}_\Psi$
> of Definition 4.1; the "Euclidean gap" is the projected-gradient residual that stops the E methods (Appendix A.4, *Cost
> accounting and stopping*), and the backtracked weights $W$ are the $W_s$ of (19).

This notebook solves the sparse Poisson PSF deconvolution problem (Section 5.2 of the paper)

$$\min_{x\in[x_{\min},x_{\max}]^n} F(x)=\operatorname{KL}(y\,\Vert\,Ax+b)+\lambda\sum_j x_j-\mu\operatorname{top}_Q(x)$$

with the DC split $f=\operatorname{KL}(y\Vert Ax+b)$, $g=\lambda\sum_jx_j+\iota_{\rm box}$, $h=\mu\operatorname{top}_Q$.
Every DCA method runs Algorithm 1: pick $v^k\in\partial h(x^k)$, pick a block, and approximately minimise the convex block
surrogate $Q_k(z)=\operatorname{KL}(y\Vert r+Bz)+(\lambda\mathbf 1-v^k_I)^\top z$.

**Aim.** The experiment is built to show two effects, each isolated by a pair of methods that differ in one ingredient:

1. *Coordinate blocks versus the full vector.* `B-GS(Burg)` and `B-Full` take the same backtracked Burg step and solve their
   subproblems to the same relative accuracy; the first updates one Gauss–Southwell-selected coordinate per iteration, the
   second the whole vector.
   `E-GS(PG)` and `E-Full` are the Euclidean pair.
2. *Burg versus Euclidean geometry.* `B-GS(Burg)` against `E-GS(PG)`, and `B-Full` against `E-Full`: same block rule,
   different inner geometry (each Gauss–Southwell variant orders its batch by its own geometry's score).

`B-GS(Burg)` and `B-Full` are the methods of interest and are drawn with thick lines; random coordinates, `B-GS(PG)` and the
convex FISTA reference are kept for context.

**Inner steps.** Each inner step minimises an upper model of $Q_k$ in the method's geometry (Burg or Euclidean) whose weights
are found by standard backtracking (see *Inner Solver*); every accepted step is a descent step because the exact
remainder is checked against the model. The *safe constants* below remain the weights of the stationarity measures and
of the Gauss–Southwell scores, the reference of the $\varepsilon_k$ rule, and the step weights when backtracking is
switched off:

* **Burg** ($\psi=-\log$): the block relative-smoothness constant $L_j=\sum_{i:A_{ij}>0}y_i$ (Definition 3.1). The same $L_j$
  are jointly valid for the full vector (the kernel $\Psi=\sum_jL_j\psi_j$ is separable), so coordinate and full Burg steps use
  the same weights.
* **Euclidean:** the largest curvature of the block on the box. For a coordinate block with its frozen offset $r$,
  $W_j=\sum_iy_iA_{ij}^2/(r_i+A_{ij}x_{\min})^2$ (recomputed per block, one column). For the full vector, the Hessian row-sum
  bound $A^\top\big(y\odot A\mathbf 1/(b+x_{\min}A\mathbf 1)^2\big)$ (setup, two passes), which must hold with every pixel at
  $x_{\min}$ at once.

**Inner accuracy ($\varepsilon_k$ rule).** Algorithm 1 needs an $\varepsilon_k$-approximate minimiser of the block
surrogate, $Q_k(x^{k+1}_{I_k})-\min Q_k\le\varepsilon_k$. Here this is certified, not scheduled: after every inner step the
gradient of the new point $z$ is formed (the product the next step needs anyway) and gives a computable upper bound
$U(z)\ge Q_k(z)-\min Q_k$, namely $Q_k(z)$ minus the best lower bound on $\min Q_k$ seen in the solve, each lower bound being a
convex duality certificate (the Poisson box dual with the minimiser bound $U_j$ for every block and, for a coordinate, the
derivative bracket $|Q_k'(z)|\cdot$ width of the interval that must contain the minimiser; see *Frozen Target*). The solve stops as soon as
$$U(z)\;\le\;\varepsilon_k:=\rho\,\max\{\Delta_k,\;Q_k(x^k)-Q_k(z)\},\qquad \rho=\texttt{inner\_relative\_tolerance}=0.1 ,$$
which implies $Q_k(z)-\min Q_k\le\varepsilon_k$ outright.
Here $\Delta_k=\max_t\{(d-v^k)_{I_k}^\top(x^k-t)-\sum_{j\in I_k}L_jD_B(t_j,x^k_j)\}$ is the Burg gap contribution of the selected
block (Definition 4.1 with the constants $L_j$) and $Q_k(x^k)-Q_k(z)$ the decrease achieved so far; by relative smoothness both
are lower bounds on the available decrease $Q_k(x^k)-\min Q_k$, and neither depends on the inner geometry, so the same rule
applies to every method. The rule is what the proofs need: writing $D_k=Q_k(x^k)-\min Q_k$ and $e_k=Q_k(x^{k+1})-\min Q_k$,
the accepted decrease is $Q_k(x^k)-Q_k(x^{k+1})=D_k-e_k\ge D_k-\rho\max\{\Delta_k,D_k-e_k\}$, so either
$D_k-e_k\ge D_k-\rho\Delta_k\ge(1-\rho)\Delta_k$ or $D_k-e_k\ge D_k/(1+\rho)\ge(1-\rho)\Delta_k$; hence
$F(x^k)-F(x^{k+1})\ge Q_k(x^k)-Q_k(x^{k+1})\ge(1-\rho)\Delta_k$ and Theorems 4.1 and 4.2 hold with $n$ replaced by
$n/(1-\rho)$ and without the $\sum_k\varepsilon_k$ term (an optional absolute floor `inner_absolute_tolerance` is 0 here). A
solve also ends when no step can change the block: a block at a bound with its gradient pointing out (error 0, no step), or a
step that changes $Q_k$ by less than its floating-point resolution $8\,\epsilon(1+|Q_k(x^k)|)$ (the point is the block
minimiser to working precision, but the certificate cannot fall below roundoff; this is where $\rho\Delta_k$ is below
resolution, e.g. a coordinate at $x_{\min}$ whose Burg contribution is tiny; such solves are reported separately as
uncertified). `max_inner_budget` ($10^3$ steps) is only a safeguard. The certified error $U$ of every accepted solve and its
target are recorded and summarised in *Inner accuracy* below, next to the true error $Q_k(x^{k+1}_{I_k})-\min Q_k$ measured
afterwards against an independent reference minimiser (`brentq` for a coordinate block, L-BFGS-B with the box bounds for the
full vector; `measure_reference`), which is reporting only: uncharged (ledger category `reference`), excluded from the
clock and never handed to the method. The certificates are loose by a factor that depends on the block (the ratio
$U/\text{true error}$ is in the table); the price of a certified condition (2) is the extra inner steps this costs, which
every method pays with the same rule, the same certificates and the same $\rho$.

**FISTA** remains the external convex reference of the paper ($\mu=0$), with its own backtracking, restarts and momentum.

| # | Method | Inner geometry (safe constant) | Block selection | Batch ordering | Stopping measure |
|---|---|---|---|---|---|
| 1 | `E-Random` | Euclidean, block box bound | uniform random coordinate | – | Euclidean gap (projected gradient) |
| 2 | `E-GS(PG)` | Euclidean, block box bound | Gauss–Southwell on a random batch of 16, one selected | projected gradient | Euclidean gap |
| 3 | `E-Full` | Euclidean, row-sum box bound | all coordinates | – | Euclidean gap |
| 4 | `B-Random` | Burg, $L_j$ | uniform random coordinate | – | Burg gap (Def. 4.1) |
| 5 | **`B-GS(Burg)`** | Burg, $L_j$ | Gauss–Southwell on a random batch of 16, one selected | Burg gap contribution $\Delta_j$ | Burg gap |
| 6 | `B-GS(PG)` | Burg, $L_j$ | Gauss–Southwell on a random batch of 16, one selected | projected gradient | Burg gap |
| 7 | **`B-Full`** | Burg, $L_j$ | all coordinates | – | Burg gap |
| 8 | `FISTA` | Euclidean, backtracking (reference) | convex baseline with $\mu=0$ | – | convex projected gradient |

**Both stationarity measures are computed for every method** at every stopping check and, for reporting, every `log_every_passes` $=5$ passes (uncharged):

* **Burg gap** $\operatorname{Gap}_\Psi(x)=\min_{v\in\partial h(x)}\sum_j\max_{t\in[x_{\min},x_{\max}]}\{(d_j-v_j)(x_j-t)-L_jD_B(t,x_j)\}$,
  with $d=\nabla f(x)+\lambda\mathbf 1$, $D_B(t,u)=t/u-\log(t/u)-1$ and the block relative-smoothness constants
  $L_j=\sum_{i:A_{ij}>0}y_i$ (Definition 3.1 / 4.1 with $\Psi=\sum_jL_j\psi_j$).
* **Euclidean gap** (projected-gradient/KKT residual) $\Vert x-\Pi_{\rm box}(x-(d-v))\Vert_2$, with $v\in\partial h(x)$
  chosen to minimise the unit-weight Euclidean DC gap at top-$Q$ ties (unique away from ties). The unit-weight Euclidean DC gap
  of Remark 4.1 ($\Psi=\tfrac12\Vert\cdot\Vert^2$) is also stored as `euclidean_dc_gap_unit`.

**Two protocols.** With the configuration below the notebook runs the main-text protocol (backtracking in both geometries,
$\varepsilon_k$ rule). Setting `inner_relative_tolerance=None` switches every method to the fixed schedule
$t_k=$ `inner_budget0` $+$ `inner_budget_growth` $\cdot\lfloor k/N\rfloor$ inner steps per subproblem ($N=n$ for coordinate
blocks, $N=1$ for the full vector; certificates then are reporting only), and `backtracking_geometries=()` replaces the
backtracked weights by the safe constants $L_j$ (Burg) and the box bounds (Euclidean); together with `seeds=range(105, 135)`
this is the safe-step protocol of Appendix A.4 (Figure 4, Table 2). Data, starting points, block rules, stopping measures,
tolerances, budget and the audited MatVec ledger are the same in both protocols.

## Configuration

**Instance** (Table 3 of the paper)**.** 32×32 image ($n=1024$ pixels, $d=1024$ detector pixels), $s=25$ spikes at Chebyshev separation $\ge4$ with
lognormal amplitudes (log-standard deviation 0.5) scaled to $3\cdot10^4$ photons in total, so about $1.1\cdot10^3$ photons per
spike (per-seed medians 890–1230); Gaussian PSF with $\sigma=0.3$ px truncated at radius 2 (25 nonzeros per column, 98.5% of an
interior spike's counts on its own detector pixel); detector background $b=0.5$ counts per pixel; $\lambda=\mu=0.1$ (the paper's single weight $\lambda$), $Q=25$;
box $[10^{-6},10^6]$; the flat random start $x^0_j\sim N(1,0.1^2)$ (clipped to the box, which never happens at this variance:
a start at $x_{\min}$ would leave a true spike invisible to the Burg gap, see *Interpretation*); fifty seeds (105–154). What
this instance does:

* **Narrow PSF.** Each spike dominates its own detector pixel, so near the solution the Burg constant $L_j$ (the counts on the
  column's support) is close to the local Burg curvature $x_j^2h_j(x)$ at the spikes (ratio 1.01 at $x^\star$ in the table
  "Why block size and geometry matter here"), while at the start it is 2.4 times the curvature at the spikes and about
  $4\cdot10^5$ times at background pixels.
* **Background $b=0.5$.** Every row intensity stays above $b$, so the Euclidean box constants are finite; they are tight at
  the start (ratio 1.0 at background pixels, 8.5 at the spikes) and about $4\cdot10^6$ times the curvature at the spikes near
  the solution. With backtracking in both geometries these constants no longer set the step lengths; they enter through the
  stationarity measures, the Gauss–Southwell scores and the reference $\Delta_k$ of the $\varepsilon_k$ rule.
* **$3\cdot10^4$ photons.** The Poisson noise floor of the relative error, about $\sqrt{\sum_j(a_j+b)}/\Vert a\Vert_2$ for the true
  amplitudes $a$, is 2.5–2.7% on these seeds, so the converged coordinate methods end at 2.5% error and the 5% target lies
  about twice above the floor; 2% is at the floor and 1% is unattainable.

Stopping tolerances are absolute: Euclidean methods stop when the Euclidean gap is at most `tol_pg` $=10^{-3}$, Burg methods
when the Burg gap is at most `tol_burg` $=10^{-3}$, and FISTA when its convex projected-gradient residual is at most `tol_pg`;
every run is capped at `max_passes` $=2000$ MatVec-pass equivalents. Each block solve stops at the accuracy
$\varepsilon_k=\max\{$`inner_absolute_tolerance`$,$ `inner_relative_tolerance`$\cdot\max\{\Delta_k,Q_k(x^k)-Q_k(z)\}\}$ with
`inner_relative_tolerance` $=\rho=0.1$ and no floor, certified by the dual bound of the current point (the $\varepsilon_k$ rule
above), with `max_inner_budget` $=1000$ steps as a safeguard; the schedule `inner_budget0 + inner_budget_growth * floor(k/N)`
is used only when `inner_relative_tolerance=None`. With `measure_reference=True` the true error of every accepted solve is
also measured afterwards against an independent reference minimiser, for the tables only (uncharged, untimed, never handed
to the method). The Gauss–Southwell methods score a random batch of `gs_batch_size` $=16$
coordinates per iteration. The `fista_*` parameters set the backtracking of the FISTA reference and, for the geometries
listed in `backtracking_geometries` (both), of the DCA methods, whose inner steps use backtracked local weights instead of
the safe constants (see Inner Solver). All methods share the data, the starting point and the MatVec budget. `workers` only
spreads independent runs over processes (results do not depend on it).

In [ ]:
import copy
import math
import time
from contextlib import contextmanager
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.special import kl_div
import matplotlib.pyplot as plt

FLOAT_EPS = np.finfo(float).eps

# name, user numbering, inner geometry, block selection, batch ordering score, stopping measure
METHOD_SPECS = [
    dict(name="E-Random", number=1, geometry="euclidean", selection="random", score=None, stop="pg"),
    dict(name="E-GS(PG)", number=2, geometry="euclidean", selection="gs_batch", score="pg", stop="pg"),
    dict(name="E-Full", number=3, geometry="euclidean", selection="full", score=None, stop="pg"),
    dict(name="B-Random", number=4, geometry="burg", selection="random", score=None, stop="burg"),
    dict(name="B-GS(Burg)", number=5, geometry="burg", selection="gs_batch", score="burg", stop="burg"),
    dict(name="B-GS(PG)", number=6, geometry="burg", selection="gs_batch", score="pg", stop="burg"),
    dict(name="B-Full", number=7, geometry="burg", selection="full", score=None, stop="burg"),
    dict(name="FISTA", number=8, geometry="euclidean", selection="fista", score=None, stop="convex_pg"),
]
METHOD_NAMES = [spec["name"] for spec in METHOD_SPECS]
SPEC_BY_NAME = {spec["name"]: spec for spec in METHOD_SPECS}

CONFIG_PSF = dict(
    img_shape=(32, 32), sparsity=25, largest_q=25,
    psf_sigma=0.30, psf_radius=2, min_spike_separation=4,
    support_max_attempts=256, total_photons=3e4, background=5e-1,
    lambda_=0.1, mu=0.1, x0_mean=1.0, x0_var=0.01,
    xmin=1e-6, xmax=1e6, amplitude_log_sigma=0.5,
    seeds=range(105, 155), workers=17,                  # validation run: two seeds
    inner_budget0=30, inner_budget_growth=1, max_inner_budget=1000,
    inner_relative_tolerance=0.1, inner_absolute_tolerance=0.0,   # eps_k rule: a block solve stops once its certified
                                                      # error bound U(z) >= Q_k(z)-min Q_k (dual certificate from the
                                                      # gradient of the new point) is <= max(inner_absolute_tolerance,
                                                      # (inner_relative_tolerance)*max(Delta_k, achieved decrease)); None = fixed schedule t_k
    measure_reference=True,                           # reporting only: the true error of every accepted solve against an
                                                      # independent reference minimiser (brentq / L-BFGS-B), uncharged,
                                                      # untimed, never handed to the method; False skips it
    max_outer=None,
    tol_pg=1e-3, tol_burg=1e-3,
    max_passes=2000.0, check_every_passes=10.0, log_every_passes=5.0, cache_every_passes=100.0,
    fista_max_iterations=2000000, gs_batch_size=16,
    fista_backtracking_factor=2.0, fista_scale_shrink=0.5, fista_max_backtracks=50,
    positive_scale_floor=1e-9, support_threshold=0.01, record_inner=False,
    backtracking_geometries=("euclidean", "burg"),   # inner steps of these geometries use standard backtracking
                                                      # (FISTA's policy) instead of the safe constants; () = safe steps
)


def check_config(cfg):
    n = int(np.prod(cfg["img_shape"]))
    if not (0 < cfg["xmin"] < cfg["xmax"] < np.inf):
        raise ValueError("Require a finite positive box.")
    if not (0 <= cfg["mu"] <= cfg["lambda_"] < np.inf):
        raise ValueError("Require 0 <= mu <= lambda < infinity.")
    if not (1 <= cfg["sparsity"] <= n and 0 <= cfg["largest_q"] <= n):
        raise ValueError("Invalid support size or top-Q count.")
    if not (1 <= cfg["gs_batch_size"] <= n):
        raise ValueError("The Gauss-Southwell batch must contain between 1 and n coordinates.")
    if cfg["fista_backtracking_factor"] <= 1 or not (0 < cfg["fista_scale_shrink"] <= 1):
        raise ValueError("Invalid FISTA backtracking parameters.")
    if not set(cfg.get("backtracking_geometries", ())) <= {"euclidean", "burg"}:
        raise ValueError("backtracking_geometries must be a subset of {'euclidean', 'burg'}.")
    if cfg.get("inner_relative_tolerance") is not None:
        if not (0 < cfg["inner_relative_tolerance"] < 1):
            raise ValueError("inner_relative_tolerance must lie in (0, 1) (or be None for the fixed schedule).")
        floor_ = cfg.get("inner_absolute_tolerance", 0.0)
        if not (np.isfinite(floor_) and floor_ >= 0):
            raise ValueError("inner_absolute_tolerance must be finite and nonnegative.")
    for name in ("background", "max_passes", "tol_pg", "tol_burg", "check_every_passes",
                 "log_every_passes", "cache_every_passes", "positive_scale_floor"):
        if not np.isfinite(cfg[name]) or cfg[name] <= 0:
            raise ValueError(f"{name} must be finite and positive.")
    for name in ("inner_budget0", "max_inner_budget", "fista_max_backtracks", "support_max_attempts",
                 "fista_max_iterations", "gs_batch_size"):
        if int(cfg[name]) != cfg[name] or cfg[name] < 1:
            raise ValueError(f"Invalid integer limit: {name}.")
    if int(cfg["inner_budget_growth"]) != cfg["inner_budget_growth"] or cfg["inner_budget_growth"] < 0:
        raise ValueError("inner_budget_growth must be a nonnegative integer.")
    if cfg["max_outer"] is not None and cfg["max_outer"] < 1:
        raise ValueError("max_outer must be positive or None.")


def configuration_table(cfg):
    return pd.DataFrame({"parameter": list(cfg), "value": [repr(v) for v in cfg.values()]})


def method_table():
    return pd.DataFrame([dict(number=m["number"], method=m["name"], inner_geometry=m["geometry"],
                              selection=m["selection"], batch_ordering=m["score"] or "-",
                              stopping_measure=dict(pg="Euclidean gap (projected gradient)", burg="Burg gap",
                                                    convex_pg="convex projected gradient")[m["stop"]])
                         for m in METHOD_SPECS])

In [ ]:
from IPython.display import display, Markdown
display(Markdown(
    f"Active configuration: **{CONFIG_PSF['img_shape'][0]} x {CONFIG_PSF['img_shape'][1]}**, "
    f"{CONFIG_PSF['sparsity']} spikes, Q={CONFIG_PSF['largest_q']}, lambda={CONFIG_PSF['lambda_']}, mu={CONFIG_PSF['mu']}, "
    f"PSF sigma={CONFIG_PSF['psf_sigma']}, radius={CONFIG_PSF['psf_radius']}, "
    f"box=[{CONFIG_PSF['xmin']}, {CONFIG_PSF['xmax']}], seeds={CONFIG_PSF['seeds']}, "
    f"MatVec budget={CONFIG_PSF['max_passes']} passes, tolerances: Euclidean gap <= {CONFIG_PSF['tol_pg']}, "
    f"Burg gap <= {CONFIG_PSF['tol_burg']}."))
display(method_table())
display(configuration_table(CONFIG_PSF))

## Data and Arithmetic

Columns of the truncated Gaussian PSF are normalised (column-stochastic $A$). Spike amplitudes are lognormal and scaled to the
photon total; the data are ordinary Poisson counts (zeros included). The data generation and the random starting point depend
only on the seed, so every method and both protocols see the same instance and start. Padded per-column tables allow a batch of `gs_batch_size` (16) partial
gradients to be computed from those columns alone.

For $D_B(z,u)=\sum_j[z_j/u_j-\log(z_j/u_j)-1]$ with weights $W_j$, the Burg model minimiser is
$z_j=\operatorname{clip}(W_ju_j/(W_j+u_jg_j))$ when the denominator is positive and $x_{\max}$ otherwise; the Euclidean one is
$\operatorname{clip}(u_j-g_j/W_j)$. Scaled products, a fused multiply-add near cancellation and `log1p` series keep these
formulas accurate.

In [ ]:
def make_gaussian_psf_sparse_matrix(img_shape, sigma, radius):
    height, width = map(int, img_shape)
    offsets = [(di, dj, np.exp(-(di*di+dj*dj)/(2*sigma*sigma)))
               for di in range(-radius, radius+1) for dj in range(-radius, radius+1)]
    rows, cols, values = [], [], []
    for i in range(height):
        for j in range(width):
            local = [(int((i+di)*width+j+dj), w) for di, dj, w in offsets
                     if 0 <= i+di < height and 0 <= j+dj < width]
            total = sum(w for _, w in local)
            for row, w in local:
                rows.append(row)
                cols.append(i*width+j)
                values.append(w/total)
    A = sp.csc_matrix((values, (rows, cols)), shape=(height*width, height*width))
    A.eliminate_zeros()
    A.sort_indices()
    return A


def separated_support(shape, count, separation, rng, max_attempts):
    height, width = shape
    for _ in range(max_attempts):
        selected = []
        for index in rng.permutation(height*width):
            i, j = divmod(int(index), width)
            if all(max(abs(i-divmod(other, width)[0]), abs(j-divmod(other, width)[1]))
                   >= separation for other in selected):
                selected.append(int(index))
                if len(selected) == count:
                    return np.array(sorted(selected), dtype=int)
    raise ValueError("Could not construct the requested separated support.")


def column_tables(A):
    """Padded row indices and values of every column (padding has value 0)."""
    A = sp.csc_matrix(A)
    counts = np.diff(A.indptr)
    width = int(counts.max()) if len(counts) else 0
    filled = np.arange(width)[None, :] < counts[:, None]
    rows = np.zeros((A.shape[1], width), dtype=np.int64)
    values = np.zeros((A.shape[1], width))
    rows[filled], values[filled] = A.indices, A.data
    return rows, values, counts.astype(np.int64)


def prepare_problem(cfg, seed):
    check_config(cfg)
    started = time.perf_counter()
    rng = np.random.default_rng(seed)
    A = make_gaussian_psf_sparse_matrix(cfg["img_shape"], cfg["psf_sigma"], cfg["psf_radius"])
    support = separated_support(cfg["img_shape"], cfg["sparsity"], cfg["min_spike_separation"],
                                rng, cfg["support_max_attempts"])
    truth = np.zeros(A.shape[1])
    truth[support] = rng.lognormal(0.0, cfg["amplitude_log_sigma"], len(support))
    truth *= cfg["total_photons"]/float((A @ truth).sum())
    y = rng.poisson(A @ truth+cfg["background"]).astype(float)
    start_rng = np.random.default_rng(seed+1000003)
    x0 = np.clip(start_rng.normal(cfg["x0_mean"], np.sqrt(cfg["x0_var"]), A.shape[1]),
                 cfg["xmin"], cfg["xmax"])
    col_rows, col_vals, col_nnz = column_tables(A)
    problem = dict(A=A, A_csr=A.tocsr(), AT=A.T.tocsr(),
                   colsum=np.asarray(A.sum(axis=0)).ravel(),
                   y=y, x_star=truth, support=support, x0=x0, seed=seed,
                   col_rows=col_rows, col_vals=col_vals, col_nnz=col_nnz)
    problem["shared_setup_seconds"] = time.perf_counter()-started
    problem["shared_setup_counts"] = dict(matrix_nonzeros=A.nnz, synthetic_forward_contributions=2*A.nnz,
                                          column_sum_elements=A.nnz)
    return problem


def t_minus_log1p(t):
    """t - log(1+t) for t > -1, with a series near zero."""
    t = np.asarray(t, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        direct = t-np.log1p(t)
    series = t*t*(0.5+t*(-1/3+t*(1/4-t/5)))
    return np.where(np.abs(t) < 1e-4, series, direct)


def t_minus_log1p_scalar(t):
    if abs(t) < 1e-4:
        return t*t*(0.5+t*(-1/3+t*(1/4-t/5)))
    return t-math.log1p(t)


def product_ratio(a, b, c):
    """Compute a*b/c without overflowing a representable intermediate product."""
    a, b, c = np.broadcast_arrays(np.asarray(a, float), np.asarray(b, float), np.asarray(c, float))
    ma, ea = np.frexp(a)
    mb, eb = np.frexp(b)
    mc, ec = np.frexp(c)
    with np.errstate(over="ignore", under="ignore", invalid="raise", divide="raise"):
        return np.ldexp(ma*mb/mc, ea+eb-ec)


def accurate_multiply_add(a, b, c):
    if hasattr(math, "fma"):
        return math.fma(float(a), float(b), float(c))
    from decimal import Decimal, localcontext
    with localcontext() as context:
        context.prec = 60
        return float(Decimal.from_float(float(a))*Decimal.from_float(float(b))+Decimal.from_float(float(c)))


def _weighted_burg_safe(z, u, weight):
    """Overflow-safe weight*D_B(z,u) for extreme ratios (scaled products)."""
    with np.errstate(over="ignore", under="ignore", invalid="ignore", divide="ignore"):
        t = (z-u)/u
        near = np.abs(t) < 0.5
        close = weight*t_minus_log1p(np.where(near, t, 0.0))
        far = product_ratio(weight, z, u)-weight*(1+np.log(z)-np.log(u))
    return np.where(weight > 0, np.where(near, close, far), 0.0)


def weighted_burg(z, u, weight):
    """Elementwise weight*D_B(z,u); entries with zero weight contribute exactly zero."""
    z, u, weight = np.asarray(z, float), np.asarray(u, float), np.asarray(weight, float)
    if not (z.shape == u.shape == weight.shape):
        z, u, weight = np.broadcast_arrays(z, u, weight)
    if z.size and (z.min() <= 0 or u.min() <= 0 or weight.min() < 0):
        raise ValueError("Invalid weighted Burg divergence.")
    with np.errstate(over="ignore", under="ignore", invalid="ignore", divide="ignore"):
        t = (z-u)/u
        near = np.abs(t) < 0.5
        # near 1: t-log1p(t) (series for |t|<1e-4); far from 1: ratio-1-log(ratio) has no cancellation
        value = np.where(near, t_minus_log1p(np.where(near, t, 0.0)), z/u-1.0-(np.log(z)-np.log(u)))
        out = np.where(weight > 0, weight*value, 0.0)
    if not np.all(np.isfinite(out)):
        out = _weighted_burg_safe(z, u, weight)
    return out


def mirror_proposal(u, gradient, scale, geometry, lower, upper):
    u, gradient, scale = np.broadcast_arrays(np.asarray(u, float), np.asarray(gradient, float),
                                            np.asarray(scale, float))
    if np.any(scale <= 0) or not np.all(np.isfinite(scale)):
        raise FloatingPointError("Step scales must be finite and positive.")
    if geometry == "euclidean":
        with np.errstate(over="ignore"):
            return np.clip(u-gradient/scale, lower, upper)
    if geometry != "burg":
        raise ValueError("Unknown inner geometry.")
    # Use the dimensionless ratio instead of forming scale*u or u*gradient.
    ratio = product_ratio(gradient, u, scale)
    denominator = 1+ratio
    trial = np.full(u.shape, upper, dtype=float)
    interior = denominator > 0
    with np.errstate(over="ignore", under="ignore"):
        trial[interior] = u[interior]/denominator[interior]
    near_cancellation = np.flatnonzero(np.abs(denominator).ravel() < 1e-6)
    for index in near_cancellation:
        a, g, weight = u.flat[index], gradient.flat[index], scale.flat[index]
        exact_denominator = accurate_multiply_add(g, a, weight)
        trial.flat[index] = (float(product_ratio(weight, a, exact_denominator))
                             if exact_denominator > 0 else upper)
    return np.clip(trial, lower, upper)


def model_scores(x, gradient, weights, geometry, lower, upper, work=None):
    """Per-coordinate model decrease max_t {g(x-t) - W D(t,x)} (the block contribution Delta_j)."""
    if work is not None:
        work.extra("gap_model_proposals", (70 if geometry == "burg" else 35)*len(x))
    trial = mirror_proposal(x, gradient, weights, geometry, lower, upper)
    delta = trial-x
    boundary = (trial == lower) | (trial == upper)
    if geometry == "burg":
        # At an interior model optimum the gap equals W*D_B(x,trial).
        # Boundary KKT terms avoid subtracting two huge forward-model terms.
        raw = weighted_burg(x, trial, weights)
        derivative = gradient+product_ratio(weights, delta/trial, x)
        critical = boundary & (np.abs(1+product_ratio(gradient, x, weights)) < 1e-6)
        for j in np.flatnonzero(critical):
            denominator = accurate_multiply_add(gradient[j], x[j], weights[j])
            derivative[j] = (denominator-float(product_ratio(weights[j], x[j], trial[j])))/x[j]
    else:
        raw = 0.5*weights*delta*delta
        derivative = gradient+weights*delta
    raw = raw-np.where(boundary, derivative*delta, 0.0)
    roundoff = 128*FLOAT_EPS*(1+np.abs(gradient*delta)+np.abs(raw))
    if not np.all(np.isfinite(raw)) or np.any(raw < -roundoff):
        raise FloatingPointError("Materially negative or nonfinite GS/DC gap.")
    return dict(scores=np.where(raw < 0, 0.0, raw), trial=trial, roundoff=roundoff)


def top_q_mask(x, q):
    mask = np.zeros(len(x))
    if q:
        mask[np.argsort(-x, kind="stable")[:q]] = 1.0
    return mask


def top_q_sum(x, q):
    return float(np.partition(x, len(x)-q)[-q:].sum()) if q else 0.0


def original_objective(x, intensity, y, cfg, convex=False):
    return float(kl_div(y, intensity).sum()+cfg["lambda_"]*x.sum()
                 -(0.0 if convex else cfg["mu"]*top_q_sum(x, cfg["largest_q"])))


def projected_residual(x, gradient, lower, upper):
    """x - clip(x - g, lower, upper) without cancellation: it equals g exactly where the step stays in the box."""
    trial = x-gradient
    return np.where(trial < lower, x-lower, np.where(trial > upper, x-upper, gradient))


def support_metrics(x, support, threshold):
    truth = np.zeros(len(x), dtype=bool)
    truth[support] = True
    estimated = x > threshold*float(x.max())
    hits = int(np.count_nonzero(truth & estimated))
    count = int(estimated.sum())
    top = np.argsort(-x, kind="stable")[:len(support)]
    return dict(support_precision=hits/max(count, 1), support_recall=hits/max(len(support), 1),
                support_f1=2*hits/max(len(support)+count, 1), estimated_support_size=count,
                topk_support_f1=float(np.count_nonzero(truth[top]))/max(len(support), 1))

## Accounting and Cutoff Ties

**MatVec-pass equivalents.** One pass is $\operatorname{nnz}(A)$ touched matrix entries (a column of $A$ counts its own
nonzeros). Every product a method uses is charged, and only those (ledger category in brackets):

| operation | charged | when |
|---|---|---|
| intensity $s^0=Ax^0+b$, gradient at $x^0$ | 1 + 1 pass | setup, all methods [setup, stopping] |
| setup quantities | 1 pass each | all: column sums $A^\top\mathbf 1$; DCA: $L_j=\sum_{i:A_{ij}>0}y_i$ (Burg weights, the minimiser bound $U_j$ in every certificate, the Burg gap); E-Full: row sums and the Euclidean row-sum box bound; FISTA: row sums (its initial Lipschitz estimate) [setup] |
| coordinate gradient $w_j=A_{:j}^\top(y/s)$ | 1 column | each random draw, unless the gradient at the current $x$ is known [inner] |
| Gauss–Southwell batch | `gs_batch_size` $=16$ columns | each GS iteration, unless the gradient at the current $x$ is known; the selected $w_j$ is reused [selection] |
| coordinate offset $r=s_{\rm rows}-A_{:j}x_j$ | 1 column, plus the touched rows of $A$ where cancellation forces a rebuild | only when a step can move the block [offset] |
| Euclidean safe weight of a coordinate block | 1 column | once per block that can move [inner] |
| step point $s=r+Bz$ | 1 column / 1 pass | each safe step [inner] |
| gradient at a step point | 1 column / 1 pass | after every accepted step under the $\varepsilon_k$ rule (it gives the next step and the certificate of the $\varepsilon_k$ decision, for coordinate and full blocks alike); with the fixed schedule, before every further step and after the last step only for full blocks [inner] |
| FISTA (reference, backtracking) | 1 pass per trial, 1 pass per gradient at the extrapolated point | each iteration [inner] |
| stopping check | 1 pass | coordinate methods and FISTA when `check_every_passes` have passed since the last check, unless the gradient at $x$ is known; free for full DCA [stopping] |
| intensity refresh | 1 pass | coordinate methods, every `cache_every_passes` (numerical hygiene) [cache] |

Only data layouts (CSR/CSC copies, padded column tables, row extraction) are free, for every method. The budget `max_passes`
applies to this count only. Estimated vector-operation counts are kept in a separate ledger (`modeled_work_passes`) but never
stop a run. Reporting-only work (the gaps recorded between stopping checks, the Burg constants FISTA only reports with,
post-run checks) is counted in a separate `diagnostics` category, excluded from the budget and from the wall-clock time, and
never handed to the method; the independent reference minimiser behind the true-error column of the inner-accuracy table
(`brentq` for a coordinate, L-BFGS-B for the full vector; Brent: one column forward and one column transpose product per
derivative evaluation, L-BFGS-B: one forward and one transpose pass per function evaluation) is tallied the same way in its
own uncharged category `reference`. The audit cell after the validation recounts every product independently of this ledger. The
coordinate offset has its own category: it could be avoided by forming step intensities as $s_{\rm rows}+A_{:j}(z-x_j)$,
which would make every coordinate method about one column per moving block cheaper.

**Ties.** At a top-$Q$ cutoff tie $x_j=\tau$ the admissible subgradients satisfy $0\le v_j\le\mu$ and
$\sum_{j\in T}v_j=\mu q_T$. To minimise a separable gap over this set, solve the monotone allocation equation
$v_j=\operatorname{clip}(d_j^0+W_j\alpha,0,\mu)$, $\sum_{j\in T}v_j=\mu q_T$ by bisection; it makes all interior model
trials equal, which is the KKT condition of the allocation problem for both the Burg and the quadratic model.
The DCA steps themselves use the deterministic top-$Q$ subgradient (stable index order at ties) for every method.

In [ ]:
class WorkLimit(RuntimeError):
    pass


@dataclass
class Work:
    nnz: int
    max_passes: float
    counts: dict = field(default_factory=lambda: dict(setup=0, selection=0, inner=0, offset=0,
                                                      cache=0, stopping=0, diagnostics=0, reference=0))
    operations: dict = field(default_factory=dict)
    events: dict = field(default_factory=lambda: dict(target_evaluations=0, inner_steps=0, rejected_trials=0,
                                                      gs_batches=0, gs_candidates=0, gs_cached_batches=0, certificates=0,
                                                      forward_products=0, transpose_products=0, offsets_formed=0,
                                                      safe_constants_formed=0, majorization_violations=0,
                                                      restarts=0, extrapolations=0, extrapolation_products=0,
                                                      adhoc_setup=0, curvature_estimates=0, reference_solves=0))
    reporting: bool = False
    started: float = field(default_factory=time.perf_counter)
    charged: int = 0
    excluded_seconds: float = 0.0
    stopped: float = None

    @contextmanager
    def untimed(self):
        """Reporting work (tables, plots, post-run checks) is excluded from the method's wall-clock time."""
        started = time.perf_counter()
        try:
            yield
        finally:
            self.excluded_seconds += time.perf_counter()-started

    @property
    def seconds(self):
        end = time.perf_counter() if self.stopped is None else self.stopped
        return end-self.started-self.excluded_seconds

    @property
    def passes(self):
        """MatVec-pass equivalents charged to the method (the budget and the x-axis)."""
        return self.charged/self.nnz

    @property
    def total_passes(self):
        return sum(self.counts.values())/self.nnz

    @property
    def modeled_work_passes(self):
        return (2*sum(self.counts.values())+sum(self.operations.values()))/(2*self.nnz)

    def charge(self, contributions, category):
        contributions = int(contributions)
        if self.reporting:
            self.counts["diagnostics"] += contributions
            return
        if (self.charged+contributions)/self.nnz > self.max_passes+1e-12:
            raise WorkLimit("matvec_budget")
        self.counts[category] += contributions
        self.charged += contributions

    def charge_free(self, contributions, category="diagnostics"):
        """Uncharged products: 'diagnostics' (plots, tables and post-run checks) and 'reference' (the independent
        reference minimiser that reports the true error of a block solve; see Inner Solver). Neither influences the
        method; neither counts towards its cost or budget; both appear in the ledger."""
        self.counts[category] += int(contributions)

    def extra(self, family, units):
        # Vector bookkeeping ledger (reported only).
        self.operations[family] = self.operations.get(family, 0)+int(math.ceil(units))


def charge_top_q(work, n):
    work.extra("top_q_sorting", n*(math.ceil(math.log2(max(n, 2)))+6))


def minimizing_subgradient(x, base_gradient, weights, q, mu, work=None):
    """Minimise the sum of separable DC gaps over the top-Q subdifferential.

    At a cutoff tie x_j=tau, the allocation is v_j=clip(d_j+W_j*a,0,mu).
    Interior entries then have a common model trial, hence a common gap
    derivative with respect to v. This satisfies the allocation KKT system
    for either Burg or quadratic models, including clipped model trials.
    """
    n = len(x)
    if work is not None:
        charge_top_q(work, n)
    v = mu*top_q_mask(x, q)
    if mu == 0 or q == 0 or q == n:
        return v, dict(tie_size=0, tie_iterations=0, allocation_error=0.0)
    cutoff = float(np.partition(x, n-q)[n-q])
    above, tied = x > cutoff, x == cutoff
    slots = q-int(above.sum())
    m = int(tied.sum())
    if slots == m:
        return v, dict(tie_size=m, tie_iterations=0, allocation_error=0.0)
    d, w = base_gradient[tied], weights[tied]
    target = mu*slots
    low, high = float(np.min(-d/w)), float(np.max((mu-d)/w))
    allocation = np.clip(d, 0.0, mu)
    for steps in range(100):
        middle = low+0.5*(high-low)
        allocation = np.clip(d+w*middle, 0.0, mu)
        if allocation.sum() < target:
            low = middle
        else:
            high = middle
        if high == low or np.nextafter(low, np.inf) >= high:
            break
    # Restore the sum constraint after the final floating-point root rounding.
    remainder = target-float(allocation.sum())
    order = np.argsort(-np.minimum(allocation, mu-allocation), kind="stable")
    for j in order:
        change = np.clip(remainder, -allocation[j], mu-allocation[j])
        allocation[j] += change
        remainder -= change
        if abs(remainder) <= 8*FLOAT_EPS*max(1.0, target):
            break
    v[above], v[~(above | tied)], v[tied] = mu, 0.0, allocation
    if work is not None:
        work.extra("tie_gap_minimization", (steps+1)*8*m+20*m)
    error = abs(float(v[tied].sum())-target)
    if error > 128*FLOAT_EPS*max(1.0, target):
        raise FloatingPointError("Top-Q tie allocation lost feasibility.")
    return v, dict(tie_size=m, tie_iterations=steps+1, allocation_error=error)


ALL_CONSTANTS = ("column_sum", "ysupport", "row_sum", "euclid_cap")


def constant_needs(spec, cfg=None):
    """Setup quantities each method's algorithm uses. With Euclidean backtracking ("euclidean" in
    cfg["backtracking_geometries"]) the full Euclidean method needs the row sums for its curvature estimate but no
    longer the box bound."""
    if spec is None:
        return set(ALL_CONSTANTS)
    if spec["selection"] == "fista":
        return {"column_sum", "row_sum"}                  # gradient A^T 1 - A^T(y/s); Lipschitz estimate
    # Every DCA method uses L_j (the minimiser bound U_j in all certificates; the Burg weights and Burg gap).
    needs = {"column_sum", "ysupport"}
    if spec["geometry"] == "euclidean" and spec["selection"] == "full":
        needs |= {"row_sum"}                              # Lipschitz estimate (backtracking) or the safe diagonal weights
        if cfg is None or "euclidean" not in cfg.get("backtracking_geometries", ()):
            needs |= {"euclid_cap"}
    return needs


def geometry_constants(problem, cfg, work, needs=ALL_CONSTANTS):
    """Setup quantities, one pass each, charged when the method uses them: column sums A^T 1 (stored with the problem,
    charged here per method), L_j=sum_{i:A_ij>0} y_i (formed uncharged for FISTA, which only reports the Burg gap), the row
    sums A 1 and the Euclidean box bound A^T(y*A1/(b+x_min*A1)^2) (full Euclidean DCA; FISTA uses the row sums)."""
    A, y = problem["A"], problem["y"]
    work.extra("constant_setup_vectors", 2*A.nnz+4*len(y))
    if "column_sum" in needs:
        work.charge(A.nnz, "setup")
    support = A.copy()
    support.data = np.ones_like(support.data)
    if "ysupport" in needs:
        work.charge(A.nnz, "setup")
    else:
        work.charge_free(A.nnz)
    ysupport = np.asarray(support.T @ y).ravel()
    floor = cfg["positive_scale_floor"]
    row_sum = euclid_cap = None
    if "row_sum" in needs:
        work.charge(A.nnz, "setup")
        row_sum = np.asarray(A @ np.ones(A.shape[1])).ravel()
    if "euclid_cap" in needs:
        work.charge(A.nnz, "setup")
        minimum_intensity = cfg["background"]+cfg["xmin"]*row_sum
        euclid_cap = np.maximum(floor, np.asarray(problem["AT"] @ (y*row_sum/minimum_intensity**2)).ravel())
    return dict(burg=np.maximum(ysupport, floor), ysupport=ysupport, unit=np.ones(A.shape[1]), row_sum=row_sum,
                euclid_cap=euclid_cap)

## Frozen Target and Objective-Error Certificate

At each outer iteration the anchor, $v^k$, $r=b+A_{-I}x^k_{-I}$ and $c=\lambda\mathbf 1-v^k_I$ are frozen. The target gradient is
$B^\top(\mathbf 1-y/(r+Bz))+c$; for scalar blocks the likelihood rows not touched by the column are an additive constant.

**Dual certificate.** For $p_i\ge0$ (positive when $y_i>0$) and any box $[\ell,U]$ that contains a minimiser,
$D(p)=(\mathbf 1-p)^\top r+\sum_iy_i\log p_i+\sum_j\min\{a_j(p)\ell_j,a_j(p)U_j\}$ with $a(p)=B^\top(\mathbf 1-p)+c$ is a lower bound
on $\min Q_k$. With $p=\theta y/s$, $b_0=B^\top\mathbf 1+c$ and $w=B^\top(y/s)$,

$$Q(z)-D(p)=\Big(\sum_iy_i\Big)(\theta-1-\log\theta)+\sum_j\max\{a_j(z_j-\ell_j),\,a_j(z_j-U_j)\},\qquad a=b_0-\theta w .$$

**Lemma (minimiser bound).** Every minimiser satisfies $z^\star_j\le U_j:=\max\{x_{\min},\min\{x_{\max},
\sum_{i:B_{ij}>0}y_i/b_{0,j}\}\}$. *Proof.* If $z^\star_j>x_{\min}$ then $\partial_jQ(z^\star)\le0$, i.e.
$b_{0,j}\le\sum_iB_{ij}y_i/s_i\le\sum_{i:B_{ij}>0}y_i/z^\star_j$ because $s_i=r_i+\sum_kB_{ik}z^\star_k\ge B_{ij}z^\star_j$. $\square$
Hence the certificate may use $U_j$ instead of $x_{\max}=10^6$. The bound is a convex function of $\theta$; its exact minimiser is
found from the sorted breakpoints $b_{0,j}/w_j$ (closed form for scalar blocks). $\theta=1$ (the box Frank–Wolfe gap on
$[\ell,U]$) and the smallest $\theta$ that makes all coefficients nonnegative are always tried as well.
A `128*eps` cushion covers arithmetic at the scale of the counts, coefficients, iterate and endpoints.

**Safe weights (no line search).** A step from $u$ minimises $g^\top(z-u)+\sum_jW_jD(z_j,u_j)$ over the box, with
$D=D_B$ (Burg) or $D(z,u)=\tfrac12(z-u)^2$ (Euclidean). It decreases $Q_k$ whenever the likelihood remainder satisfies
$R(u,z)=\sum_iy_iD_B(s_i(z),s_i(u))\le\sum_jW_jD(z_j,u_j)$ for *all* $u,z$ in the box; the weights below guarantee this.

* *Burg, $W_j=L_j=\sum_{i:B_{ij}>0}y_i$.* The Hessian is $H(u)=B^\top\operatorname{diag}(y/s^2)B$ with $s=r+Bu$. Since
  $B_{ij}u_j\le s_i$, the row sums of $\operatorname{diag}(u)H\operatorname{diag}(u)$ satisfy
  $u_j\sum_iy_iB_{ij}(Bu)_i/s_i^2\le\sum_{i:B_{ij}>0}y_i=L_j$ at every point, and a symmetric nonnegative matrix is dominated
  by the diagonal of its row sums, so $\nabla^2f\preceq\operatorname{diag}(L_j/u_j^2)=\nabla^2\big(\sum_jL_j\psi_j\big)$: $f$ is
  relatively smooth with respect to $\sum_jL_j\psi_j$ jointly, for scalar blocks and for the full vector alike.
* *Euclidean, coordinate block.* With the offset $r$ frozen, $Q''(z)=\sum_iy_iB_i^2/(r_i+B_iz)^2$ decreases in $z$, so
  $W=Q''(x_{\min})=\sum_iy_iB_i^2/(r_i+B_ix_{\min})^2$ bounds the curvature on the whole box. It is formed from a single read
  of the column, as $(r_i/B_i+x_{\min})^{-2}$.
* *Euclidean, full vector.* On the box $s\ge s_{\min}=r+x_{\min}B\mathbf 1$, so $H(u)\mathbf 1\le
  B^\top(y\odot B\mathbf 1/s_{\min}^2)$, and again $H\preceq\operatorname{diag}(H\mathbf 1)$.

The validation checks the majorisation on random point pairs, and the solver still compares the exact remainder with the
model after every step; a violation would mean an invalid constant and is counted (it never occurs).

In [ ]:
def theta_slack(theta):
    """theta - 1 - log(theta): log1p series near 1, direct formula for small theta (theta-1 would lose theta's digits)."""
    return t_minus_log1p_scalar(theta-1.0) if theta > 0.5 else theta-1.0-math.log(theta)


def optimal_dual_scale(z, ysum, bcoef, w, lower, upper):
    """Exact minimiser of the convex bound G(theta) over theta > 0 (vector blocks)."""
    positive = w > 0
    if ysum <= 0 or not positive.any():
        return 1.0
    wp = w[positive]
    breakpoints = bcoef[positive]/wp
    jumps = wp*(upper[positive]-lower)
    order = np.argsort(breakpoints, kind="stable")
    breakpoints = breakpoints[order]
    cumulative = np.concatenate(([0.0], np.cumsum(jumps[order])))
    first = float(wp @ (z[positive]-lower))
    denominator = ysum-first+cumulative
    with np.errstate(divide="ignore", invalid="ignore"):
        roots = np.where(denominator > 0, ysum/np.where(denominator > 0, denominator, 1.0), np.inf)
    lower_end = np.concatenate(([0.0], breakpoints))
    upper_end = np.concatenate((breakpoints, [np.inf]))
    crossing = np.flatnonzero(roots <= upper_end)
    if crossing.size == 0:
        return 1.0
    k = int(crossing[0])
    theta = float(max(roots[k], lower_end[k]))
    return theta if (np.isfinite(theta) and theta > 0) else 1.0


def _certificate_candidate(theta, z, ysum, bcoef, w, lower, upper):
    coefficient = bcoef-theta*w
    bound = np.where(coefficient >= 0, lower, upper)
    poisson = ysum*theta_slack(theta)
    raw = poisson+float(coefficient @ (z-bound))
    scale = (1+ysum+float((np.abs(bcoef)+theta*np.abs(w)) @ (np.abs(z)+np.abs(bound)))+abs(poisson))
    cushion = 128*FLOAT_EPS*scale
    if raw < -cushion or not np.isfinite(raw):
        raise FloatingPointError("Invalid Poisson dual bound.")
    return dict(gap=max(0.0, raw)+cushion, raw_gap=raw, cushion=cushion, theta=theta,
                certificate_type="poisson_box_dual")


def poisson_box_certificate(z, y, bcoef, w, lower, upper, work=None, ysum=None):
    """Bound on Q(z)-min Q with p=theta*y/s on the box [lower, upper] (upper may be the certified U)."""
    if work is not None:
        work.extra("box_dual_certificate", 16*len(y)+60*len(z))
        work.events["certificates"] += 1
    ysum = float(y.sum()) if ysum is None else float(ysum)
    upper = np.broadcast_to(np.asarray(upper, float), z.shape)
    thetas = [1.0]
    positive = w > 0
    if positive.any():
        feasible = min(1.0, float(np.min(bcoef[positive]/w[positive])))
        if 0 < feasible < 1:
            thetas.append(float(np.nextafter(feasible, 0.0)))
        best_theta = optimal_dual_scale(z, ysum, bcoef, w, lower, upper)
        if best_theta not in thetas:
            thetas.append(best_theta)
    best = None
    for theta in thetas:
        result = _certificate_candidate(theta, z, ysum, bcoef, w, lower, upper)
        if best is None or result["gap"] < best["gap"]:
            best = result
    return best


def poisson_box_certificate_scalar(z, ysum, b, w, lower, upper, work=None):
    """Scalar-block version of poisson_box_certificate with the closed-form optimal theta."""
    if work is not None:
        work.extra("box_dual_certificate", 76)
        work.events["certificates"] += 1
    thetas = [1.0]
    if w > 0:
        feasible = min(1.0, b/w)
        if 0 < feasible < 1:
            thetas.append(math.nextafter(feasible, 0.0))
        breakpoint = b/w
        d0 = ysum-w*(z-lower)
        theta = ysum/d0 if d0 > 0 else math.inf
        if theta > breakpoint:
            d1 = ysum+w*(upper-z)
            theta = max(ysum/d1 if d1 > 0 else math.inf, breakpoint)
        if 0 < theta < math.inf and theta not in thetas:
            thetas.append(theta)
    best = None
    for theta in thetas:
        a = b-theta*w
        bound = lower if a >= 0 else upper
        poisson = ysum*theta_slack(theta)
        raw = poisson+a*(z-bound)
        scale = 1+ysum+(abs(b)+theta*abs(w))*(abs(z)+abs(bound))+abs(poisson)
        cushion = 128*FLOAT_EPS*scale
        if raw < -cushion or not math.isfinite(raw):
            raise FloatingPointError("Invalid Poisson dual bound.")
        gap = max(0.0, raw)+cushion
        if best is None or gap < best["gap"]:
            best = dict(gap=gap, raw_gap=raw, cushion=cushion, theta=theta, certificate_type="poisson_box_dual")
    return best


class FrozenTarget:
    def __init__(self, B, y, offset, c, anchor, cfg, work, scalar=False, colsum=None, row_sum=None,
                 ysupport=None, euclid_cap=None, BT=None, offset_builder=None):
        self.scalar, self.work, self.cfg = scalar, work, cfg
        self.y, self.c, self.anchor = [np.array(a, dtype=float, copy=True) for a in (y, c, anchor)]
        for array in (self.y, self.c, self.anchor):
            array.flags.writeable = False
        self._offset, self._offset_builder = None, offset_builder
        if offset is not None:
            self._set_offset(offset)
        elif offset_builder is None:
            raise ValueError("Need a frozen offset or an offset builder.")
        # Ad-hoc blocks (validation only) derive missing quantities from B; experiment blocks receive them all
        # precomputed and charged at setup, which the MatVec audit checks through the `adhoc_setup` event.
        adhoc = sum(value is None for value in (colsum, ysupport))
        work.events["adhoc_setup"] += adhoc
        if scalar:
            self.B = np.array(B, dtype=float, copy=True)
            self.B.flags.writeable = False
            self.BT, self.nnz = None, self.B.size
            self.colsum = np.array([self.B.sum() if colsum is None else float(np.ravel(colsum)[0])])
            self.row_sum = self.B
        else:
            # Shared, never-modified CSR operators of the problem (no per-iteration copy).
            self.B = B
            self.BT = B.T.tocsr() if BT is None else BT
            self.nnz = self.B.nnz
            self.colsum = np.asarray(self.B.sum(axis=0)).ravel() if colsum is None else np.array(colsum, float)
            self.row_sum = None if row_sum is None else np.array(row_sum, float)     # formed (charged) on first use
        self.bcoef = self.colsum+self.c
        if self.bcoef.min() <= 0:
            raise ValueError("Require a positive column sum plus linear cost.")
        self.ysum = float(self.y.sum())
        if ysupport is None:
            ysupport = (np.array([self.ysum]) if scalar else
                        np.asarray((self.B != 0).astype(float).T @ self.y).ravel())
        self.ysupport = np.atleast_1d(np.asarray(ysupport, float))
        # Lemma: every block minimiser satisfies z_j <= ysupport_j/bcoef_j.
        self.upper_eff = np.clip(self.ysupport/self.bcoef, cfg["xmin"], cfg["xmax"])
        self._euclid_cap = None if euclid_cap is None else np.atleast_1d(np.asarray(euclid_cap, float))
        self._caps = {}

    def _set_offset(self, offset):
        offset = np.array(offset, dtype=float, copy=True)
        if offset.min() <= 0:
            raise ValueError("Require a positive frozen offset.")
        offset.flags.writeable = False
        self._offset = offset

    @property
    def offset(self):
        """Frozen part of the intensity; for coordinate blocks it is formed (and charged) on first use."""
        if self._offset is None:
            self._set_offset(self._offset_builder())
        return self._offset

    def row_sums(self):
        """Row sums B1 of a vector block: supplied when the method paid for them at setup, else formed here (charged)."""
        if self.row_sum is None:
            self.work.charge(self.nnz, "inner")
            self.row_sum = np.asarray(self.B @ np.ones(self.B.shape[1])).ravel()
        return self.row_sum

    def cap(self, geometry):
        """Safe weights of the block, valid on the whole box (no line search needed). Burg: L_j (relative smoothness),
        known from setup. Euclidean: the largest curvature of this block over the box -- for a coordinate block
        sum_i y_i B_i^2/(r_i+B_i x_min)^2 with its frozen offset r (one column), for the full vector the Hessian row-sum
        bound B^T(y*B1/(r+x_min*B1)^2) (supplied from setup in experiments)."""
        if geometry not in self._caps:
            floor = self.cfg["positive_scale_floor"]
            if geometry == "burg":
                value = self.ysupport
            elif geometry == "euclidean":
                value = self._euclid_cap
                if value is None:
                    xmin = self.cfg["xmin"]
                    self.work.events["safe_constants_formed"] += 1
                    if self.scalar:
                        # (r_i+B_i x_min)/B_i = r_i/B_i + x_min: a single read of the column.
                        self.work.charge(self.nnz, "inner")
                        with np.errstate(divide="ignore"):             # a zero entry contributes zero curvature
                            value = np.array([float(self.y @ (1.0/(self.offset/self.B+xmin))**2)])
                    else:
                        # H(z)1 <= B^T(y*B1/(r+xmin*B1)^2) on the box: row sums and one transpose product.
                        row_sum = self.row_sums()
                        self.work.charge(self.nnz, "inner")
                        smin = self.offset+xmin*row_sum
                        value = np.asarray(self.BT @ (self.y*row_sum/smin**2)).ravel()
            else:
                raise ValueError("Unknown inner geometry.")
            self._caps[geometry] = np.maximum(floor, value)
        return self._caps[geometry]

    def _count(self, event, category):
        # Charged products only; reporting ('diagnostics') and reference-solver ('reference') products are tallied
        # uncharged and not as algorithm events. Neither influences the method.
        if category in ("diagnostics", "reference"):
            self.work.charge_free(self.nnz, category)
        else:
            self.work.charge(self.nnz, category)
            if not self.work.reporting:
                self.work.events[event] += 1

    def mv(self, z, category="inner"):
        self._count("forward_products", category)
        return self.B*z[0] if self.scalar else self.B @ z

    def tmv(self, vector, category="inner"):
        self._count("transpose_products", category)
        return np.array([self.B @ vector]) if self.scalar else self.BT @ vector

    def certificate(self, z, w):
        if self.scalar:
            return poisson_box_certificate_scalar(float(z[0]), self.ysum, float(self.bcoef[0]), float(w[0]),
                                                  self.cfg["xmin"], float(self.upper_eff[0]), self.work)
        return poisson_box_certificate(z, self.y, self.bcoef, w, self.cfg["xmin"], self.upper_eff,
                                       self.work, ysum=self.ysum)

    def evaluate(self, z, intensity=None, w=None, gradient=True, certify=True):
        """Value at z (one forward product unless the intensity is supplied); with gradient=True also the
        gradient (one transpose product unless w is supplied) and, if certify, the objective-error certificate."""
        z = np.asarray(z, float)
        xmin, xmax = self.cfg["xmin"], self.cfg["xmax"]
        if not (z.min() >= xmin and z.max() <= xmax):
            raise ValueError("Infeasible inner point.")
        self.work.extra("target_vectors_values", 18*len(self.y)+8*len(z))
        s = self.offset+self.mv(z) if intensity is None else np.array(intensity, dtype=float, copy=True)
        if not (s.min() > 0 and np.isfinite(s.max())):
            raise FloatingPointError("Nonpositive or nonfinite intensity.")
        terms = kl_div(self.y, s)
        value = (float(terms.sum())+float(self.c[0])*float(z[0]) if self.scalar else
                 float(terms.sum()+self.c @ z))
        self.work.events["target_evaluations"] += 1
        state = dict(z=z.copy(), s=s, w=None, grad=None, value=value, likelihood_terms=terms, cert=None)
        if gradient or w is not None:
            self.complete(state, w, certify=certify)
        return state

    def complete(self, state, w=None, category="inner", certify=True):
        """Attach the gradient (one transpose product unless w is supplied) and, if certify, the certificate."""
        if state["w"] is None:
            w = self.tmv(self.y/state["s"], category) if w is None else w
            state["w"] = np.array(w, dtype=float, copy=True).reshape(-1)
            state["grad"] = self.bcoef-state["w"]
        if certify and state["cert"] is None:
            state["cert"] = self.certificate(state["z"], state["w"])
        return state

    def rowsum_curvature(self, state):
        """Largest row sum of H(u): an upper bound on its largest eigenvalue (FISTA's initial Lipschitz estimate)."""
        self.work.extra("curvature_row_bound", 8*len(self.y)+6*len(state["z"]))
        return max(self.cfg["positive_scale_floor"], float(np.max(self.tmv(self.y/state["s"]**2*self.row_sums()))))

    def local_curvature(self, state, geometry):
        """Initial weights of the backtracking at the current point z (one charged product): the smallest multiple of the
        geometry's kernel whose model majorises the block's curvature at z (via Hessian row sums; H <= diag(H1) for the
        nonnegative H). Coordinate block: Q''(z) = sum_i y_i B_i^2/s_i^2 for Euclidean, z^2 Q''(z) for Burg (one read of
        the column). Full block, Euclidean kernel |.|^2/2: the largest row sum of H(z) for every coordinate, as for FISTA
        (one transpose product). Full block, Burg kernel sum_j L_j psi_j: theta*L_j with theta = max_j z_j (H(z) z)_j / L_j,
        the local looseness of the safe constants (one transpose product; Az = s - offset is known)."""
        if not self.work.reporting:
            self.work.events["curvature_estimates"] += 1
        z, s = state["z"], state["s"]
        floor = self.cfg["positive_scale_floor"]
        if self.scalar:
            self.work.charge(self.nnz, "inner")
            self.work.extra("curvature_scalar", 6*len(self.y))
            curvature = float(np.square(self.B) @ (self.y/s**2))
            value = np.array([curvature if geometry == "euclidean" else curvature*float(z[0])**2])
        elif geometry == "euclidean":
            self.work.extra("curvature_row_bound", 8*len(self.y)+6*len(z))
            value = np.full(len(z), float(np.max(self.tmv(self.y/s**2*self.row_sums()))))
        else:
            self.work.extra("curvature_row_bound", 8*len(self.y)+10*len(z))
            local = z*np.asarray(self.tmv(self.y*(s-self.offset)/s**2)).ravel()
            weights = np.maximum(floor, self.ysupport)
            value = float(np.max(local/weights))*weights
        return np.maximum(floor, value)


def make_target(problem, x, s, v, selected, cfg, work, constants):
    """Frozen DCA block target; v is the full subgradient (full block) or v_j (coordinate block)."""
    if selected is None:
        work.extra("frozen_target_copy", 5*len(x)+len(s))
        return FrozenTarget(problem["A_csr"], problem["y"], np.full(len(s), cfg["background"]),
                            cfg["lambda_"]-v, x, cfg, work, colsum=problem["colsum"],
                            row_sum=constants["row_sum"], ysupport=constants["ysupport"],
                            euclid_cap=constants["euclid_cap"], BT=problem["AT"]), None
    A = problem["A"]
    begin, end = A.indptr[selected], A.indptr[selected+1]
    rows, values = A.indices[begin:end], A.data[begin:end]
    s_rows, x_j = s[rows].copy(), float(x[selected])

    def build_offset():
        # Formed only when the solver attempts a step; charged as one column read.
        work.charge(len(values), "offset")
        work.events["offsets_formed"] += 1
        work.extra("frozen_coordinate_copy", 10*len(values)+8)
        offset = s_rows-values*x_j
        risky = offset < 1e-7*np.maximum(s_rows, 1.0)
        if risky.any():
            submatrix = problem["A_csr"][rows[risky], :]
            work.extra("offset_rebuild_extraction", submatrix.nnz+2*len(x))
            other = x.copy()
            other[selected] = 0.0
            work.charge(submatrix.nnz, "offset")
            offset[risky] = cfg["background"]+np.asarray(submatrix @ other).ravel()
        return offset

    return FrozenTarget(values, problem["y"][rows], None, np.array([cfg["lambda_"]-v]),
                        x[selected:selected+1], cfg, work, scalar=True,
                        colsum=np.array([problem["colsum"][selected]]),
                        ysupport=np.array([constants["ysupport"][selected]]), offset_builder=build_offset), rows

## Inner Solver

Both geometries minimise the same frozen $Q_k$. From the current point $u$ with gradient $g$:

* **Burg:** $z_j=\operatorname{clip}\big(W_ju_j/(W_j+u_jg_j)\big)$, the minimiser of $g^\top(t-u)+\sum_jW_jD_B(t_j,u_j)$;
* **Euclidean:** $z=\operatorname{clip}(u-g/W)$, the minimiser of $g^\top(t-u)+\tfrac12\sum_jW_j(t_j-u_j)^2$.

**Weights.** With *safe steps* (geometry not in `backtracking_geometries`) the weights are the constants above, $W_j=L_j$ for
Burg and the box bound for Euclidean, valid on the whole box, so no line search is needed. With *standard backtracking*
(this version, both geometries) the weights are found as in the FISTA reference: at the first step of a block solve they are
the local curvature of the block in the geometry's metric (coordinate block: $Q''(u)=\sum_iy_iB_i^2/s_i^2$ for Euclidean and
$u^2Q''(u)$ for Burg, one column read; full block: the smallest multiple of the method's kernel that majorises the Hessian at $u$ through its row
sums, i.e. the largest row sum of $H(u)$ for the Euclidean kernel $\tfrac12\Vert\cdot\Vert^2$, as for FISTA, and
$\theta L_j$ with $\theta=\max_j u_j(H(u)u)_j/L_j$, the local looseness of the safe constants, for the Burg kernel
$\sum_jL_j\psi_j$; one transpose product), at every further step they are multiplied by `fista_scale_shrink` (0.5), and they
are multiplied by `fista_backtracking_factor` (2) until the geometry's upper model holds, $Q(z)\le Q(u)+g^\top(z-u)+
\tfrac12\sum_jW_j(z_j-u_j)^2$ or $Q(z)\le Q(u)+g^\top(z-u)+\sum_jW_jD_B(z_j,u_j)$ (at most `fista_max_backtracks` trials).
Every trial costs one forward product and is charged; rejected trials are counted (`rejected_trials`). The two geometries
therefore receive exactly the same curvature information and the same backtracking policy; they differ only in the step
formula and in the divergence of the upper model.
**Stopping ($\varepsilon_k$ rule).** At the anchor the solve computes $\Delta_k$ (the Burg model decrease with the weights
$L_j$, i.e. `model_scores` at the anchor gradient) and the dual certificate of the anchor. After every accepted step it forms
the gradient of the new point $z$ (one charged transpose product, the one the next step would need) and from it a certified
upper bound on the error, $U(z)=Q_k(z)-\ell$, where $\ell$ is the best lower bound on $\min Q_k$ seen so far in the solve:
the Poisson box dual bound with the optimal scale $\theta$ and the minimiser bound $U_j$ (every block) and, for a coordinate
block, the derivative bracket $|Q_k'(z)|\cdot(\text{width of the bracket } [x_{\min},U_j]$ narrowed by the sign of every
certified derivative$)$, both with a roundoff cushion. Since every certificate is a lower bound on the same frozen minimum,
$U(z)\ge Q_k(z)-\min Q_k$, and the solve stops with status `inner_tolerance` as soon as
$U(z)\le\max\{\varepsilon_{\rm abs},\rho\max\{\Delta_k,Q_k(x^k)-Q_k(z)\}\}$; the returned point then satisfies condition (2)
with this $\varepsilon_k$ by certificate, not by measurement. No reference solver takes part: the rule, the certificates and
$\rho$ are the same for every method and every geometry. The gradient of the returned point is kept and reused by the driver
(full blocks) or discarded (coordinate blocks). A step that changes $Q_k$ by less than its floating-point
resolution ends the solve (`change_below_resolution`; the point is the block minimiser to working precision, but a
certificate cannot fall below roundoff, so these solves are reported as uncertified). The cap `budget` $=$ `max_inner_budget` ends a solve that has not
reached its target (`budget_used`; the point is still accepted). A block that sits at a bound with its gradient pointing out
of the box (`block_stationary`, exact error 0), or whose step is below floating-point resolution (`step_below_resolution`),
stops immediately; a coordinate block of the first kind forms neither its offset nor its weight. After each step the exact
remainder is compared with the model; a violation would discard the step and stop the solve (it never occurs). When the
MatVec budget runs out inside a solve, the accepted outer iterate is kept and the unfinished candidate is exposed separately
(`pending`). For reporting only (`measure_reference=True`), the true error of the returned point is then measured against
an independent reference minimiser (`reference_minimum`: `brentq` on $Q_k'$ over $[x_{\min},U_j]$ to machine precision for a
coordinate block, L-BFGS-B with the box bounds, warm-started at the returned point, for the full vector). Its products go to
the uncharged ledger category `reference`, its time is excluded from the method's clock and its result is never handed to
the method; the inner-accuracy table shows the certified and the true error side by side. With `inner_relative_tolerance=None` the solver runs the
fixed schedule $t_k$ of the safe-step protocol (Appendix A.4, Figure 4 and Table 2 of the paper; certificates reporting only, no gradient after the last step of a coordinate block).

In [ ]:
def stable_target_change(target, old, new):
    target.work.extra("target_change_check", 45*len(target.y)+6*len(new["z"]))
    delta = new["z"]-old["z"]
    remainder = float(weighted_burg(new["s"], old["s"], target.y).sum())
    linear = float(old["grad"] @ delta)
    return linear+remainder, linear, remainder


def upper_model_cushion(target, old_s, new_s, remainder, penalty):
    # The remainder is evaluated with log1p/series arithmetic. An absolute
    # O(eps) allowance would let line search take oversized steps near a root.
    relative_change = np.abs(new_s-old_s)/np.minimum(new_s, old_s)
    arithmetic = 128*FLOAT_EPS
    return (arithmetic*(abs(remainder)+abs(penalty)+float(target.y @ relative_change))
            +arithmetic**2*target.ysum+np.finfo(float).tiny)


def bracket_certificate(target, state, bracket):
    if not target.scalar:
        return
    z, derivative = float(state["z"][0]), float(state["grad"][0])
    if derivative >= 0:
        bracket[1] = min(bracket[1], z)
        distance = z-bracket[0]
    else:
        bracket[0] = max(bracket[0], z)
        distance = bracket[1]-z
    roundoff = 128*FLOAT_EPS*(1+target.ysum+(abs(float(target.bcoef[0]))+abs(float(state["w"][0])))*abs(distance))
    bound = abs(derivative)*max(distance, 0.0)+roundoff
    if bound < state["cert"]["gap"]:
        state["cert"] = dict(state["cert"], gap=bound, cushion=roundoff, certificate_type="scalar_derivative_bracket")


def reference_minimum(target, start):
    """Minimum of the frozen block target by an independent reference solver, for the eps_k measurement. Coordinate
    block: `brentq` on Q' over [x_min, U_j] (machine precision). Full block: L-BFGS-B with the box bounds, warm-started.
    Reporting only in this version: its products are tallied in the uncharged ledger category 'reference' (Brent: one
    column product and one column transpose product per derivative evaluation; L-BFGS-B: one forward and one transpose
    product per function evaluation), its time is excluded from the method's clock, and its result is never handed to
    the method (the stopping decision uses the certified bound).
    Returns the minimum value, the minimiser and the dual/bracket certificate at the minimiser (how far the reference
    itself is from optimal, reporting)."""
    from scipy.optimize import brentq, minimize
    work, cfg = target.work, target.cfg
    xmin, xmax = cfg["xmin"], cfg["xmax"]
    y, offset, c, bcoef = target.y, target.offset, target.c, target.bcoef
    if target.scalar:
        upper = float(target.upper_eff[0])

        def derivative(z):
            s = offset+target.mv(np.array([z]), "reference")
            return float(bcoef[0])-float(target.tmv(y/s, "reference")[0])

        work.extra("reference_solver", 60*len(y))
        at_bound = True
        if derivative(xmin) >= 0:
            z_ref = xmin                                   # minimiser at the lower bound (derivative points out)
        elif upper <= xmin or derivative(upper) <= 0:
            z_ref = upper                                  # at the minimiser bound U_j (= x_max when clipped)
        else:
            z_ref = brentq(derivative, xmin, upper, xtol=1e-300, rtol=4*FLOAT_EPS, maxiter=500)
            at_bound = False
        z_ref = np.array([z_ref])
        s_ref = offset+target.mv(z_ref, "reference")
        w_ref = target.tmv(y/s_ref, "reference")
        value = float(kl_div(y, s_ref).sum())+float(c[0])*float(z_ref[0])
        # error bound of the reference itself: 0 at a bound with the derivative pointing out, else |Q'| x bracket width
        certificate = 0.0 if at_bound else abs(float(bcoef[0]-w_ref[0]))*max(z_ref[0]-xmin, upper-z_ref[0])
    else:
        def fun(z):
            s = offset+target.mv(z, "reference")
            work.extra("reference_solver", 20*len(y)+6*len(z))
            return float(kl_div(y, s).sum()+c @ z), bcoef-target.tmv(y/s, "reference")

        result = minimize(fun, np.asarray(start, float), jac=True, method="L-BFGS-B",
                          bounds=[(xmin, xmax)]*len(start),
                          options=dict(ftol=1e-15, gtol=1e-10, maxiter=50000, maxfun=100000, maxls=100))
        z_ref = np.clip(result.x, xmin, xmax)
        s_ref = offset+target.mv(z_ref, "reference")
        w_ref = target.tmv(y/s_ref, "reference")
        value = float(kl_div(y, s_ref).sum()+c @ z_ref)
        certificate = poisson_box_certificate(z_ref, y, bcoef, w_ref, xmin, target.upper_eff, ysum=target.ysum)["gap"]
    return value, z_ref, certificate


def solve_inner(target, geometry, budget, initial_s=None, initial_w=None, progress=None, keep_trace=False,
                final_gradient=True, certify=None):
    """At most `budget` accepted steps on the block.

    Safe steps (geometry not in `backtracking_geometries`): Burg z = argmin_t {g(t-u) + L_j D_B(t,u)} with the weights
    L_j, Euclidean z = clip(u - g/W) with the block's box bound W; no line search. Backtracking (geometry in
    `backtracking_geometries`): the same step formulas with weights found by standard backtracking -- they start at the
    local curvature of the block in the geometry's metric (one charged product), are multiplied by `fista_scale_shrink`
    at every step and by `fista_backtracking_factor` until the geometry's upper model (quadratic or weighted Burg)
    holds; every trial costs one forward product and rejected trials are counted.
    In every case the exact likelihood remainder is compared with the model after each step, so Q_k never increases;
    a violation of a *safe* step would mean an invalid constant (the step is then discarded and counted). The solve ends
    after `budget` steps, or earlier if no step can change the block. `final_gradient=False` skips the gradient after the
    last step (coordinate blocks do not need it).

    eps_k rule (`inner_relative_tolerance` = rho set, this version): after every accepted step the gradient of the new
    point is formed (one charged transpose product) and a *certified* upper bound on its error is computed from it,
    U(z) >= Q_k(z) - min Q_k: the Poisson box dual bound with the optimal scale theta and the minimiser bound U_j (every
    block) and, for a coordinate block, the derivative bracket (both convexity bounds; see *Frozen Target*). Every bound
    is a lower bound on min Q_k, so the best lower bound seen within the frozen subproblem (anchor included) is kept and
    U(z) = Q_k(z) - (best lower bound). The solve stops with status `inner_tolerance` as soon as
    U(z) <= eps_k = max(inner_absolute_tolerance, rho*max{Delta_k, Q_k(x^k)-Q_k(z)}), which certifies condition (2) of
    the paper outright: the true error is at most U(z). Delta_k is the Burg model decrease at the anchor with the safe
    weights L_j (the block's contribution to the Burg gap of Definition 4.1) and Q_k(x^k)-Q_k(z) the decrease achieved so
    far; both are lower bounds on the available decrease Q_k(x^k)-min Q_k, and either way
    phi(x^k)-phi(x^{k+1}) >= Q_k(x^k)-Q_k(x^{k+1}) >= (1-rho)*Delta_k, which is what the proofs of Theorems 4.1-4.2 use
    (with n/(1-rho) instead of n and no sum of eps_k). The same rule, certificates and rho apply to every method,
    whatever its inner geometry; no reference solver takes part in the decision. A solve whose accepted step changes Q_k
    by less than 8 eps (1+|Q_k(x^k)|), the floating-point resolution of Q_k, ends with `change_below_resolution` (the
    block is at its minimiser to working precision but the certificate cannot fall below roundoff; this happens when
    rho*Delta_k is below resolution, e.g. a coordinate at x_min whose Burg contribution is tiny; such solves are reported
    separately as uncertified); `budget` (= max_inner_budget) is only a cap. For reporting only (`measure_reference`),
    the true error of the returned point is measured afterwards against an independent reference minimiser
    (`reference_minimum`: brentq for a coordinate, L-BFGS-B for the full vector), whose products are tallied uncharged
    in the ledger category 'reference', excluded from the method's clock and never handed to the method; it shows how
    loose the certificates are.
    Without the rule, certificates are reporting only and formed for full blocks at the anchor and at the end."""
    cfg, work = target.cfg, target.work
    xmin, xmax = cfg["xmin"], cfg["xmax"]
    backtracking = geometry in cfg.get("backtracking_geometries", ())
    rho = cfg.get("inner_relative_tolerance")                         # None = fixed schedule
    certify = (rho is not None or not target.scalar) if certify is None else certify
    initial = target.evaluate(target.anchor, initial_s, initial_w, certify=certify)
    state, steps, violations, trace = initial, 0, 0, []
    status, weights, scale = "budget_used", None, None
    bracket = [xmin, float(target.upper_eff[0]) if target.scalar else xmax]     # narrowed by every certified derivative
    eps_target = eps_reference = eps_true = eps_achieved = eps_certified = reference_value = reference_cert = np.nan
    lower_bound = -np.inf                                             # best certified lower bound on min Q_k so far
    if rho is not None:
        eps_reference = float(model_scores(initial["z"], initial["grad"], np.maximum(cfg["positive_scale_floor"],
                                           target.ysupport), "burg", xmin, xmax, work)["scores"].sum())
        eps_floor = float(cfg.get("inner_absolute_tolerance", 0.0))
        eps_target = max(eps_floor, float(rho)*eps_reference)
        resolution = 8*FLOAT_EPS*(1+abs(initial["value"]))
        if target.scalar:
            bracket_certificate(target, initial, bracket)
        lower_bound = initial["value"]-initial["cert"]["gap"]          # the anchor's certificate
        eps_certified = max(0.0, initial["value"]-lower_bound)
    try:
        while steps < budget:
            z, g = state["z"], state["grad"]
            if np.all((g == 0) | ((z <= xmin) & (g > 0)) | ((z >= xmax) & (g < 0))):
                status = "block_stationary"          # at a bound with the gradient pointing out: no step can move it
                break
            if backtracking:
                scale = (target.local_curvature(state, geometry) if scale is None
                         else np.maximum(cfg["positive_scale_floor"], scale*cfg["fista_scale_shrink"]))
                accepted, proposal = False, z
                for _ in range(cfg["fista_max_backtracks"]):
                    work.extra("inner_step_proposal", (45 if geometry == "burg" else 12)*len(z))
                    proposal = mirror_proposal(z, g, scale, geometry, xmin, xmax)
                    if np.array_equal(proposal, z):
                        break                                      # below floating-point resolution
                    candidate = target.evaluate(proposal, gradient=False)       # one forward product per trial
                    change, linear, remainder = stable_target_change(target, state, candidate)
                    delta = proposal-z
                    penalty = (float(weighted_burg(proposal, z, scale).sum()) if geometry == "burg"
                               else 0.5*float(scale @ (delta*delta)))
                    work.extra("inner_model_divergence", 35*len(delta)+6*len(target.y))
                    cushion = upper_model_cushion(target, state["s"], candidate["s"], remainder, penalty)
                    if keep_trace:
                        trace.append(dict(remainder=remainder, penalty=penalty, cushion=cushion, change=change,
                                          weights=np.array(scale, copy=True), old_z=z.copy(), new_z=proposal.copy(),
                                          gradient=g.copy(), old_value=state["value"], new_value=candidate["value"]))
                    if remainder <= penalty+cushion:
                        accepted = True
                        break
                    work.events["rejected_trials"] += 1
                    scale = scale*cfg["fista_backtracking_factor"]
                if not accepted:
                    status = "step_below_resolution" if np.array_equal(proposal, z) else "backtracking_limit"
                    break
                weights = scale
            else:
                if weights is None:
                    weights = target.cap(geometry)       # Burg: L_j from setup; Euclidean: the block's box bound
                work.extra("inner_step_proposal", (45 if geometry == "burg" else 12)*len(z))
                proposal = mirror_proposal(z, g, weights, geometry, xmin, xmax)
                if np.array_equal(proposal, z):
                    status = "step_below_resolution"
                    break
                candidate = target.evaluate(proposal, gradient=False)       # one forward product
                change, linear, remainder = stable_target_change(target, state, candidate)
                delta = proposal-z
                penalty = (float(weighted_burg(proposal, z, weights).sum()) if geometry == "burg"
                           else 0.5*float(weights @ (delta*delta)))
                work.extra("inner_model_divergence", 35*len(delta)+6*len(target.y))
                cushion = upper_model_cushion(target, state["s"], candidate["s"], remainder, penalty)
                if keep_trace:
                    trace.append(dict(remainder=remainder, penalty=penalty, cushion=cushion, change=change,
                                      weights=np.array(weights, copy=True), old_z=z.copy(), new_z=proposal.copy(),
                                      gradient=g.copy(), old_value=state["value"], new_value=candidate["value"]))
                if remainder > penalty+cushion:
                    violations += 1
                    work.events["majorization_violations"] += 1
                    status = "majorization_violation"
                    break
            steps += 1
            work.events["inner_steps"] += 1
            stop = False
            if rho is not None:
                # eps_k decision: certified error of the new point. Its gradient (one charged transpose product, the
                # one the next step needs) gives the dual certificate and, for a coordinate, the derivative bracket;
                # every certificate is a lower bound on min Q_k, and the best one seen in this solve is kept.
                state = target.complete(candidate)
                bracket_certificate(target, state, bracket)
                lower_bound = max(lower_bound, state["value"]-state["cert"]["gap"])
                eps_certified = max(0.0, state["value"]-lower_bound)
                eps_achieved = initial["value"]-state["value"]
                eps_target = max(eps_floor, float(rho)*max(eps_reference, eps_achieved))
                if eps_certified <= eps_target:
                    status, stop = "inner_tolerance", True
                elif abs(change) <= resolution:
                    status, stop = "change_below_resolution", True
            elif (steps < budget) or final_gradient:
                state = target.complete(candidate, certify=False)   # one transpose product: the next gradient
            else:
                state = candidate
            if stop:
                break
            if progress is not None:
                progress(state, steps)
    except WorkLimit as error:
        status = str(error)
    if certify:
        if state["w"] is None:
            target.complete(state, category="diagnostics", certify=False)   # reporting gradient for the certificate
        target.complete(state)                        # certificate of the returned point (no product)
        bracket_certificate(target, state, bracket)
    if rho is not None:
        if steps > 0:
            lower_bound = max(lower_bound, state["value"]-state["cert"]["gap"])
            eps_certified = max(0.0, state["value"]-lower_bound)
        if cfg.get("measure_reference", True) and steps > 0 and not work.reporting:
            # Reporting only: the true error of the returned point against an independent reference minimiser
            # (uncharged, untimed, never handed to the method), to show how loose the certificates are.
            with work.untimed():
                reference_value, _, reference_cert = reference_minimum(target, state["z"])
            work.events["reference_solves"] += 1
            eps_true = max(0.0, state["value"]-reference_value)
    cert, anchor_cert = state["cert"], initial["cert"]
    return state, dict(status=status, steps=steps, budget=budget, violations=violations, trace=trace,
                       eps_target=eps_target, eps_reference=eps_reference, eps_true=eps_true, eps_achieved=eps_achieved,
                       eps_certified=eps_certified, reference_certificate=reference_cert,
                       gap=np.nan if cert is None else cert["gap"],
                       anchor_gap=np.nan if anchor_cert is None else anchor_cert["gap"],
                       certificate_type=None if cert is None else cert["certificate_type"],
                       anchor_w=initial["w"], anchor_value=initial["value"])

## Stationarity Measures and Block Selection

`dc_measures` evaluates, at any point, the **Burg gap** (Definition 4.1 with $\Psi=\sum_jL_j\psi_j$, minimised over
$\partial h$ at ties), the **Euclidean gap** (projected-gradient residual $\Vert x-\Pi(x-(d-v))\Vert_2$ with the
Euclidean-gap-minimising $v$), its sup-norm, the unit-weight Euclidean DC gap of Remark 4.1, and the convex residual ($\mu=0$).
A stopping check costs one full gradient pass unless the gradient is already available (full methods and FISTA reuse it).

**Batch Gauss–Southwell.** Each GS iteration draws a fresh batch $J$ of $B=$ `gs_batch_size` $=16$ distinct coordinates uniformly at random, computes
$g_j=\nabla_jf(x)+\lambda-v_j$ for $j\in J$ from those $B$ columns only, scores them and updates the best one:

* `burg` ordering: the Burg gap contribution $\Delta_j=\max_t\{g_j(x_j-t)-L_jD_B(t,x_j)\}$ (the block contribution $\Delta^k_i$ of Theorem 4.2 for the coordinate $j$);
* `pg` ordering: the projected-gradient magnitude $|x_j-\Pi(x_j-g_j)|$.

Because every coordinate is in $J$ with probability $B/n$, $\mathbb E[\max_{j\in J}\Delta_j]\ge\frac1B\mathbb E[\sum_{j\in J}\Delta_j]=\frac1n\sum_j\Delta_j\ge
\frac1n\operatorname{Gap}_\Psi(x)$, so the Burg-scored batch rule keeps the worst-case $\mathcal O(n/K)$ guarantee of Theorem 4.2
(the block descent inequality in the proof of Theorem 4.2, Appendix B.5, holds for whichever block is updated; the adaptive factor $\Gamma_k$ is stated for the full rule). The PG-ordered rule has no such guarantee in the Burg gap.
Random methods draw one coordinate uniformly with replacement. Methods sharing a selection rule use the same random stream.

In [ ]:
def full_gradient(problem, s, work, category):
    work.extra("full_gradient_vectors", 2*len(s))
    if category == "diagnostics":
        work.charge_free(problem["A"].nnz)
    else:
        work.charge(problem["A"].nnz, category)
    return np.asarray(problem["AT"] @ (problem["y"]/s)).ravel()


def dc_measures(problem, x, w, constants, cfg, work=None):
    lower, upper, q, mu = cfg["xmin"], cfg["xmax"], cfg["largest_q"], cfg["mu"]
    base = problem["colsum"]+cfg["lambda_"]-w
    if work is not None:
        work.extra("stationarity_vectors", 30*len(x))
    vb, burg_ties = minimizing_subgradient(x, base, constants["burg"], q, mu, work)
    burg = model_scores(x, base-vb, constants["burg"], "burg", lower, upper, work)
    ve, euclid_ties = minimizing_subgradient(x, base, constants["unit"], q, mu, work)
    euclid = model_scores(x, base-ve, constants["unit"], "euclidean", lower, upper, work)
    residual = projected_residual(x, base-ve, lower, upper)
    return dict(burg_gap=float(burg["scores"].sum()),
                euclidean_gap=float(np.linalg.norm(residual)),
                euclidean_gap_inf=float(np.max(np.abs(residual))),
                euclidean_dc_gap_unit=float(euclid["scores"].sum()),
                convex_pg_residual=float(np.linalg.norm(projected_residual(x, base, lower, upper))),
                burg_tie_size=burg_ties["tie_size"], euclid_tie_size=euclid_ties["tie_size"])


def stationarity(problem, x, s, constants, cfg, work, cached_w=None, category="stopping"):
    w = full_gradient(problem, s, work, category) if cached_w is None else cached_w
    return dc_measures(problem, x, w, constants, cfg, work), w


def batch_scores(problem, x, s, mask, constants, cfg, batch, score, w_batch=None):
    if w_batch is None:
        rows, values = problem["col_rows"][batch], problem["col_vals"][batch]
        w_batch = np.einsum("ij,ij->i", values, problem["y"][rows]/s[rows])
    g_batch = problem["colsum"][batch]+cfg["lambda_"]-cfg["mu"]*mask[batch]-w_batch
    if score == "burg":
        scores = model_scores(x[batch], g_batch, constants["burg"][batch], "burg", cfg["xmin"], cfg["xmax"])["scores"]
    elif score == "pg":
        scores = np.abs(projected_residual(x[batch], g_batch, cfg["xmin"], cfg["xmax"]))
    else:
        raise ValueError("Unknown Gauss-Southwell score.")
    return scores, w_batch, g_batch


def select_block(problem, x, s, mask, constants, spec, rngs, cfg, work, cached_w=None):
    """Return (coordinate or None, v for the target, partial gradient w_j or None, info).
    A Gauss-Southwell batch reads its gs_batch_size (16) columns unless the gradient at the current x is known (cached_w)."""
    selection = spec["selection"]
    if selection == "full":
        return None, cfg["mu"]*mask, None, {}
    n = len(x)
    if selection == "random":
        work.extra("uniform_coordinate_sampling", 8)
        j = int(rngs["random"].integers(n))
        return j, float(cfg["mu"]*mask[j]), None, {}
    if selection != "gs_batch":
        raise ValueError("Unknown coordinate selection.")
    batch = rngs["batch"].choice(n, size=cfg["gs_batch_size"], replace=False)
    if cached_w is None:
        touched, known = int(problem["col_nnz"][batch].sum()), None
        work.charge(touched, "selection")
    else:
        touched, known = 0, cached_w[batch]
        work.events["gs_cached_batches"] += 1
    work.extra("gs_batch_scores", (70 if spec["score"] == "burg" else 12)*len(batch)+4*touched)
    scores, w_batch, g_batch = batch_scores(problem, x, s, mask, constants, cfg, batch, spec["score"], known)
    work.events["gs_batches"] += 1
    work.events["gs_candidates"] += len(batch)
    k = int(np.argmax(scores))
    j = int(batch[k])
    return j, float(cfg["mu"]*mask[j]), float(w_batch[k]), dict(
        selected_score=float(scores[k]), batch_max_score=float(scores.max()), batch_size=len(batch),
        selected_gradient=float(g_batch[k]))


def update_top_q_mask(x, mask, j, old, q, work):
    if q == 0:
        return 0.0
    was_top = bool(mask[j])
    difference = x[j]-old if was_top else 0.0
    if q == len(x):
        return float(difference)
    if was_top and x[j] < old:
        work.extra("top_q_coordinate_scan", 5*len(x))
        entrant = int(np.argmax(np.where(mask == 0, x, -np.inf)))
        if x[entrant] > x[j] or (x[entrant] == x[j] and entrant < j):
            mask[j], mask[entrant] = 0.0, 1.0
            difference += x[entrant]-x[j]
    elif not was_top and x[j] > old:
        work.extra("top_q_coordinate_scan", 5*len(x))
        inside = np.where(mask > 0, x, np.inf)
        leaving = len(x)-1-int(np.argmin(inside[::-1]))
        if x[j] > x[leaving] or (x[j] == x[leaving] and j < leaving):
            mask[leaving], mask[j] = 0.0, 1.0
            difference += x[j]-x[leaving]
    work.extra("top_q_coordinate_bookkeeping", 12)
    return float(difference)


MEASURE_COLUMNS = ("burg_gap", "euclidean_gap", "euclidean_gap_inf", "euclidean_dc_gap_unit", "convex_pg_residual")


def record_state(problem, x, s, cfg, work, method, outer, updates, phase, status,
                 diagnostics=None, info=None, objective=None):
    work.extra("reporting_metrics", 40*len(x)+14*len(s))
    if objective is None:
        objective = original_objective(x, s, problem["y"], cfg)
    row = dict(seed=problem["seed"], method=method, phase=phase, status=status,
        accepted_state=phase != "inner_candidate", outer_iteration=outer, actual_updates=updates,
        objective=float(objective), convex_objective=float(kl_div(problem["y"], s).sum()+cfg["lambda_"]*x.sum()),
        relative_x_error=float(np.linalg.norm(x-problem["x_star"])/np.linalg.norm(problem["x_star"])),
        **support_metrics(x, problem["support"], cfg["support_threshold"]),
        subproblem_certificate=np.nan, convex_certificate=np.nan, **{key: np.nan for key in MEASURE_COLUMNS})
    row.update({key: value for key, value in (diagnostics or {}).items() if key in MEASURE_COLUMNS})
    if info is not None:
        row["convex_certificate" if method == "FISTA" else "subproblem_certificate"] = info["gap"]
    row.update(matvec_passes=work.passes, total_matvec_passes=work.total_passes,
               modeled_work_passes=work.modeled_work_passes, wall_clock_time=work.seconds,
               **{key+"_passes": value/work.nnz for key, value in work.counts.items()}, **work.events)
    return row

## DCA Driver

All seven DCA methods share this driver (Algorithm 1). Per outer iteration $k$: select the block; freeze $Q_k$; solve it to
the certified accuracy $\varepsilon_k=\rho\max\{\Delta_k,Q_k(x^k)-Q_k(z)\}$ (the solve stops once its dual error bound is
at most $\varepsilon_k$; at most `max_inner_budget` steps); accept the result (every step decreased $Q_k$, so $F$ never
increases). The certified error, the true error against the reporting-only reference minimiser, the target, $\Delta_k$,
the achieved decrease and the status of every accepted solve are kept
(`run["inner"]`). With `inner_relative_tolerance=None` the inner budget is the
schedule $t_k=t_0+\lfloor k/N\rfloor$ ($N=n$ for coordinate methods, $N=1$ for full methods) of the safe-step protocol.

**Stopping criteria of the outer loop** (checked at $x^0$ and then after accepted updates):

| Criterion | Status |
|---|---|
| method's own measure $\le$ tolerance (Euclidean gap $\le$ `tol_pg` for methods 1–3, Burg gap $\le$ `tol_burg` for 4–7) | `stationarity_reached` |
| next matrix product would exceed `max_passes` | `matvec_budget` |
| `max_outer` reached (off by default) | `outer_iteration_limit` |
| a full block cannot move: every step is below floating-point resolution | `no_step_possible` |
| the exact remainder exceeds the safe model (an invalid constant; never observed) | `majorization_violation` |
| floating-point failure | `numerical_failure: ...` |

Full methods check after every outer iteration (the gradient at the new iterate comes free from the inner solve).
Coordinate methods check every `check_every_passes` passes; a check costs one full gradient pass, charged to the method
(a check is also forced after $n$ consecutive attempts that charged nothing, so that blocks needing no step with a known
gradient can never spin the loop for free; this never happens in practice).
Between checks, both measures are also computed every `log_every_passes` for the plots and tables only: those gradients are
not charged and are never handed to the method. Coordinate methods refresh the intensity cache every `cache_every_passes`
passes. The inner geometry changes the outer iterates: the blocks are solved
inexactly (to the relative accuracy $\rho$), and the two geometries' steps make different amounts of progress on the same block.

In [ ]:
def inner_budget(cfg, outer, blocks):
    """Cap on the accepted steps of subproblem k. With the eps_k rule (`inner_relative_tolerance` set) the solve ends when its
    certified error is at most eps_k and `max_inner_budget` is only a safeguard. Without it, the fixed schedule
    t_k = t_0 + growth*floor(k/N): one more step per sweep of n coordinate updates (N=n) or per outer iteration of a
    full method (N=1), following the remark after Theorem 4.1 (t_k = k gives eps_k <= C/k)."""
    if cfg.get("inner_relative_tolerance") is not None:
        return int(cfg["max_inner_budget"])
    return int(min(cfg["max_inner_budget"], cfg["inner_budget0"]+cfg["inner_budget_growth"]*(outer//blocks)))


def stop_rule(spec, cfg):
    return dict(pg=("euclidean_gap", cfg["tol_pg"]), burg=("burg_gap", cfg["tol_burg"]),
                convex_pg=("convex_pg_residual", cfg["tol_pg"]))[spec["stop"]]


class AuditLog:
    """Online checks of every outer attempt plus a thinned table of rows for export."""

    def __init__(self, keep_first=2000, keep_every=64):
        self.keep_first, self.keep_every = keep_first, keep_every
        self.rows = []
        self.inner = []           # every accepted solve: true and certified eps_k, target, reference Delta_k, steps, status
        self.counts = dict(attempted=0, accepted=0, changed=0, zero_step=0, inner_steps=0, change_violations=0,
                           majorization_violations=0, gs_violations=0)

    def add(self, row):
        c = self.counts
        c["attempted"] += 1
        c["majorization_violations"] += int(row["violations"])
        if row["accepted"]:
            c["accepted"] += 1
            c["changed"] += int(row["actual_change"])
            c["zero_step"] += int(row["steps"] == 0)
            c["inner_steps"] += int(row["steps"])
            c["change_violations"] += int(row["target_change"] > row["target_change_cushion"])
            self.inner.append((row["eps_true"], row["eps_certified"], row["eps_target"], row["eps_reference"],
                               row["eps_achieved"], row["gap"], row["reference_certificate"], row["steps"], row["status"],
                               row["matvec_passes"]))
        if "selected_score" in row:
            c["gs_violations"] += int(row["selected_score"] != row["batch_max_score"])
        if (not row["accepted"]) or row["attempted_outer"] <= self.keep_first or row["attempted_outer"] % self.keep_every == 0:
            self.rows.append(row)

    def frame(self):
        return pd.DataFrame(self.rows)

    def inner_frame(self):
        frame = pd.DataFrame(self.inner, columns=["eps", "eps_certified", "eps_target", "eps_reference", "eps_achieved",
                                                  "online_certificate", "reference_certificate", "steps", "status",
                                                  "matvec_passes"])
        # explicit dtypes: an empty frame (a run without accepted solves) must concatenate with the others as floats
        return frame.astype({column: float for column in frame.columns if column != "status"} | {"status": str})


def initialize_run(problem, cfg, work, spec=None):
    x = problem["x0"].copy()
    s = constants = None
    status = "running"
    needs = constant_needs(spec, cfg)
    try:
        work.extra("method_setup_vectors", 3*len(x)+len(problem["y"]))
        work.charge(problem["A"].nnz, "setup")
        s = np.asarray(problem["A_csr"] @ x).ravel()+cfg["background"]
        constants = geometry_constants(problem, cfg, work, needs)
        diagnostic, cached_w = stationarity(problem, x, s, constants, cfg, work)
    except WorkLimit as error:
        status = str(error)+"_during_setup"
        work.reporting = True
        if s is None:
            work.charge(problem["A"].nnz, "diagnostics")
            s = np.asarray(problem["A_csr"] @ x).ravel()+cfg["background"]
        if constants is None:
            constants = geometry_constants(problem, cfg, work, needs)
        diagnostic, cached_w = stationarity(problem, x, s, constants, cfg, work)
    return x, s, constants, diagnostic, cached_w, status


def run_dca(problem, cfg, spec):
    method, geometry, selection = spec["name"], spec["geometry"], spec["selection"]
    work = Work(problem["A"].nnz, cfg["max_passes"])
    x, s, constants, diagnostic, cached_w, status = initialize_run(problem, cfg, work, spec)
    n = len(x)
    blocks = 1 if selection == "full" else n
    measure_key, tolerance = stop_rule(spec, cfg)
    charge_top_q(work, n)
    mask = top_q_mask(x, cfg["largest_q"])
    with work.untimed():
        history = [record_state(problem, x, s, cfg, work, method, 0, 0, "initial", "started", diagnostic)]
    current_objective = best_completed = history[-1]["objective"]
    work.extra("objective_cache_setup", 12*len(s))
    likelihood_terms = kl_div(problem["y"], s)
    audit, pending = AuditLog(), None
    outer = updates = free_attempts = 0
    rngs = dict(random=np.random.default_rng(problem["seed"]+1000033),
                batch=np.random.default_rng(problem["seed"]+2000003))
    next_check = work.passes+cfg["check_every_passes"]
    next_log = work.passes+cfg["log_every_passes"]
    next_inner_log = work.passes+cfg["log_every_passes"]
    next_cache = work.passes+cfg["cache_every_passes"]
    if status == "running" and diagnostic[measure_key] <= tolerance:
        status = "stationarity_reached"
    while status == "running":
        if cfg["max_outer"] is not None and outer >= cfg["max_outer"]:
            status = "outer_iteration_limit"
            break
        budget = inner_budget(cfg, outer, blocks)
        charged_before = work.charged
        try:
            selected, v, selected_w, selection_info = select_block(problem, x, s, mask, constants, spec, rngs, cfg, work,
                                                                   cached_w)
            target, rows = make_target(problem, x, s, v, selected, cfg, work, constants)
            if selected is None:
                inner_s, inner_w = s, cached_w
            else:
                inner_s = s[rows]
                inner_w = (np.array([selected_w]) if selected_w is not None else
                           None if cached_w is None else cached_w[selected:selected+1])

            def progress(candidate, steps):
                # Unfinished full-vector candidates are logged on their own schedule (never displacing accepted rows).
                nonlocal next_inner_log
                if cfg["record_inner"] and selected is None and work.passes >= next_inner_log:
                    with work.untimed():
                        history.append(record_state(problem, candidate["z"], candidate["s"], cfg, work, method, outer,
                            updates, "inner_candidate", "solving", info=candidate["cert"]))
                    next_inner_log = work.passes+cfg["log_every_passes"]

            state, info = solve_inner(target, geometry, budget, inner_s, inner_w, progress, final_gradient=selected is None)
            changed = not np.array_equal(state["z"], target.anchor)
            change, linear, remainder = (stable_target_change(target,
                dict(z=target.anchor, s=inner_s, grad=target.bcoef-info["anchor_w"]), state)
                if changed else (0.0, 0.0, 0.0))
            accepted = info["status"] not in ("matvec_budget", "majorization_violation")
            audit.add(dict(seed=problem["seed"], method=method, attempted_outer=outer+1,
                selected_coordinate=-1 if selected is None else selected, inner_budget=budget, accepted=accepted,
                actual_change=changed and accepted, target_change=change,
                target_change_cushion=128*FLOAT_EPS*(1+abs(linear)+abs(remainder)+abs(info["anchor_value"])),
                matvec_passes=work.passes, **selection_info,
                **{key: value for key, value in info.items() if key not in {"trace", "anchor_w", "anchor_value"}}))
            if not accepted:
                pending = dict(selected=selected, rows=rows, state=state, info=info, budget=budget)
                status = info["status"]
                break
            if selected is None and not changed:
                # The full block cannot move (every safe step is below floating-point resolution): stop, do not loop.
                status = "no_step_possible"
            if selected is None:
                x, s, cached_w = state["z"].copy(), state["s"].copy(), state["w"].copy()
                likelihood_terms = state["likelihood_terms"].copy()
                charge_top_q(work, len(x))
                mask = top_q_mask(x, cfg["largest_q"])
                work.extra("accepted_full_objective", 12*len(s)+8*len(x))
                current_objective = original_objective(x, s, problem["y"], cfg)
            else:
                work.extra("coordinate_cache_update", 8*len(rows)+12)
                old = float(x[selected])
                likelihood_difference = float(state["likelihood_terms"].sum()-likelihood_terms[rows].sum())
                x[selected], s[rows] = state["z"][0], state["s"]
                likelihood_terms[rows] = state["likelihood_terms"]
                top_difference = update_top_q_mask(x, mask, selected, old, cfg["largest_q"], work)
                current_objective += likelihood_difference+cfg["lambda_"]*(x[selected]-old)-cfg["mu"]*top_difference
                if changed:
                    cached_w = None
            outer += 1
            updates += int(changed)
            # Safeguard: an attempt that charged nothing (a block that needs no step and whose gradient was known) cannot
            # trigger the pass-based stopping check; after n such attempts in a row a check is forced (never observed).
            free_attempts = free_attempts+1 if work.charged == charged_before else 0
            best_completed = min(best_completed, current_objective)
            if selected is not None and work.passes >= next_cache:
                work.charge(problem["A"].nnz, "cache")
                s = np.asarray(problem["A_csr"] @ x).ravel()+cfg["background"]
                work.extra("objective_cache_refresh", 20*len(s)+8*len(x))
                likelihood_terms = kl_div(problem["y"], s)
                current_objective = original_objective(x, s, problem["y"], cfg)
                best_completed = min(best_completed, current_objective)
                cached_w = None
                next_cache = work.passes+cfg["cache_every_passes"]
            checked = selected is None or work.passes >= next_check or free_attempts >= n
            logged = checked or work.passes >= next_log or outer == 1
            if checked:
                # Stopping check: charged to the method (free for full methods, whose gradient is cached).
                diagnostic, cached_w = stationarity(problem, x, s, constants, cfg, work, cached_w)
                next_check = work.passes+cfg["check_every_passes"]
                free_attempts = 0
                if diagnostic[measure_key] <= tolerance:
                    status = "stationarity_reached"
                reported = diagnostic
            if logged:
                with work.untimed():
                    if not checked:
                        # Reporting only: not charged, and the gradient is not handed to the method.
                        reported, _ = stationarity(problem, x, s, constants, cfg, work, cached_w, category="diagnostics")
                    history.append(record_state(problem, x, s, cfg, work, method, outer, updates,
                        "accepted", "accepted", reported, info, current_objective))
                next_log = work.passes+cfg["log_every_passes"]
        except WorkLimit as error:
            status = str(error)
            break
        except FloatingPointError as error:
            status = "numerical_failure: "+str(error)
            break
    work.reporting, work.stopped = True, time.perf_counter()
    work.charge(problem["A"].nnz, "diagnostics")
    direct_s = np.asarray(problem["A_csr"] @ x).ravel()+cfg["background"]
    intensity_error = float(np.max(np.abs(direct_s-s)))
    work.extra("final_cache_validation", 12*len(s)+8*len(x))
    objective_error = abs(current_objective-original_objective(x, direct_s, problem["y"], cfg))
    s = direct_s
    diagnostic, cached_w = stationarity(problem, x, s, constants, cfg, work)
    pending_summary = None
    if pending is not None:
        px, ps = x.copy(), s.copy()
        if pending["selected"] is None:
            px, ps = pending["state"]["z"].copy(), pending["state"]["s"].copy()
        else:
            px[pending["selected"]] = pending["state"]["z"][0]
            ps[pending["rows"]] = pending["state"]["s"]
        pending_summary = record_state(problem, px, ps, cfg, work, method, outer, updates,
            "inner_candidate", status, info=pending["info"])
        pending_summary["inner_budget"] = pending["budget"]
    final = record_state(problem, x, s, cfg, work, method, outer, updates, "final_accepted", status, diagnostic)
    history.append(final)
    best_completed = min(best_completed, final["objective"])
    return dict(spec=spec, history=pd.DataFrame(history), audit=audit.frame(), audit_counts=audit.counts,
        inner=audit.inner_frame(),
        final_x=x, final_intensity=s, work=work, pending=pending, pending_summary=pending_summary,
        status=status, best_completed=best_completed, objective_cache_error=objective_error,
        intensity_cache_error=intensity_error, constants=constants)

## Convex Baseline (FISTA)

FISTA minimises the convex problem $f(x)+\lambda\sum_jx_j$ ($\mu=0$) with projected accelerated gradients, the same
upper-model backtracking test (scalar Lipschitz estimate), function-value restart, gradient restart and a restart whenever the
extrapolation would leave the box. Per iteration it needs one transpose product (the gradient at the extrapolated point $y_k$)
and one forward product per trial; the intensity of $y_k=x_k+\beta_k(x_k-x_{k-1})$ follows from linearity,
$s(y_k)=s_k+\beta_k(s_k-s_{k-1})$, so no forward product is spent on it. It stops when its own convex projected-gradient
residual at $x_k$ is at most `tol_pg`; that residual needs the gradient at $x_k$, which is free when $y_k=x_k$ and otherwise
costs one charged product every `check_every_passes` (as for the coordinate methods). Its iterates are also evaluated under the
nonconvex objective and both DC measures every `log_every_passes` passes, with uncharged reporting products.

In [ ]:
def run_fista(problem, cfg, spec):
    method = spec["name"]
    work = Work(problem["A"].nnz, cfg["max_passes"])
    x, s, constants, diagnostic, cached_w, setup_status = initialize_run(problem, cfg, work, spec)
    target, _ = make_target(problem, x, s, np.zeros(len(x)), None, cfg, work, constants)
    lower, upper = cfg["xmin"], cfg["xmax"]
    x_state = target.evaluate(x, s, cached_w)          # gradient at x0 from the initial check (no new product)
    with work.untimed():
        history = [record_state(problem, x, s, cfg, work, method, 0, 0, "initial", "started", diagnostic, x_state["cert"])]
    best_completed = history[-1]["objective"]
    momentum, scale = 1.0, None
    y, y_state = x, x_state                            # extrapolated point; the same state object when y == x
    iterations = 0
    next_check = work.passes+cfg["check_every_passes"]
    next_log = work.passes+cfg["log_every_passes"]
    status = "iteration_limit" if setup_status == "running" else setup_status
    cap = cfg["fista_max_iterations"] if cfg["max_outer"] is None else min(cfg["max_outer"], cfg["fista_max_iterations"])
    try:
        for _ in range(cap):
            if setup_status != "running":
                break
            target.complete(y_state)                   # gradient at y (one transpose product unless known)
            if x_state["w"] is None and work.passes >= next_check:
                target.complete(x_state, category="stopping")
            if x_state["w"] is not None:
                # Any check, free (y = x) or paid, restarts the timer, as for the coordinate methods.
                next_check = work.passes+cfg["check_every_passes"]
                residual = float(np.linalg.norm(projected_residual(x, x_state["grad"], lower, upper)))
                work.extra("convex_stopping_vectors", 10*len(x))
                if residual <= cfg["tol_pg"]:
                    status = "stationarity_reached"
                    break
            anchor = y_state
            scale = (target.rowsum_curvature(anchor) if scale is None
                     else max(cfg["positive_scale_floor"], scale*cfg["fista_scale_shrink"]))
            accepted = False
            for _ in range(cfg["fista_max_backtracks"]):
                work.extra("fista_proposal_vectors", 12*len(x))
                proposal = mirror_proposal(y, anchor["grad"], scale, "euclidean", lower, upper)
                candidate = target.evaluate(proposal, gradient=False)       # one forward product
                change, linear, remainder = stable_target_change(target, anchor, candidate)
                delta = proposal-y
                penalty = 0.5*scale*float(delta @ delta)
                work.extra("fista_model_norm", 20*len(x)+6*len(s))
                if remainder <= penalty+upper_model_cushion(target, anchor["s"], candidate["s"], remainder, penalty):
                    accepted = True
                    break
                work.events["rejected_trials"] += 1
                scale *= cfg["fista_backtracking_factor"]
            if not accepted:
                status = "backtracking_limit"
                break
            if (candidate["value"] > x_state["value"]+128*FLOAT_EPS*(1+abs(x_state["value"]))
                    and not np.array_equal(y, x)):
                work.extra("fista_restart", 3*len(x))
                work.events["restarts"] += 1
                y, y_state, momentum = x, x_state, 1.0
                continue
            new_momentum = 0.5*(1+np.sqrt(1+4*momentum*momentum))
            beta = (momentum-1)/new_momentum
            extrapolated = proposal+beta*(proposal-x)
            if (float((y-proposal) @ (proposal-x)) > 0 or np.any(extrapolated < lower)
                    or np.any(extrapolated > upper)):
                beta, new_momentum = 0.0, 1.0
            work.extra("fista_momentum", 20*len(x))
            previous_s = x_state["s"]
            x, x_state = proposal, candidate
            if beta == 0.0:
                y, y_state = x, x_state
            else:
                work.events["extrapolations"] += 1
                y = extrapolated
                extrapolated_s = x_state["s"]+beta*(x_state["s"]-previous_s)     # A is linear: no forward product
                if extrapolated_s.min() > 0:
                    y_state = target.evaluate(y, intensity=extrapolated_s, gradient=False)
                else:
                    work.events["extrapolation_products"] += 1       # roundoff fallback: one forward product
                    y_state = target.evaluate(y, gradient=False)
            s = x_state["s"]
            momentum = new_momentum
            iterations += 1
            work.events["inner_steps"] += 1
            work.extra("original_objective_baseline", 12*len(s)+8*len(x))
            best_completed = min(best_completed, original_objective(x, s, problem["y"], cfg))
            if work.passes >= next_log or iterations == 1:
                with work.untimed():
                    # Reporting only: an uncharged gradient when the one at x is not known; never handed to FISTA.
                    w_report = x_state["w"] if x_state["w"] is not None else target.tmv(target.y/s, "diagnostics")
                    diagnostic = dc_measures(problem, x, w_report, constants, cfg, work)
                    history.append(record_state(problem, x, s, cfg, work, method, iterations, iterations,
                        "baseline", "iterating", diagnostic, target.certificate(x, w_report)))
                next_log = work.passes+cfg["log_every_passes"]
    except WorkLimit as error:
        status = str(error)
    except FloatingPointError as error:
        status = "numerical_failure: "+str(error)
    work.reporting, work.stopped = True, time.perf_counter()
    work.charge(problem["A"].nnz, "diagnostics")
    direct_s = np.asarray(problem["A_csr"] @ x).ravel()+cfg["background"]
    intensity_error = float(np.max(np.abs(direct_s-s)))
    state = target.evaluate(x, direct_s)
    diagnostic, _ = stationarity(problem, x, direct_s, constants, cfg, work, state["w"])
    final = record_state(problem, x, direct_s, cfg, work, method, iterations, iterations,
        "final_baseline", status, diagnostic, state["cert"])
    history.append(final)
    return dict(spec=spec, history=pd.DataFrame(history), audit=pd.DataFrame(), audit_counts={},
        final_x=x, final_intensity=direct_s, work=work, pending=None, pending_summary=None, status=status,
        best_completed=min(best_completed, final["objective"]), objective_cache_error=0.0,
        intensity_cache_error=intensity_error, constants=constants)


def run_method(problem, cfg, spec):
    return run_fista(problem, cfg, spec) if spec["selection"] == "fista" else run_dca(problem, cfg, spec)

## Metrics, Summaries and Figures

* **Objective reference** $F^\star$: the smallest objective of any completed accepted state on the same seed (empirical, not a
  certified global optimum; nonconvex problem). Unfinished inner candidates never define it.
* **Solution recovery:** relative error $\Vert x-x^\star\Vert_2/\Vert x^\star\Vert_2$.
* **Support recovery:** top-$s$ support F1 (fraction of true spikes among the $s$ largest entries; $s=25$), and thresholded
  support precision/recall/F1 with the estimate $\{j:x_j>0.01\max_kx_k\}$.
* **Stationarity:** Burg gap and Euclidean gap (both computed for every method at every stopping check and every `log_every_passes`).
* **Equal-work comparison:** every run is sampled at fixed MatVec levels (the last accepted state at or below the level;
  stopped runs keep their final state), so methods are compared at the same cost regardless of when they stopped.
* Figures: medians with central 68% bands (16th–84th percentile) for the heavy-tailed quantities — the Burg gap (best-so-far
  values, matching the $\min_{k\le K}$ form of Theorem 4.2) and the relative error (log scale) — and means ± one population
  standard deviation for the bounded support metrics. The Euclidean gap and the objective gap $F-F^\star$ are kept in the
  tables (the objective reference is empirical and the Euclidean gap is not the stopping measure of the Burg methods) but
  are not plotted.
  Colour encodes the block rule, line style the inner geometry (solid Burg, dashed Euclidean, dotted FISTA).
  The two methods this experiment is about, `B-GS(Burg)` and `B-Full`, are drawn with thick lines.
  Log-scale panels draw values below $10^{-16}$ (roundoff) at $10^{-16}$.
  Bands describe the tested seeds; they are not confidence intervals.

In [ ]:
import concurrent.futures
import multiprocessing

PLOT_COLORS = {"random": "#2a78d6", "gs_batch:pg": "#eb6834", "gs_batch:burg": "#1baf7a",
               "full": "#eda100", "fista": "#52514e"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
GAP_PLOT_FLOOR = 1e-16   # log-scale panels show values below this (roundoff) at the floor
HIGHLIGHT_METHODS = ("B-GS(Burg)", "B-Full")
KEY_METHODS = ["B-GS(Burg)", "B-Full", "E-GS(PG)", "E-Full"]   # the two block sizes in the two geometries


def method_style(spec):
    key = spec["selection"] if spec["selection"] != "gs_batch" else "gs_batch:"+spec["score"]
    linestyle = ":" if spec["selection"] == "fista" else ("-" if spec["geometry"] == "burg" else "--")
    highlight = spec["name"] in HIGHLIGHT_METHODS
    return dict(color=PLOT_COLORS[key], linestyle=linestyle, linewidth=3.0 if highlight else 1.4,
                alpha=1.0 if highlight else 0.85, zorder=3 if highlight else 2)


def run_comparison(cfg, methods=METHOD_SPECS, verbose=True, workers=1):
    """Run every method on every seed. With workers > 1 (and the 'fork' start method, e.g. Linux) the independent runs are
    spread over worker processes; each run is sequential and deterministic, so the results do not depend on `workers`."""
    problems = {seed: prepare_problem(cfg, seed) for seed in cfg["seeds"]}
    jobs = [(seed, spec) for seed in cfg["seeds"] for spec in methods]
    finished = {}
    if workers > 1 and "fork" in multiprocessing.get_all_start_methods():
        context = multiprocessing.get_context("fork")
        with concurrent.futures.ProcessPoolExecutor(workers, mp_context=context) as pool:
            futures = {pool.submit(run_method, problems[seed], cfg, spec): (seed, spec["name"]) for seed, spec in jobs}
            for future in concurrent.futures.as_completed(futures):
                finished[futures[future]] = future.result()
    else:
        for seed, spec in jobs:
            finished[(seed, spec["name"])] = run_method(problems[seed], cfg, spec)
    runs = {(seed, spec["name"]): finished[(seed, spec["name"])] for seed, spec in jobs}   # fixed order
    if verbose:
        for (seed, name), run in runs.items():
            end = run["history"].iloc[-1]
            print(f"seed={seed} {name:11s} status={run['status']:24s} outer={int(end.outer_iteration):8d} "
                  f"passes={end.matvec_passes:8.1f} burg={end.burg_gap:.2e} eucl={end.euclidean_gap:.2e} "
                  f"relerr={end.relative_x_error:.4f} topF1={end.topk_support_f1:.2f} "
                  f"time={end.wall_clock_time:.1f}s", flush=True)
    results = pd.concat([run["history"] for run in runs.values()], ignore_index=True)
    reference = {seed: min(run["best_completed"] for (s, _), run in runs.items() if s == seed)
                 for seed in cfg["seeds"]}
    results["empirical_best_accepted"] = results.seed.map(reference)
    results["objective_gap"] = results.objective-results.empirical_best_accepted
    pending = pd.DataFrame([run["pending_summary"] for run in runs.values() if run["pending_summary"] is not None])
    audits = [run["audit"] for run in runs.values() if not run["audit"].empty]
    audit_counts = pd.DataFrame([dict(seed=seed, method=method, **run["audit_counts"])
                                 for (seed, method), run in runs.items() if run["audit_counts"]])
    return dict(config=copy.deepcopy(cfg), methods=[m["name"] for m in methods], runs=runs, problems=problems,
                results=results, audit=pd.concat(audits, ignore_index=True) if audits else pd.DataFrame(),
                audit_counts=audit_counts, pending=pending, reference=reference)


SUMMARY_COLUMNS = ["seed", "method", "status", "outer_iteration", "actual_updates", "matvec_passes", "wall_clock_time",
                   "objective_gap", "burg_gap", "euclidean_gap", "convex_pg_residual", "relative_x_error", "topk_support_f1",
                   "support_precision", "support_recall", "support_f1", "estimated_support_size", "convex_certificate"]


def comparison_summary(comparison):
    final = comparison["results"].query("phase in ['final_accepted', 'final_baseline']")
    return final[SUMMARY_COLUMNS].reset_index(drop=True)


def _mean_std(values):
    return f"{np.mean(values):.4f} ± {np.std(values):.4f}"


def method_summary(summary, methods):
    rows = []
    for method in methods:
        group = summary[summary.method == method]
        spec = SPEC_BY_NAME.get(method, {})
        statuses = group.status.value_counts()
        rows.append({
            "#": spec.get("number"), "method": method,
            "stopping measure": dict(pg="Euclidean gap", burg="Burg gap", convex_pg="convex PG").get(spec.get("stop"), "-"),
            "stopped by tolerance": f"{int(statuses.get('stationarity_reached', 0))}/{len(group)}",
            "other statuses": ", ".join(f"{k}×{v}" for k, v in statuses.items() if k != "stationarity_reached") or "-",
            "MatVec passes (median)": float(group.matvec_passes.median()),
            "wall s (median)": float(group.wall_clock_time.median()),
            "Burg gap (median)": float(group.burg_gap.median()),
            "Euclidean gap (median)": float(group.euclidean_gap.median()),
            "F-F* (median)": float(group.objective_gap.median()),
            "rel. error (mean ± std)": _mean_std(group.relative_x_error),
            "top-s F1 (mean ± std)": _mean_std(group.topk_support_f1),
            "precision (mean)": float(group.support_precision.mean()),
            "thresholded F1 (mean)": float(group.support_f1.mean()),
        })
    return pd.DataFrame(rows)


def at_work_levels(comparison, levels):
    """Last accepted state at or below each MatVec level, with best-so-far gaps up to that level."""
    data = comparison["results"].query("accepted_state").sort_values("matvec_passes", kind="stable")
    rows = []
    for (seed, method), run in data.groupby(["seed", "method"], sort=False):
        costs = run.matvec_passes.to_numpy()
        for level in levels:
            upto = run[costs <= level]
            if upto.empty:
                continue
            last = upto.iloc[-1]
            rows.append(dict(seed=seed, method=method, level=level, relative_x_error=last.relative_x_error,
                             topk_support_f1=last.topk_support_f1, support_precision=last.support_precision,
                             objective_gap=last.objective_gap, burg_gap=float(np.nanmin(upto.burg_gap.to_numpy())),
                             euclidean_gap=float(np.nanmin(upto.euclidean_gap.to_numpy()))))
    return pd.DataFrame(rows)


def equal_work_table(comparison, levels, metric, statistic="median"):
    table = at_work_levels(comparison, levels)
    pivot = table.pivot_table(index="method", columns="level", values=metric, aggfunc=statistic)
    return pivot.reindex([m for m in comparison["methods"] if m in pivot.index])


def ranking_table(comparison, targets, methods):
    """Rank methods by the MatVec cost to reach each target (more seeds reached first, then fewer median passes).

    Final states are not used for ranking: after convergence they differ only at roundoff level, and methods stop on
    different measures. Ties share the best rank.
    """
    table = pd.DataFrame(index=methods)
    for label, metric, threshold, larger in targets:
        costs = target_costs(comparison, metric, threshold, larger)
        stats = costs.groupby("method").agg(reached=("reached", "sum"), passes=("passes", "median")).reindex(methods)
        # Medians are compared at the displayed resolution (whole passes), so equal displayed values tie.
        keys = list(zip(-stats.reached.to_numpy(), np.round(stats.passes.fillna(np.inf).to_numpy())))
        table[label+": median passes"] = [f"{p:.0f}" if np.isfinite(p) else "-" for p in stats.passes]
        table[label+": seeds"] = [f"{int(r)}/{len(costs.seed.unique())}" for r in stats.reached]
        table[label+": rank"] = [1+sum(other < key for other in keys) for key in keys]
    return table.reset_index().rename(columns={"index": "method"})


def target_costs(comparison, metric="relative_x_error", threshold=0.01, larger_is_better=False):
    rows = []
    accepted = comparison["results"].query("accepted_state")
    for (seed, method), group in accepted.groupby(["seed", "method"], sort=False):
        reached = group[group[metric] >= threshold] if larger_is_better else group[group[metric] <= threshold]
        hit = reached.sort_values("matvec_passes", kind="stable")
        row = None if hit.empty else hit.iloc[0]
        rows.append(dict(seed=seed, method=method, metric=metric, threshold=threshold, reached=row is not None,
            passes=np.nan if row is None else row.matvec_passes,
            seconds=np.nan if row is None else row.wall_clock_time))
    return pd.DataFrame(rows)


def target_cost_table(comparison, targets):
    tables = []
    for label, metric, threshold, larger in targets:
        costs = target_costs(comparison, metric, threshold, larger)
        agg = costs.groupby("method", sort=False).agg(reached=("reached", "sum"), runs=("reached", "size"),
                                                       median_passes=("passes", "median"))
        agg[label] = [f"{int(r)}/{int(n)} @ {p:.0f}" if r else f"0/{int(n)}"
                      for r, n, p in zip(agg.reached, agg.runs, agg.median_passes)]
        tables.append(agg[[label]])
    table = pd.concat(tables, axis=1)
    return table.reindex([m for m in comparison["methods"] if m in table.index])


def assert_comparison(comparison):
    cfg = comparison["config"]
    for (seed, method), run in comparison["runs"].items():
        spec, x, problem = run["spec"], run["final_x"], comparison["problems"][seed]
        assert np.all((x >= cfg["xmin"]) & (x <= cfg["xmax"]))
        assert np.allclose(problem["A"] @ x+cfg["background"], run["final_intensity"], rtol=1e-11, atol=1e-10)
        final = run["history"].iloc[-1]
        actual_objective = original_objective(x, run["final_intensity"], problem["y"], cfg)
        assert abs(final.objective-actual_objective) <= 1e-10*(1+abs(actual_objective))
        assert run["objective_cache_error"] <= 1e-7*(1+abs(actual_objective))
        assert run["work"].passes <= cfg["max_passes"]+1e-8
        if run["status"] == "stationarity_reached":
            key, tolerance = stop_rule(spec, cfg)
            assert final[key] <= tolerance*(1+1e-6)+1e-12, (seed, method, key, final[key])
        if run["audit_counts"]:
            counts = run["audit_counts"]
            assert counts["accepted"] == int(final.outer_iteration)
            assert counts["majorization_violations"] == 0 and counts["change_violations"] == 0
            assert counts["gs_violations"] == 0
            accepted = run["history"].query("accepted_state").objective.to_numpy()
            assert np.all(np.diff(accepted) <= 1e-7*(1+np.abs(accepted[:-1])))
        good = run["audit"].query("accepted") if not run["audit"].empty else run["audit"]
        if len(good):
            assert (good.steps <= good.inner_budget).all()
            assert (good.target_change <= good.target_change_cushion).all()
        inner = run.get("inner")
        if inner is not None and len(inner) and cfg.get("inner_relative_tolerance") is not None:
            # eps_k rule: every solve that stopped by the rule is certified within its target, every target is the rule's
            # value, and no solve ended by the fixed-schedule exit.
            reached = inner[inner.status == "inner_tolerance"]
            assert (reached.eps_certified <= reached.eps_target).all(), (seed, method)
            assert (reached.eps <= reached.eps_certified+1e-9).all(), (seed, method)
            stepped = inner[inner.steps > 0]
            expected = np.maximum(cfg.get("inner_absolute_tolerance", 0.0),
                                  cfg["inner_relative_tolerance"]*np.maximum(stepped.eps_reference, stepped.eps_achieved))
            assert np.allclose(stepped.eps_target, expected, rtol=1e-12, atol=0), (seed, method)
            assert (inner.steps <= cfg["max_inner_budget"]).all()
            if spec["selection"] == "gs_batch":
                assert (good.batch_size == cfg["gs_batch_size"]).all()
                assert np.array_equal(good.selected_score.to_numpy(), good.batch_max_score.to_numpy())
    return dict(runs=len(comparison["runs"]),
                accepted_blocks=int(comparison["audit_counts"].accepted.sum())
                if not comparison["audit_counts"].empty else 0,
                inner_steps=int(comparison["audit_counts"].inner_steps.sum())
                if not comparison["audit_counts"].empty else 0)


def resampled_curves(group, metric, axis, best_so_far=False):
    records = []
    for _, run in group.groupby("seed", sort=False):
        points = run[np.isfinite(run[metric])].sort_values(axis, kind="stable")
        if len(points):
            value = points[metric].to_numpy()
            records.append((points[axis].to_numpy(), np.minimum.accumulate(value) if best_so_far else value))
    if not records:
        return None, None
    start = max(cost[0] for cost, _ in records)
    stop = max(cost[-1] for cost, _ in records)
    grid = np.geomspace(max(start, 1e-3), max(stop, start*1.0001), 300)
    curves = np.vstack([value[np.maximum(np.searchsorted(cost, grid, side="right")-1, 0)]
                        for cost, value in records])
    return grid, curves


def _style_axis(ax, title, xlabel):
    ax.set_title(title, fontsize=10, color=INK, loc="left")
    ax.set_xlabel(xlabel, fontsize=9, color=INK_2)
    ax.grid(True, color=GRID, linewidth=0.6, linestyle="-")
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#bdbcb7")
    ax.tick_params(colors=INK_2, labelsize=8)


def plot_comparison(comparison, axis="matvec_passes", methods=None, panels=None, titles=None):
    data = comparison["results"].query("accepted_state")
    cfg = comparison["config"]
    xlabel = dict(matvec_passes="MatVec-pass equivalents", wall_clock_time="wall-clock seconds").get(axis, axis)
    specs = [SPEC_BY_NAME[m] for m in (methods or comparison["methods"])]
    if panels is None:
        panels = [
            [("burg_gap", "Burg gap (best so far, median, 68% band)", True, "median", "log", cfg["tol_burg"]),
             ("relative_x_error", "Relative error ||x - x*|| / ||x*|| (median, 68% band)", False, "median", "log", None)],
            [("topk_support_f1", "Top-s support F1 (mean ± std)", False, "mean", "linear", None),
             ("support_precision", "Thresholded support precision (mean ± std)", False, "mean", "linear", None)]]
    titles = titles or ["Stationarity and solution error over seeds",
                        "Support recovery over seeds"]
    figures = []
    for number, row in enumerate(panels):
        fig, axes = plt.subplots(1, len(row), figsize=(5.2*len(row), 4.4), constrained_layout=True)
        axes = np.atleast_1d(axes)
        fig.patch.set_facecolor("#fcfcfb")
        for ax, (metric, title, best, statistic, scale, tolerance) in zip(axes, row):
            ax.set_facecolor("#fcfcfb")
            for spec in specs:
                grid, curves = resampled_curves(data[data.method == spec["name"]], metric, axis, best)
                if curves is None:
                    continue
                if scale == "log":
                    curves = np.maximum(curves, GAP_PLOT_FLOOR)
                style = method_style(spec)
                if statistic == "median":
                    center = np.median(curves, axis=0)
                    low, high = np.quantile(curves, [0.16, 0.84], axis=0)
                else:
                    center, spread = curves.mean(axis=0), curves.std(axis=0)
                    low, high = center-spread, center+spread
                    if scale == "linear":   # bounded metrics in [0, 1]
                        low, high = np.clip(low, 0.0, 1.0), np.clip(high, 0.0, 1.0)
                ax.plot(grid, center, label=spec["name"], solid_capstyle="round", **style)
                if scale == "log":
                    low = np.maximum(low, GAP_PLOT_FLOOR)
                ax.fill_between(grid, low, high, color=style["color"], alpha=0.10, linewidth=0)
            if tolerance is not None:
                ax.axhline(tolerance, color="#8a8984", linewidth=0.9, linestyle="-")
                ax.annotate("tolerance", xy=(1.0, tolerance), xycoords=("axes fraction", "data"),
                            xytext=(-4, 3), textcoords="offset points", ha="right", va="bottom",
                            fontsize=8, color=INK_2)
            ax.set_xscale("log")
            if scale == "symlog":
                ax.set_yscale("symlog", linthresh=1e-6)
            elif scale == "log":
                ax.set_yscale("log")
            else:
                ax.set_ylim(-0.03, 1.05)
            _style_axis(ax, title, xlabel)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=min(8, max(4, len(labels))), frameon=False, fontsize=9,
                   bbox_to_anchor=(0.5, -0.09 if len(labels) <= 4*len(row) else -0.16), labelcolor=INK)
        fig.suptitle(titles[number], fontsize=11, color=INK, x=0.01, ha="left")
        figures.append(fig)
    return figures


def plot_headline(comparison, axis="matvec_passes"):
    """The four methods that separate block size from geometry: relative error and Burg gap."""
    cfg = comparison["config"]
    panels = [[("relative_x_error", "Relative error ||x - x*|| / ||x*|| (median, 68% band)", False, "median", "log", None),
               ("burg_gap", "Burg gap (best so far, median, 68% band)", True, "median", "log", cfg["tol_burg"])]]
    return plot_comparison(comparison, axis, methods=[m for m in KEY_METHODS if m in comparison["methods"]],
                           panels=panels, titles=["Coordinate blocks versus full vector, Burg versus Euclidean geometry"])[0]


SPEEDUPS = [("block size, Burg: B-Full / B-GS(Burg)", "B-Full", "B-GS(Burg)"),
            ("block size, Euclidean: E-Full / E-GS(PG)", "E-Full", "E-GS(PG)"),
            ("geometry, coordinate: E-GS(PG) / B-GS(Burg)", "E-GS(PG)", "B-GS(Burg)"),
            ("geometry, full: E-Full / B-Full", "E-Full", "B-Full")]


def headline_table(comparison, targets, methods=KEY_METHODS, speedups=SPEEDUPS):
    """Median passes to each target for the key methods and per-seed cost ratios (slower/faster).

    A seed on which the slower method misses the target enters with the budget as its cost, so that ratio (and the median, if
    any such seed is present) is a lower bound, marked with >=. Seeds on which the faster method misses are left out when the
    slower one misses too; if the slower one reaches the target there, no ratio is reported (n/a).
    """
    budget, seeds = comparison["config"]["max_passes"], comparison["config"]["seeds"]
    rows = []
    for label, metric, threshold, larger in targets:
        costs = target_costs(comparison, metric, threshold, larger).set_index(["method", "seed"])
        row = {"target": label}
        for method in methods:
            group = costs.loc[method]
            reached = int(group.reached.sum())
            row[method] = f"{group.passes.median():.0f} ({reached}/{len(seeds)})" if reached else f"- (0/{len(seeds)})"
        for name, slow, fast in speedups:
            ratios, bound, undefined = [], False, False
            for seed in seeds:
                quick, other = costs.loc[(fast, seed)], costs.loc[(slow, seed)]
                if not quick.reached:
                    undefined |= bool(other.reached)
                    continue
                ratios.append((other.passes if other.reached else budget)/quick.passes)
                bound |= not other.reached
            row[name] = ("n/a" if undefined or not ratios else
                         ("≥ " if bound else "")+f"{np.median(ratios):.1f}×")
        rows.append(row)
    return pd.DataFrame(rows).set_index("target")


def curvature_looseness(problem, cfg, x):
    """Safe constant over local diagonal curvature h_j(x)=sum_i y_i A_ij^2/(Ax+b)_i^2 for each coordinate (uncharged).

    Burg: L_j/(x_j^2 h_j); Euclidean coordinate: W_j/h_j with the block box bound for the offset frozen at x;
    Euclidean full vector: the row-sum box bound A^T(y*A1/(b+x_min*A1)^2) over h_j.
    """
    A, y, b, xmin = problem["A"], problem["y"], cfg["background"], cfg["xmin"]
    x = np.maximum(np.asarray(x, float), xmin)
    intensity = np.asarray(A @ x).ravel()+b
    rows, vals = problem["col_rows"], problem["col_vals"]         # padded columns, padding has value 0
    weights = y[rows]*vals**2
    local = (weights/intensity[rows]**2).sum(axis=1)
    offset = intensity[rows]-vals*x[:, None]                     # r = Ax + b - A_:j x_j, row by row
    with np.errstate(divide="ignore", invalid="ignore"):
        coordinate = np.where(vals > 0, weights/(offset+vals*xmin)**2, 0.0).sum(axis=1)
        support = A.copy()
        support.data = np.ones_like(support.data)
        burg = np.asarray(support.T @ y).ravel()
        row_sum = np.asarray(A @ np.ones(A.shape[1])).ravel()
        full = np.asarray(problem["AT"] @ (y*row_sum/(b+xmin*row_sum)**2)).ravel()
        return pd.DataFrame(dict(burg=burg/(x*x*local), euclidean_coordinate=coordinate/local,
                                 euclidean_full=full/local))


def looseness_table(cfg):
    records = []
    for seed in cfg["seeds"]:
        problem = prepare_problem(cfg, seed)
        spikes = np.zeros(problem["A"].shape[1], dtype=bool)
        spikes[problem["support"]] = True
        for label, x, mask in (("start x0, spikes", problem["x0"], spikes),
                               ("start x0, background pixels", problem["x0"], ~spikes),
                               ("truth x*, spikes", problem["x_star"], spikes)):
            ratios = curvature_looseness(problem, cfg, x)[mask]
            records.append(dict(seed=seed, point=label, **ratios.median().to_dict()))
    table = pd.DataFrame(records).groupby("point", sort=False)[["burg", "euclidean_coordinate", "euclidean_full"]].median()
    return table.rename(columns=dict(burg="Burg L_j / (x_j^2 h_j)", euclidean_coordinate="Euclidean block W_j / h_j",
                                     euclidean_full="Euclidean full bound / h_j"))

## Independent Validation

The checks below use analytic optima, finite differences, independent scalar minimisation, dense linear algebra and SciPy
reference optimisers (`brentq`, `L-BFGS-B`, `SLSQP`, `minimize_scalar`). These references appear only here; they are not the
implemented solvers. The checks that matter most for this notebook are the ones behind "no line search": the safe weights of
both geometries majorise the exact likelihood remainder for every pair of points in the box (scalar blocks with their frozen
offset and full vectors), every solver step is therefore a descent step, and the MatVec charge of a safe solve is exactly
one forward product per step plus one transpose product per gradient. For the backtracking of this version (both geometries), the checks
verify that every accepted step satisfies the geometry's upper model and decreases $Q_k$, that every trial (accepted or rejected) and the
curvature estimate are charged, that no safe constant is formed, that the first weight is the local curvature in the geometry's metric, and that a long
backtracking solve of either geometry reaches the block minimiser. For the $\varepsilon_k$ rule of this version, the checks
verify on a scalar block that the solve stops with a certified error bound at most $\varepsilon_k$, that the true error
(against an independent `brentq` minimiser, computed here in the check) is below the certified bound and equals the
reporting-only `eps_true` to roundoff, that $\Delta_k$ equals an independent minimisation of the Burg model with $L_j$, that
exactly one transpose product per step (the gradient behind the certificate) and nothing else is charged while the
reporting reference is tallied uncharged in its own ledger category, that tighter tolerances give longer and more accurate
solves, that a block started at its minimiser ends at once, and that an absolute floor acts as an absolute tolerance; on the
small problem, that the certified bound dominates the true error of every accepted solve, that the solves stopped by the
rule have certified and true errors within their targets, that for `B-GS(Burg)` $\Delta_k$ equals the selected
Gauss–Southwell score, that no coordinate solve ends by the step cap, and that the full-block reference minimum agrees with
an independent dense L-BFGS-B minimisation without charging anything.

In [ ]:
def run_validation(f):
    import numpy as np
    import scipy.sparse as sp
    from scipy.optimize import brentq, minimize, minimize_scalar

    rng = np.random.default_rng(31827)
    cfg = dict(f.CONFIG_PSF, xmin=0.05, xmax=12.0, max_passes=1e9, backtracking_geometries=(),
               inner_relative_tolerance=None)                          # fixed-step unit tests (no eps_k rule)
    checks = dict(target_gradient_cases=0, solver_cases=0, safe_steps_checked=0, safe_constant_pairs=0,
                  dual_bound_cases=0, minimiser_bound_cases=0, gap_model_cases=0, tie_cases=0)
    worst_gradient_error = worst_solver_error = 0.0

    def make(B, y, r, c, a, local=cfg):
        scalar = np.ndim(B) == 1
        matrix = np.asarray(B, float) if scalar else sp.csr_matrix(B)
        work = f.Work(max(matrix.size if scalar else matrix.nnz, 1), local["max_passes"])
        return f.FrozenTarget(matrix, np.asarray(y), np.asarray(r), np.atleast_1d(c), np.atleast_1d(a),
                              local, work, scalar=scalar)

    def direct(B, y, r, c, z):
        matrix = np.asarray(B).reshape(-1, len(z))
        s = r+matrix @ z
        positive = y > 0
        value = float(np.sum(s-y)+np.sum(y[positive]*np.log(y[positive]/s[positive]))+c @ z)
        gradient = matrix.T @ (1-y/s)+c
        return value, gradient

    for dimension in (1, 3):
        for seed in range(4):
            B = rng.uniform(0.1, 1.0, (5, dimension))
            y = rng.poisson(np.linspace(0, 15, 5)).astype(float)
            r, c = rng.uniform(0.2, 0.8, 5), rng.uniform(0, 0.3, dimension)
            a = rng.uniform(0.3, 2, dimension)
            matrix = B[:, 0] if dimension == 1 else B
            block = make(matrix, y, r, c, a)
            state = block.evaluate(a)
            value, gradient = direct(B, y, r, c, a)
            assert abs(state["value"]-value) < 1e-11*(1+abs(value))
            assert np.allclose(state["grad"], gradient, rtol=1e-12, atol=1e-12)
            finite = np.empty(dimension)
            for j in range(dimension):
                step = 1e-5*max(1.0, abs(a[j]))
                plus, minus = a.copy(), a.copy()
                plus[j] += step
                minus[j] -= step
                finite[j] = (direct(B, y, r, c, plus)[0]-direct(B, y, r, c, minus)[0])/(2*step)
            error = float(np.max(np.abs(finite-gradient)))
            worst_gradient_error = max(worst_gradient_error, error)
            assert np.allclose(finite, gradient, rtol=2e-7, atol=2e-7)
            checks["target_gradient_cases"] += 1
            if dimension == 1:
                derivative = lambda z: float(direct(B, y, r, c, np.array([z]))[1][0])
                optimum = (cfg["xmin"] if derivative(cfg["xmin"]) >= 0 else cfg["xmax"]
                           if derivative(cfg["xmax"]) <= 0 else brentq(derivative, cfg["xmin"], cfg["xmax"], xtol=1e-13))
                reference_x = np.array([optimum])
            else:
                reference = minimize(lambda z: direct(B, y, r, c, z), a, jac=True, method="L-BFGS-B",
                    bounds=[(cfg["xmin"], cfg["xmax"])]*dimension,
                    options=dict(ftol=1e-15, gtol=1e-12, maxiter=30000, maxls=100))
                reference_x = reference.x
            optimum_value = direct(B, y, r, c, reference_x)[0]
            # Lemma: the minimiser lies below U_j = sum_{i:B_ij>0} y_i / (B^T 1 + c)_j.
            assert np.all(reference_x <= block.upper_eff*(1+1e-7)+1e-9)
            checks["minimiser_bound_cases"] += 1
            for trial in (a, reference_x, np.full(dimension, cfg["xmin"]), np.full(dimension, cfg["xmax"])):
                evaluated = block.evaluate(trial)
                actual_error = evaluated["value"]-optimum_value
                assert actual_error <= evaluated["cert"]["gap"]+1e-9*(1+abs(optimum_value))
                theta = evaluated["cert"]["theta"]
                p = theta*y/evaluated["s"]
                dual_coefficient = B.T @ (1-p)+c
                positive = y > 0
                box_term = np.minimum(dual_coefficient*cfg["xmin"], dual_coefficient*block.upper_eff).sum()
                dual = float((1-p) @ r+np.sum(y[positive]*np.log(p[positive]))+box_term)
                assert abs(evaluated["value"]-dual-evaluated["cert"]["raw_gap"]) <= 2e-10*(1+abs(dual))
                assert dual <= optimum_value+1e-9*(1+abs(optimum_value))
                checks["dual_bound_cases"] += 1
            # Safe-step solver (no line search): every step satisfies the majorisation that makes it a descent step,
            # the value never increases, the frozen data stay untouched, and many steps approach the reference optimum.
            for geometry in ("burg", "euclidean"):
                block = make(matrix, y, r, c, a)
                frozen = tuple(array.tobytes() for array in (block.y, block.offset, block.c, block.anchor))
                start_error = block.evaluate(a)["value"]-optimum_value
                state, info = f.solve_inner(block, geometry, 20000, keep_trace=True)
                assert info["violations"] == 0 and info["steps"] <= 20000
                for step in info["trace"]:
                    assert step["remainder"] <= step["penalty"]+step["cushion"]
                    old_value = direct(B, y, r, c, step["old_z"])[0]
                    new_value = direct(B, y, r, c, step["new_z"])[0]
                    model_value = old_value+step["gradient"] @ (step["new_z"]-step["old_z"])+step["penalty"]
                    assert new_value <= model_value+1e-10*(1+abs(old_value))
                    assert new_value <= old_value+1e-10*(1+abs(old_value))
                    checks["safe_steps_checked"] += 1
                actual_error = state["value"]-optimum_value
                worst_solver_error = max(worst_solver_error, actual_error)
                assert actual_error <= 1e-3*max(start_error, 1e-12)+1e-9, (geometry, dimension, seed, actual_error, start_error)
                assert frozen == tuple(array.tobytes() for array in (block.y, block.offset, block.c, block.anchor))
                assert np.all((state["z"] >= cfg["xmin"]) & (state["z"] <= cfg["xmax"]))
                checks["solver_cases"] += 1
            print(f"Independent target/solver case: dimension={dimension}, case={seed} passed.")

    # When a pixel explains its rows (diagonal A, zero offset) one Burg step with W = L_j = y_j is the exact block minimiser
    # (the EM/Richardson-Lucy fixed point y/(d+c)); a Euclidean step with its box bound only decreases the value.
    y, diagonal, c = np.array([0., 2., 8., 100.]), np.array([1., 2., 0.5, 1.]), np.full(4, .2)
    analytic = np.clip(y/(diagonal+c), cfg["xmin"], cfg["xmax"])
    for j in range(4):
        block = make(np.array([diagonal[j]]), y[[j]], np.array([1e-300]), c[[j]], np.array([1.0]))
        state, info = f.solve_inner(block, "burg", 1)
        assert info["steps"] <= 1 and np.isclose(state["z"][0], analytic[j], rtol=1e-12), (j, state["z"], analytic[j])
        block = make(np.array([diagonal[j]]), y[[j]], np.array([1e-300]), c[[j]], np.array([1.0]))
        start = block.evaluate(np.array([1.0]))["value"]
        state, info = f.solve_inner(block, "euclidean", 1)
        assert state["value"] <= start+1e-12*(1+abs(start))
    checks["burg_one_step_exact_cases"] = 4
    zero = make(np.eye(3), np.zeros(3), np.ones(3), np.full(3, .1), np.ones(3))
    for geometry in ("burg", "euclidean"):
        zero_state, zero_info = f.solve_inner(zero, geometry, 5)
        assert np.array_equal(zero_state["z"], np.full(3, cfg["xmin"])), (geometry, zero_state["z"])

    large_cfg = dict(cfg, xmin=1e-3, xmax=1e8)
    large_y = np.array([0., 1e6])
    large_optimum = np.clip(large_y/1.1-.01, large_cfg["xmin"], large_cfg["xmax"])
    block = make(np.eye(2), large_y, np.full(2, .01), np.full(2, .1), np.ones(2), large_cfg)
    exact = direct(np.eye(2), large_y, np.full(2, .01), np.full(2, .1), large_optimum)[0]
    for point in (np.ones(2), large_optimum, np.array([1e6, 1e4])):
        evaluated = block.evaluate(point)
        assert evaluated["value"]-exact <= evaluated["cert"]["gap"]+1e-8
        checks["dual_bound_cases"] += 1

    # Regression: a tiny optimal theta (bright counts, near-zero offset) must not lose precision in theta-1-log(theta).
    tiny_cfg = dict(cfg, xmin=1e-6, xmax=1e8)
    for scalar_block in (True, False):
        B_t, y_t, r_t, c_t = np.array([0.0513]), np.array([100563.]), np.array([5.45e-9]), np.array([0.0434])
        block = make(B_t if scalar_block else B_t.reshape(1, 1), y_t, r_t, c_t, np.array([1.0]), tiny_cfg)
        evaluated = block.evaluate(np.array([tiny_cfg["xmin"]]))
        derivative = lambda z: float(B_t[0]*(1-y_t[0]/(r_t[0]+B_t[0]*z))+c_t[0])
        optimum = brentq(derivative, tiny_cfg["xmin"], tiny_cfg["xmax"], xtol=1e-12)
        exact = direct(B_t.reshape(1, 1), y_t, r_t, c_t, np.array([tiny_cfg["xmin"]]))[0]-\
            direct(B_t.reshape(1, 1), y_t, r_t, c_t, np.array([optimum]))[0]
        assert evaluated["cert"]["gap"] >= exact*(1-1e-12), (scalar_block, evaluated["cert"], exact)
        checks["dual_bound_cases"] += 1

    # The exact theta minimiser beats a fine grid of the convex dual bound, for vector and scalar blocks.
    for _ in range(6):
        m = int(rng.integers(1, 6))
        B = rng.uniform(0.05, 1.0, (7, m))
        y = rng.poisson(rng.uniform(0, 20, 7)).astype(float)
        block = make(B[:, 0] if m == 1 else B, y, rng.uniform(0.1, 1, 7), rng.uniform(0, .3, m), rng.uniform(0.1, 3, m))
        z = rng.uniform(cfg["xmin"], 5, m)
        evaluated = block.evaluate(z)
        w, ysum = evaluated["w"], float(y.sum())
        grid = np.geomspace(1e-4, 10, 4001)
        values = [f._certificate_candidate(t, z, ysum, block.bcoef, w, cfg["xmin"], block.upper_eff)["raw_gap"]
                  for t in grid]
        assert evaluated["cert"]["raw_gap"] <= min(values)+1e-9*(1+abs(min(values)))
        checks["dual_bound_cases"] += 1

    # Scaled rational arithmetic remains finite when W*u would overflow.
    extreme = f.mirror_proposal(np.array([1e100, 1., 1.]), np.array([1e200, -2., -1.]),
        np.array([1e300, 1., 1.]), "burg", 1e-100, 1e200)
    assert np.isclose(extreme[0]/1e100, .5) and extreme[1] == extreme[2] == 1e200
    assert np.isfinite(f.weighted_burg(np.array([1e200]), np.array([1e-200]), np.array([1e-200]))).all()
    assert abs(float(f.weighted_burg(np.array([1+1e-9]), np.array([1.0]), np.array([1.0]))[0])-0.5e-18) < 1e-24

    for geometry in ("burg", "euclidean"):
        for gradient in (-100., -1.0, -0.01, 0.0, 0.1, 20.):
            a, weight = 0.7, 0.4
            gap = f.model_scores(np.array([a]), np.array([gradient]), np.array([weight]), geometry,
                                 cfg["xmin"], cfg["xmax"])
            def model(logz):
                z = np.exp(logz)
                penalty = weight*(z/a-np.log(z/a)-1) if geometry == "burg" else .5*weight*(z-a)**2
                return gradient*(z-a)+penalty
            solution = minimize_scalar(model, bounds=(np.log(cfg["xmin"]), np.log(cfg["xmax"])),
                                       method="bounded", options=dict(xatol=1e-13))
            independent = -min(solution.fun, model(np.log(cfg["xmin"])), model(np.log(cfg["xmax"])), 0.)
            assert abs(gap["scores"][0]-independent) <= 1e-8*(1+abs(independent))
            checks["gap_model_cases"] += 1

    # Compare the constrained tie allocation with independent SLSQP minimisation.
    for geometry in ("burg", "euclidean"):
        for tau in (cfg["xmin"], 1., cfg["xmax"]):
            for _ in range(4):
                x = np.array([tau, tau, tau, tau])
                base = rng.uniform(-.8, .8, 4)
                weights = rng.uniform(.05, 2., 4)
                mu, q = .7, 2
                v, metadata = f.minimizing_subgradient(x, base, weights, q, mu)
                actual = float(f.model_scores(x, base-v, weights, geometry, cfg["xmin"], cfg["xmax"])["scores"].sum())
                def objective(z):
                    t = f.mirror_proposal(x, base-z, weights, geometry, cfg["xmin"], cfg["xmax"])
                    penalty = weights*(t/x-np.log(t/x)-1) if geometry == "burg" else .5*weights*(t-x)**2
                    return float(np.sum((base-z)*(x-t)-penalty)), t-x
                reference = minimize(objective, np.full(4, mu*q/4), jac=True, method="SLSQP",
                    bounds=[(0., mu)]*4, constraints=[dict(type="eq", fun=lambda z: z.sum()-mu*q,
                                                          jac=lambda z: np.ones(4))],
                    options=dict(ftol=1e-12, maxiter=2000))
                assert abs(v.sum()-mu*q) < 1e-12 and np.all((v >= 0) & (v <= mu))
                assert actual <= reference.fun+2e-8*(1+abs(reference.fun)), (geometry, tau, actual, reference.fun)
                checks["tie_cases"] += 1

    # The safe weights majorise the exact likelihood remainder for ANY pair of points in the box (this is what makes
    # every step safe without a line search): scalar blocks with their frozen offset (Burg L_j, Euclidean box bound of the
    # block) and full vectors (Burg diag(L_j) by joint relative smoothness, Euclidean Hessian row-sum bound).
    for trial in range(40):
        scalar = trial % 2 == 0
        m = 1 if scalar else int(rng.integers(2, 6))
        B = rng.uniform(0, 1, (8, m))*(rng.uniform(size=(8, m)) < .7)
        B[:, np.flatnonzero(B.sum(axis=0) == 0)] = 0.5
        y = rng.poisson(rng.uniform(0, 30, 8)).astype(float)
        r = rng.uniform(1e-3, 0.5, 8)
        block = make(B[:, 0] if scalar else B, y, r, np.full(m, .1), np.full(m, 1.0))
        for geometry in ("burg", "euclidean"):
            weights = block.cap(geometry)
            points = [np.full(m, cfg["xmin"]), np.full(m, cfg["xmax"])]+[
                np.exp(rng.uniform(np.log(cfg["xmin"]), np.log(cfg["xmax"]), m)) for _ in range(30)]
            for u in points:
                old = block.evaluate(u)
                for z in points:
                    new = block.evaluate(z, gradient=False)
                    _, _, remainder = f.stable_target_change(block, old, new)
                    penalty = (float(f.weighted_burg(z, u, weights).sum()) if geometry == "burg"
                               else 0.5*float(weights @ (z-u)**2))
                    assert remainder <= penalty*(1+1e-10)+1e-12, (geometry, scalar, u, z, remainder, penalty)
                    checks["safe_constant_pairs"] += 1
        # the scalar Euclidean bound is the block's largest curvature on the box, attained at x_min
        if scalar:
            curvature_at_min = float(np.sum(y*B[:, 0]**2/(r+B[:, 0]*cfg["xmin"])**2))
            assert np.isclose(block.cap("euclidean")[0], max(cfg["positive_scale_floor"], curvature_at_min), rtol=1e-12)

    # Accounting of one safe solve on a scalar block with a given offset: forward products = anchor + steps, transpose
    # products = anchor + steps (minus the last one without final_gradient), one column for the Euclidean bound, nothing
    # for Burg; no curvature product and no rejected trial exist.
    for geometry in ("burg", "euclidean"):
        for final_gradient in (True, False):
            block = make(np.array([1., .5]), np.array([3., 1.]), np.array([.2, .3]), np.array([.1]), np.array([1.]))
            state, info = f.solve_inner(block, geometry, 7, final_gradient=final_gradient)
            events, steps = block.work.events, info["steps"]
            assert steps == 7 and info["status"] == "budget_used"
            assert events["forward_products"] == 1+steps
            assert events["transpose_products"] == 1+steps-(0 if final_gradient else 1)
            assert events["safe_constants_formed"] == (1 if geometry == "euclidean" else 0)
            cap_columns = 1 if geometry == "euclidean" else 0
            assert block.work.counts["inner"] == block.nnz*(events["forward_products"]+events["transpose_products"]+cap_columns)
            assert events["majorization_violations"] == 0
    checks["accounting_identities"] = 4

    # Backtracking on a scalar block, both geometries (the policy of the experiment): every accepted step satisfies the
    # geometry's upper model and decreases Q, every trial is charged (forward products = anchor + steps + rejected trials),
    # the curvature estimate costs one column read, no box bound is formed, and a long solve reaches the block minimiser.
    bt = dict(cfg, backtracking_geometries=("euclidean", "burg"))
    B2, y2, r2, c2 = np.array([1., .5]), np.array([3., 1.]), np.array([.2, .3]), np.array([.1])
    derivative = lambda z: float(B2 @ (1-y2/(r2+B2*z))+c2[0])
    minimiser = brentq(derivative, cfg["xmin"], cfg["xmax"])
    for geometry in ("euclidean", "burg"):
        for final_gradient in (True, False):
            block = make(B2, y2, r2, c2, np.array([1.]), bt)
            state, info = f.solve_inner(block, geometry, 7, final_gradient=final_gradient, keep_trace=True)
            events, steps = block.work.events, info["steps"]
            assert steps == 7 and info["status"] == "budget_used" and info["violations"] == 0, (geometry, info["status"])
            assert events["curvature_estimates"] == 1 and events["safe_constants_formed"] == 0
            assert events["forward_products"] == 1+steps+events["rejected_trials"]
            assert events["transpose_products"] == 1+steps-(0 if final_gradient else 1)
            assert block.work.counts["inner"] == block.nnz*(events["forward_products"]+events["transpose_products"]+1)
            accepted_values = [t["new_value"] for t in info["trace"] if t["remainder"] <= t["penalty"]+t["cushion"]]
            values = np.array([info["anchor_value"]]+accepted_values)
            assert len(accepted_values) == steps and np.all(np.diff(values) <= 1e-10*(1+np.abs(values[:-1])))
        block = make(B2, y2, r2, c2, np.array([1.]), bt)
        state, info = f.solve_inner(block, geometry, 400)
        assert abs(float(state["z"][0])-minimiser) <= 1e-6*(1+minimiser), (geometry, state["z"], minimiser, info["status"])
        # the first weight is the local curvature of the block in the geometry's metric
        anchor = make(B2, y2, r2, c2, np.array([1.]), bt).evaluate(np.array([1.]))
        curvature = float(np.sum(y2*B2**2/(r2+B2*1.0)**2))
        first = make(B2, y2, r2, c2, np.array([1.]), bt).local_curvature(anchor, geometry)[0]
        assert np.isclose(first, curvature if geometry == "euclidean" else curvature*1.0**2, rtol=1e-12)
    checks["backtracking_cases"] = 6

    # The eps_k rule (the policy of the experiment, both geometries, backtracking): the solve stops as soon as its
    # certified error bound is at most eps_k = max(inner_absolute_tolerance, rho*max{Delta, achieved decrease}); the
    # certified bound dominates the true error (independent brentq minimiser), which the reporting reference measures to
    # roundoff; Delta equals an independent minimisation of the Burg model with L_j; a gradient is formed after every step
    # and nothing else is charged; the reporting reference is uncharged but tallied; tighter tolerances give longer and
    # more accurate solves; a block started at its minimiser ends at once; a floor turns the rule into an absolute
    # tolerance where it dominates.
    rule = dict(bt, inner_relative_tolerance=1e-1, inner_absolute_tolerance=0.0)
    L_block = max(rule["positive_scale_floor"], float(y2.sum()))             # L_j = sum of the counts on the column's support
    g_anchor = derivative(1.0)
    burg_model = lambda logt: -(g_anchor*(1.0-np.exp(logt))-L_block*(np.exp(logt)-logt-1.0))
    solution = minimize_scalar(burg_model, bounds=(np.log(cfg["xmin"]), np.log(cfg["xmax"])), method="bounded",
                               options=dict(xatol=1e-13))
    delta_independent = max(0.0, -solution.fun, -burg_model(np.log(cfg["xmin"])), -burg_model(np.log(cfg["xmax"])))
    optimum_value = direct(B2.reshape(2, 1), y2, r2, c2, np.array([minimiser]))[0]
    anchor_value = direct(B2.reshape(2, 1), y2, r2, c2, np.array([1.0]))[0]
    for geometry in ("euclidean", "burg"):
        block = make(B2, y2, r2, c2, np.array([1.]), rule)
        state, info = f.solve_inner(block, geometry, 1000, final_gradient=False)
        assert info["status"] == "inner_tolerance" and 1 <= info["steps"] < 1000, (geometry, info["status"], info["steps"])
        assert abs(info["eps_reference"]-delta_independent) <= 1e-8*(1+delta_independent)
        true_error = direct(B2.reshape(2, 1), y2, r2, c2, state["z"])[0]-optimum_value
        achieved = anchor_value-direct(B2.reshape(2, 1), y2, r2, c2, state["z"])[0]
        assert abs(info["eps_true"]-true_error) <= 1e-10*(1+abs(anchor_value)), (geometry, info["eps_true"], true_error)
        assert abs(info["eps_achieved"]-achieved) <= 1e-10*(1+abs(anchor_value))
        assert abs(info["eps_target"]-rule["inner_relative_tolerance"]*max(info["eps_reference"], info["eps_achieved"])) <= 1e-15
        assert info["eps_certified"] <= info["eps_target"], (geometry, info["eps_certified"], info["eps_target"])
        assert true_error <= info["eps_certified"]+1e-10 and info["eps_true"] <= info["eps_certified"]+1e-12
        assert info["eps_true"] <= info["eps_target"] and true_error <= info["gap"]+1e-10, (geometry, true_error, info["gap"])
        assert info["reference_certificate"] <= 1e-9
        events, counts = block.work.events, block.work.counts
        assert events["transpose_products"] == 1+info["steps"]                 # anchor gradient and one after every step
        assert events["forward_products"] == 1+info["steps"]+events["rejected_trials"]
        assert counts["inner"] == block.nnz*(events["forward_products"]+events["transpose_products"]+1)
        assert counts["reference"] > 0 and events["reference_solves"] == 1    # the reporting reference, uncharged
        assert block.work.charged == counts["inner"]                          # nothing but the solve's own products
        assert counts["diagnostics"] == 0                                     # the certificate used the solve's gradient
        tight = make(B2, y2, r2, c2, np.array([1.]), dict(rule, inner_relative_tolerance=1e-6))
        tight_state, tight_info = f.solve_inner(tight, geometry, 1000, final_gradient=False)
        assert tight_info["status"] == "inner_tolerance" and tight_info["steps"] >= info["steps"], tight_info["status"]
        assert tight_info["eps_true"] <= tight_info["eps_target"] < info["eps_target"]
        tight_error = direct(B2.reshape(2, 1), y2, r2, c2, tight_state["z"])[0]-optimum_value
        assert -1e-12 <= tight_error <= min(true_error, tight_info["eps_target"])+1e-14
        solved = make(B2, y2, r2, c2, np.array([minimiser]), rule)
        solved_state, solved_info = f.solve_inner(solved, geometry, 1000, final_gradient=False)
        assert solved_info["status"] in ("inner_tolerance", "step_below_resolution", "change_below_resolution")
        assert solved_info["steps"] <= 3 and abs(float(solved_state["z"][0])-minimiser) <= 1e-9, solved_info
        floored = make(B2, y2, r2, c2, np.array([1.]), dict(rule, inner_absolute_tolerance=1e-1))
        floored_state, floored_info = f.solve_inner(floored, geometry, 1000, final_gradient=False)
        assert floored_info["status"] == "inner_tolerance" and floored_info["eps_target"] == 1e-1 > info["eps_target"]
        assert floored_info["eps_true"] <= 1e-1 and floored_info["steps"] <= info["steps"]
    checks["inner_tolerance_cases"] = 8

    # A budget interruption keeps the last completed descent state and reports the budget.
    limited = dict(cfg, max_passes=4.0)
    block = make(np.array([1., .5]), np.array([3., 1.]), np.array([.2, .3]), np.array([.1]), np.array([1.]), limited)
    anchor_value = make(np.array([1., .5]), np.array([3., 1.]), np.array([.2, .3]), np.array([.1]),
                        np.array([1.])).evaluate(np.array([1.]))["value"]
    state, info = f.solve_inner(block, "burg", 1000)
    assert "budget" in info["status"] and info["steps"] >= 1
    assert state["value"] <= anchor_value+1e-12

    # Small PSF problem for the outer machinery.
    small = dict(f.CONFIG_PSF, img_shape=(6, 6), sparsity=2, largest_q=2, psf_radius=1,
        min_spike_separation=2, total_photons=400., seeds=[4], max_passes=400., max_outer=None,
        record_inner=True, gs_batch_size=8)
    problem = f.prepare_problem(small, 4)
    n = len(problem["x0"])
    work = f.Work(problem["A"].nnz, 1e8)
    constants = f.geometry_constants(problem, small, work)
    x = problem["x0"].copy()
    s = np.asarray(problem["A"] @ x)+small["background"]
    dense = problem["A"].toarray()

    # Coordinate blocks built by make_target: the Burg weight is L_j from setup (no charge); the Euclidean weight is the
    # block's box bound, formed for one column, and it majorises the exact curvature Q''(z) on the whole box.
    mask0 = f.top_q_mask(x, 2)
    for j in rng.choice(n, size=6, replace=False):
        local = f.Work(problem["A"].nnz, 1e8)
        target, rows = f.make_target(problem, x, s, small["mu"]*mask0[j], int(j), small, local, constants)
        column = dense[rows, j]
        assert np.array_equal(target.B, column)
        assert np.array_equal(target.cap("burg"), np.maximum(small["positive_scale_floor"], constants["ysupport"][[j]]))
        assert local.charged == 0 and local.events["adhoc_setup"] == 0
        offset = np.array(target.offset)                                  # formed for one column
        before = local.charged
        cap = target.cap("euclidean")
        assert local.charged-before == problem["col_nnz"][j] and local.events["safe_constants_formed"] == 1
        yj = problem["y"][rows]
        for z in np.concatenate(([small["xmin"]], np.exp(rng.uniform(np.log(small["xmin"]), np.log(small["xmax"]), 20)))):
            assert float(np.sum(yj*column**2/(offset+column*z)**2)) <= cap[0]*(1+1e-12)
        checks["coordinate_safe_constant_cases"] = checks.get("coordinate_safe_constant_cases", 0)+1

    # Full-vector safe weights on the problem: the Euclidean row-sum bound dominates the Hessian row sums and the Burg
    # weights L_j dominate the Burg-scaled Hessian row sums at random points of the box.
    for _ in range(5):
        u = np.exp(rng.uniform(np.log(small["xmin"]), np.log(50), n))
        su = dense @ u+small["background"]
        H = dense.T @ np.diag(problem["y"]/su**2) @ dense
        assert np.all(H @ np.ones(n) <= constants["euclid_cap"]*(1+1e-12))
        assert np.all(u*(H @ u) <= np.maximum(constants["ysupport"], small["positive_scale_floor"])*(1+1e-12))
    checks["full_safe_weight_cases"] = 5

    # A coordinate block at a bound with its gradient pointing out cannot move: no offset, no weight, no trial product;
    # it costs its gradient column, or nothing when the gradient is known (Gauss-Southwell batch).
    for j in np.argsort(-(problem["colsum"]+small["lambda_"]-dense.T @ (problem["y"]/s))):
        bottom = x.copy()
        bottom[j] = small["xmin"]
        bottom_s = dense @ bottom+small["background"]
        bottom_mask = f.top_q_mask(bottom, 2)
        w_j = float(dense[:, j] @ (problem["y"]/bottom_s))
        if problem["colsum"][j]+small["lambda_"]-small["mu"]*bottom_mask[j]-w_j > 0:
            break
    for geometry in ("burg", "euclidean"):
        for known in (None, np.array([w_j])):
            local = f.Work(problem["A"].nnz, 1e8)
            target, rows = f.make_target(problem, bottom, bottom_s, small["mu"]*bottom_mask[j], int(j), small, local, constants)
            state, info = f.solve_inner(target, geometry, 5, bottom_s[rows], known, final_gradient=False)
            assert info["status"] == "block_stationary" and info["steps"] == 0 and state["z"][0] == small["xmin"]
            assert target._offset is None and local.events["offsets_formed"] == 0
            assert local.events["safe_constants_formed"] == 0
            assert local.charged == (problem["col_nnz"][j] if known is None else 0)
    checks["zero_step_blocks_without_offset"] = 4

    # Both stationarity measures against independent computations (no ties at the random start).
    measures, w = f.stationarity(problem, x, s, constants, small, work)
    gradient = dense.T @ (1-problem["y"]/s)+small["lambda_"]-small["mu"]*f.top_q_mask(x, 2)
    residual = x-np.clip(x-gradient, small["xmin"], small["xmax"])
    assert np.isclose(measures["euclidean_gap"], np.linalg.norm(residual), rtol=1e-12)
    burg_total = 0.0
    for j in range(n):
        L = max(problem["y"][dense[:, j] > 0].sum(), small["positive_scale_floor"])
        def negative_model(logt, j=j, L=L):
            t = np.exp(logt)
            return -(gradient[j]*(x[j]-t)-L*(t/x[j]-np.log(t/x[j])-1))
        solution = minimize_scalar(negative_model, bounds=(np.log(small["xmin"]), np.log(small["xmax"])),
                                   method="bounded", options=dict(xatol=1e-12))
        burg_total += max(0.0, -solution.fun, -negative_model(np.log(small["xmin"])), -negative_model(np.log(small["xmax"])))
    assert abs(measures["burg_gap"]-burg_total) <= 1e-6*(1+burg_total), (measures["burg_gap"], burg_total)
    checks["measure_cases"] = 2

    # Batch Gauss-Southwell: charged columns, batch without replacement, argmax of independently computed scores.
    mask = f.top_q_mask(x, 2)
    for spec_name in ("E-GS(PG)", "B-GS(Burg)", "B-GS(PG)"):
        spec = f.SPEC_BY_NAME[spec_name]
        batch_rng = np.random.default_rng(99)
        preview = np.random.default_rng(99).choice(n, size=small["gs_batch_size"], replace=False)
        local = f.Work(problem["A"].nnz, 1e8)
        j, v, w_j, info = f.select_block(problem, x, s, mask, constants, spec, dict(batch=batch_rng), small, local)
        assert local.counts["selection"] == int(problem["col_nnz"][preview].sum())
        # With the gradient at x known (after a stopping check) the same batch is scored for free.
        known = f.Work(problem["A"].nnz, 1e8)
        j_known, _, w_known, _ = f.select_block(problem, x, s, mask, constants, spec,
            dict(batch=np.random.default_rng(99)), small, known, cached_w=dense.T @ (problem["y"]/s))
        assert known.charged == 0 and j_known == j and np.isclose(w_known, w_j, rtol=1e-12)
        assert len(set(preview.tolist())) == small["gs_batch_size"]
        independent = []
        for k in preview:
            g = gradient[k]
            if spec["score"] == "pg":
                independent.append(abs(x[k]-np.clip(x[k]-g, small["xmin"], small["xmax"])))
            else:
                L = constants["burg"][k]
                model = lambda logt: -(g*(x[k]-np.exp(logt))-L*(np.exp(logt)/x[k]-logt+np.log(x[k])-1))
                solution = minimize_scalar(model, bounds=(np.log(small["xmin"]), np.log(small["xmax"])),
                                           method="bounded", options=dict(xatol=1e-12))
                independent.append(max(0.0, -solution.fun))
        independent = np.array(independent)
        assert j == int(preview[int(np.argmax(independent))]), spec_name
        assert np.isclose(w_j, dense[:, j] @ (problem["y"]/s), rtol=1e-12)
        assert info["selected_score"] == info["batch_max_score"]
    inclusion = np.zeros(n)
    batch_rng = np.random.default_rng(5)
    for _ in range(3000):
        inclusion[batch_rng.choice(n, size=small["gs_batch_size"], replace=False)] += 1
    expected = 3000*small["gs_batch_size"]/n
    assert np.all(np.abs(inclusion-expected) < 5*np.sqrt(expected))
    checks["gs_batch_draws"] = 3000

    random_rng = np.random.default_rng(7)
    draws = [f.select_block(problem, x, s, mask, constants, f.SPEC_BY_NAME["B-Random"], dict(random=random_rng),
                            small, work)[0] for _ in range(4096)]
    counts = np.bincount(draws, minlength=n)
    assert np.all(np.abs(counts-4096/n) < 5*np.sqrt(4096/n))
    assert np.any(np.diff(draws) == 0)
    checks["uniform_with_replacement_draws"] = len(draws)

    # # Inner budget t_k = t_0 + floor(k/N): one more safe step per sweep (coordinate methods) or per outer iteration (full).
    # assert f.inner_budget(small, 0, n) == 1 and f.inner_budget(small, n-1, n) == 1 and f.inner_budget(small, n, n) == 2
    # assert f.inner_budget(small, 5*n+3, n) == 6 and f.inner_budget(small, 3, 1) == 4
    # assert f.inner_budget(dict(small, max_inner_budget=3), 10, 1) == 3
    scheduled = dict(small, inner_relative_tolerance=None)
    t0, growth = small["inner_budget0"], small["inner_budget_growth"]
    assert f.inner_budget(scheduled, 0, n) == t0 and f.inner_budget(scheduled, n, n) == t0+growth
    assert f.inner_budget(scheduled, 5*n+3, n) == t0+5*growth and f.inner_budget(scheduled, 3, 1) == t0+3*growth
    assert f.inner_budget(dict(scheduled, max_inner_budget=3), 10, 1) == min(3, t0+10*growth)
    # With the eps_k rule the schedule is replaced by the cap.
    assert f.inner_budget(small, 0, n) == f.inner_budget(small, 5*n+3, 1) == small["max_inner_budget"]

    # Full comparison on the small problem: consistency checks for every run (feasibility, caches, budget, monotone
    # objective, zero majorisation violations, inner budgets respected, GS rule, stopping rule when reported).
    comparison = f.run_comparison(small, verbose=False)
    checks["small_outer_checks"] = f.assert_comparison(comparison)
    backtracked = set(small.get("backtracking_geometries", ()))
    if backtracked:
        # Backtracked methods form no box bound; each attempted block solve starts with one curvature estimate.
        counted = 0
        for (seed_, name), run in comparison["runs"].items():
            spec = f.SPEC_BY_NAME[name]
            if spec["geometry"] in backtracked and spec["selection"] != "fista":
                events = run["work"].events
                assert events["safe_constants_formed"] == 0 and 1 <= events["curvature_estimates"] <= len(run["audit"]), name
                counted += 1
        checks["backtracking_runs"] = counted
    # eps_k rule on the small problem: every DCA solve is certified; the ones stopped by the rule are within their target;
    # for B-GS(Burg) the reference Delta_k is the selected Gauss-Southwell score (the same quantity, computed independently
    # in the selection); the coordinate solves end by the rule, by block stationarity or below resolution, never by the cap.
    certified = within = 0
    for (seed_, name), run in comparison["runs"].items():
        spec = f.SPEC_BY_NAME[name]
        if spec["selection"] == "fista":
            continue
        inner, good = run["inner"], run["audit"].query("accepted")
        assert len(inner) == run["audit_counts"]["accepted"], name
        stepped = inner[inner.steps > 0]
        assert np.isfinite(stepped.eps.to_numpy(float)).all() and (stepped.online_certificate+1e-9 >= stepped.eps).all(), name
        assert (stepped.eps_certified+1e-9 >= stepped.eps).all(), name        # the certified bound dominates the true error
        reached = inner[inner.status == "inner_tolerance"]
        assert (reached.eps_certified <= reached.eps_target).all() and (reached.eps <= reached.eps_target).all(), name
        if spec["selection"] == "full" and len(stepped):
            assert (stepped.reference_certificate <= 1e-6*(1+np.abs(stepped.eps_achieved))).all(), name
        assert set(inner.status) <= {"inner_tolerance", "block_stationary", "step_below_resolution",
                                     "change_below_resolution", "budget_used", "backtracking_limit"}, (name, set(inner.status))
        if spec["selection"] != "full":
            assert "budget_used" not in set(inner.status), name
        if spec["name"] == "B-GS(Burg)":
            assert np.allclose(good.eps_reference, good.selected_score, rtol=1e-9, atol=1e-14), name
        certified += len(inner)
        within += int(len(reached))
    checks["eps_rule_measured_solves"] = certified
    checks["eps_rule_solves_within_target"] = within
    # The full-block reference minimum against an independent dense L-BFGS-B minimisation of the first subproblem.
    local = f.Work(problem["A"].nnz, 1e9)
    target, _ = f.make_target(problem, x, s, small["mu"]*f.top_q_mask(x, 2), None, small, local, constants)
    reference_value, reference_z, reference_cert = f.reference_minimum(target, x)
    c_full = small["lambda_"]-small["mu"]*f.top_q_mask(x, 2)
    dense_fun = lambda z: direct(dense, problem["y"], np.full(len(s), small["background"]), c_full, z)
    independent = minimize(dense_fun, x, jac=True, method="L-BFGS-B", bounds=[(small["xmin"], small["xmax"])]*n,
                           options=dict(ftol=1e-16, gtol=1e-12, maxiter=50000, maxfun=100000, maxls=100))
    assert abs(reference_value-independent.fun) <= 1e-8*(1+abs(independent.fun)), (reference_value, independent.fun)
    assert reference_cert <= 1e-6*(1+abs(reference_value)) and local.charged == 0 and local.counts["reference"] > 0
    checks["full_block_reference_cases"] = 1
    summary = f.comparison_summary(comparison)
    allowed = {"stationarity_reached", "matvec_budget", "no_step_possible"}
    assert set(summary.status) <= allowed, set(summary.status)
    checks["small_problem_statuses"] = summary.status.value_counts().to_dict()

    # A budget interruption inside the first inner solve keeps x0 and exposes the unfinished candidate.
    for name in ("B-Full", "E-Full", "B-Random"):
        spec = f.SPEC_BY_NAME[name]
        setup = f.Work(problem["A"].nnz, 1e9)
        f.initialize_run(problem, small, setup, spec)
        run = f.run_dca(problem, dict(small, max_passes=setup.passes+0.5, record_inner=False), spec)
        if run["status"] == "matvec_budget" and run["audit_counts"]["accepted"] == 0:
            assert np.array_equal(run["final_x"], problem["x0"]) and run["pending"] is not None
    tiny_budget = dict(small, max_passes=1.5)
    for spec in f.METHOD_SPECS:
        run = f.run_method(problem, tiny_budget, spec)
        assert np.array_equal(run["final_x"], problem["x0"])
        assert "during_setup" in run["status"]
        assert run["work"].passes <= 1.5

    # The target table must exclude a deliberately accurate but unfinished point.
    import pandas as pd
    fake = comparison["results"].iloc[[0]].copy()
    fake["accepted_state"], fake["phase"], fake["relative_x_error"] = False, "inner_candidate", 0.0
    fake["matvec_passes"] = 0.0
    augmented = dict(comparison, results=pd.concat([comparison["results"], fake], ignore_index=True))
    before, after = f.target_costs(comparison), f.target_costs(augmented)
    pd.testing.assert_frame_equal(before, after)

    # FISTA solves the convex problem, tested against its analytic diagonal optimum.
    baseline_cfg = dict(small, img_shape=(2, 2), sparsity=2, largest_q=2, max_outer=None,
        xmin=.01, xmax=20., max_passes=10000., tol_pg=1e-9, tol_burg=1e-9, backtracking_geometries=())
    diagonal = sp.eye(4, format="csc")
    rows, values, nnz = f.column_tables(diagonal)
    baseline_problem = dict(A=diagonal, A_csr=diagonal.tocsr(), AT=diagonal.T.tocsr(),
        colsum=np.ones(4), y=np.array([0., 1., 5., 12.]), x0=np.ones(4), x_star=np.array([0., 1., 5., 12.]),
        support=np.array([2, 3]), seed=0, col_rows=rows, col_vals=values, col_nnz=nnz)
    run = f.run_fista(baseline_problem, baseline_cfg, f.SPEC_BY_NAME["FISTA"])
    analytic = np.clip(baseline_problem["y"]/(1+baseline_cfg["lambda_"])-baseline_cfg["background"],
                       baseline_cfg["xmin"], baseline_cfg["xmax"])
    optimum = f.original_objective(analytic, analytic+baseline_cfg["background"], baseline_problem["y"],
                                   baseline_cfg, convex=True)
    final = run["history"].iloc[-1]
    assert run["status"] == "stationarity_reached", run["status"]
    assert final.convex_objective-optimum <= final.convex_certificate+1e-10
    checks["fista_analytic_error"] = float(final.convex_objective-optimum)

    # FISTA spends one forward product per trial and none at extrapolated points (their intensity follows by linearity).
    run = f.run_fista(problem, dict(small, max_passes=80.), f.SPEC_BY_NAME["FISTA"])
    events = run["work"].events
    trials = events["inner_steps"]+events["restarts"]+events["rejected_trials"]
    assert events["extrapolations"] > 0 and events["forward_products"] == trials+events["extrapolation_products"]
    assert run["work"].counts["inner"] == problem["A"].nnz*(events["forward_products"]+events["transpose_products"]
                                                           -(run["work"].counts["stopping"]//problem["A"].nnz-1))
    checks["fista_extrapolations_without_forward_product"] = events["extrapolations"]

    # On the diagonal problem (each pixel explains its row), one Burg step solves every block exactly, so the Burg DCA
    # methods reach the analytic DC critical point. The Euclidean methods only need to decrease the objective monotonically:
    # their box bound is about (x/x_min)^2 times the curvature at the solution.
    for spec in f.METHOD_SPECS[:-1]:
        run = f.run_dca(baseline_problem, dict(baseline_cfg, gs_batch_size=2), spec)
        objective = run["history"].query("accepted_state").objective.to_numpy()
        assert np.all(np.diff(objective) <= 1e-9*(1+np.abs(objective[:-1])))
        if spec["geometry"] == "burg":
            assert run["status"] == "stationarity_reached", (spec["name"], run["status"])
            top = np.argsort(-baseline_problem["y"])[:2]
            cost = np.full(4, baseline_cfg["lambda_"])
            cost[top] -= baseline_cfg["mu"]
            critical = np.clip(baseline_problem["y"]/(1+cost)-baseline_cfg["background"], baseline_cfg["xmin"],
                               baseline_cfg["xmax"])
            # (the outer tolerance tol_burg = 1e-9 on the Burg gap, with inexact eps_k solves, locates the critical point to
            # about 1e-4 in relative terms)
            assert np.allclose(run["final_x"], critical, rtol=1e-3, atol=1e-5), (spec["name"], run["final_x"], critical)
    checks["analytic_critical_point_runs"] = sum(spec["geometry"] == "burg" for spec in f.METHOD_SPECS[:-1])
    # With backtracking in both geometries every DCA method decreases the objective monotonically on the diagonal problem
    # and reaches its stationarity tolerance.
    backtracked_cfg = dict(baseline_cfg, gs_batch_size=2, backtracking_geometries=("euclidean", "burg"))
    statuses = {}
    for spec in f.METHOD_SPECS[:-1]:
        run = f.run_dca(baseline_problem, backtracked_cfg, spec)
        objective = run["history"].query("accepted_state").objective.to_numpy()
        assert np.all(np.diff(objective) <= 1e-9*(1+np.abs(objective[:-1])))
        assert run["work"].events["majorization_violations"] == 0
        statuses[spec["name"]] = run["status"]
    checks["diagonal_backtracking_statuses"] = statuses
    checks["worst_finite_difference_error"] = worst_gradient_error
    checks["worst_reference_objective_error"] = worst_solver_error
    checks["all_passed"] = True
    return checks

In [ ]:
import json
from types import SimpleNamespace
validation_started = time.perf_counter()
validation_report = run_validation(SimpleNamespace(**globals()))
validation_report["seconds"] = round(time.perf_counter()-validation_started, 1)
print(json.dumps(validation_report, indent=2, default=str))

## Independent MatVec Audit

The solvers keep their own ledger (`Work`). This cell recounts every touch of a matrix entry without using that ledger, for
all eight methods on the first seed, over the first `AUDIT_PASSES` MatVec passes of each run (the rules do not change later;
set `AUDIT_PASSES = CONFIG_PSF["max_passes"]` and `AUDIT_SEEDS = CONFIG_PSF["seeds"]` for complete runs):

* the sparse operators ($A$ in CSC and CSR form, $A^\top$, $(A\circ A)^\top$) are wrapped so that every product with them, and
  with any row extraction or copy of them, is tallied by its nonzeros; any other sparse method raises an error, so no product
  can bypass the count;
* the entry arrays that coordinate blocks and Gauss–Southwell batches work with (columns of $A$, the padded column table)
  are marked, and every NumPy operation on a marked array is tallied by its nonzeros, wherever it happens;
* the column sums, stored with the problem, add one pass.

The tally must equal the ledger's charged plus uncharged (`diagnostics` and `reference`) contributions exactly. Two injected
faults (an uncharged column product per trial and an uncharged sparse product per Gauss–Southwell step) must be detected.
Finally each method runs again with every uncharged product perturbed by a relative $8\varepsilon_{\rm mach}$ (the
reporting-only reference minimiser's products included), reports at different times
(every 0.37 instead of 5 passes) and every setup quantity it is not charged for multiplied by 7 (only FISTA has one: $L_j$,
which it uses for reporting the Burg gap). Its final iterate, status,
charged passes and product counts must be identical bit for bit. Together: everything a method uses is charged, and nothing
uncharged influences a method.

In [ ]:
import collections


class Tracked(np.ndarray):
    """Entry array whose NumPy operations are tallied by the nonzeros of the marked operands (results are plain arrays)."""
    tally = None

    @staticmethod
    def _plain(value):
        if isinstance(value, Tracked):
            return value.view(np.ndarray)
        if isinstance(value, (list, tuple)):
            return type(value)(Tracked._plain(item) for item in value)
        return value

    @staticmethod
    def _marked(values):
        found = []
        for value in values:
            if isinstance(value, Tracked):
                found.append(value)
            elif isinstance(value, (list, tuple)):
                found.extend(Tracked._marked(value))
        return found

    def __array_ufunc__(self, ufunc, method, *inputs, **kwargs):
        if Tracked.tally is not None:
            Tracked.tally["entries"] += sum(int(np.count_nonzero(v.view(np.ndarray))) for v in Tracked._marked(inputs))
        if "out" in kwargs:
            kwargs["out"] = Tracked._plain(kwargs["out"])
        return getattr(ufunc, method)(*Tracked._plain(inputs), **kwargs)

    def __array_function__(self, func, types, args, kwargs):
        if Tracked.tally is not None:
            marked = Tracked._marked(args)+Tracked._marked(tuple(kwargs.values()))
            Tracked.tally["entries"] += sum(int(np.count_nonzero(v.view(np.ndarray))) for v in marked)
        return func(*Tracked._plain(args), **{k: Tracked._plain(v) for k, v in kwargs.items()})


class CountedMatrix:
    """A SciPy sparse matrix whose products are tallied by the nonzeros they touch. Only the attributes below exist, so
    no other sparse operation can run unnoticed; `data` is returned marked when the entries are read column by column."""
    ALLOWED = {"nnz", "shape", "indptr", "indices", "dtype"}

    def __init__(self, matrix, tally, mark_data=False):
        object.__setattr__(self, "matrix", matrix)
        object.__setattr__(self, "tally", tally)
        object.__setattr__(self, "mark_data", mark_data)

    def __matmul__(self, other):
        self.tally["sparse"] += self.matrix.nnz
        return self.matrix @ other

    def __getitem__(self, key):
        return CountedMatrix(self.matrix[key], self.tally)

    @property
    def T(self):
        return CountedMatrix(self.matrix.T, self.tally)

    def copy(self):
        return CountedMatrix(self.matrix.copy(), self.tally)

    @property
    def data(self):
        return self.matrix.data.view(Tracked) if self.mark_data else self.matrix.data

    def __getattr__(self, name):
        if name not in CountedMatrix.ALLOWED:
            raise AttributeError(f"sparse operation '{name}' is not allowed under the MatVec audit")
        return getattr(self.matrix, name)

    def __setattr__(self, name, value):
        if name != "data":
            raise AttributeError(name)
        self.matrix.data = value


ALGORITHM_EVENTS = ("forward_products", "transpose_products", "offsets_formed", "inner_steps", "rejected_trials",
                    "gs_batches", "gs_cached_batches", "restarts", "extrapolations", "safe_constants_formed",
                    "majorization_violations")


def audited_run(problem, cfg, spec, poison=False, fault=None):
    """Run one method with every touch of a matrix entry tallied independently of its ledger.
    poison=True: uncharged reporting products perturbed, reporting at other times, free setup quantities scaled by 7
    (the reference minimiser's products are reporting only and are perturbed as well: they must not change anything).
    fault: 'column' or 'sparse' injects uncharged work that the audit must detect."""
    namespace = globals()
    tally = collections.Counter()
    unused = set(ALL_CONSTANTS)-constant_needs(spec, cfg)
    audited = dict(problem)
    audited["col_vals"] = problem["col_vals"].view(Tracked)
    audited["A"] = CountedMatrix(problem["A"], tally, mark_data=True)
    for key in ("A_csr", "AT"):
        audited[key] = CountedMatrix(problem[key], tally)
    originals = dict(init=FrozenTarget.__init__, tmv=FrozenTarget.tmv, evaluate=FrozenTarget.evaluate,
                     gradient=namespace["full_gradient"], constants=namespace["geometry_constants"],
                     select=namespace["select_block"])
    noise = 1.0+8*FLOAT_EPS

    def init(self, *args, **kwargs):
        originals["init"](self, *args, **kwargs)
        if self.scalar:                                   # the block's private copy of its column entries
            self.B = self.B.view(Tracked)

    def tmv(self, vector, category="inner"):
        result = originals["tmv"](self, vector, category)
        return result*noise if poison and (category in ("diagnostics", "reference") or self.work.reporting) else result

    def gradient(problem, s, work, category):
        result = originals["gradient"](problem, s, work, category)
        return result*noise if poison and (category == "diagnostics" or work.reporting) else result

    def constants(problem, cfg, work, needs=ALL_CONSTANTS):
        result = originals["constants"](problem, cfg, work, needs)
        if poison and "ysupport" in unused:
            result["ysupport"], result["burg"] = 7.0*result["ysupport"], 7.0*result["burg"]
        return result

    def evaluate(self, z, intensity=None, w=None, gradient=True, certify=True):
        state = originals["evaluate"](self, z, intensity, w, gradient, certify)
        if fault == "column" and self.scalar:
            self.B @ (self.y/state["s"])                  # uncharged column product
        return state

    def select(problem, x, s, *args, **kwargs):
        if fault == "sparse" and spec["selection"] == "gs_batch":
            problem["AT"] @ (problem["y"]/s)              # uncharged full gradient
        return originals["select"](problem, x, s, *args, **kwargs)

    local = dict(cfg, log_every_passes=0.37, record_inner=True) if poison else cfg
    FrozenTarget.__init__, FrozenTarget.tmv, FrozenTarget.evaluate = init, tmv, evaluate
    namespace.update(full_gradient=gradient, geometry_constants=constants, select_block=select)
    Tracked.tally = tally
    try:
        run = run_method(audited, local, spec)
    finally:
        Tracked.tally = None
        FrozenTarget.__init__, FrozenTarget.tmv = originals["init"], originals["tmv"]
        FrozenTarget.evaluate = originals["evaluate"]
        namespace.update(full_gradient=originals["gradient"], geometry_constants=originals["constants"],
                         select_block=originals["select"])
    nnz = problem["A"].nnz
    work = run["work"]
    stored = nnz*len({"column_sum"} & constant_needs(spec, cfg))
    independent = tally["sparse"]+tally["entries"]+stored
    ledger = work.charged+work.counts["diagnostics"]+work.counts["reference"]
    return run, dict(charged=work.charged/nnz, uncharged=work.counts["diagnostics"]/nnz,
                     reference=work.counts["reference"]/nnz, independent_total=independent/nnz,
                     ledger_total=ledger/nnz, matches=independent == ledger,
                     **{name+"_passes": work.counts[name]/nnz for name in
                        ("setup", "selection", "inner", "offset", "cache", "stopping")},
                     adhoc=work.events["adhoc_setup"], safe_constants_formed=work.events["safe_constants_formed"],
                     majorization_violations=work.events["majorization_violations"],
                     uncharged_setup="L_j (reporting only)" if "ysupport" in unused else "-")


def same_trajectory(first, second):
    return bool(np.array_equal(first["final_x"], second["final_x"]) and first["status"] == second["status"]
                and first["work"].charged == second["work"].charged
                and all(first["work"].events[k] == second["work"].events[k] for k in ALGORITHM_EVENTS))


AUDIT_SEEDS, AUDIT_PASSES = CONFIG_PSF["seeds"][:1], 150.0
audit_config = dict(CONFIG_PSF, max_passes=AUDIT_PASSES)
audit_started = time.perf_counter()
audit_rows = []
for seed in AUDIT_SEEDS:
    audit_problem = prepare_problem(audit_config, seed)
    for spec in METHOD_SPECS:
        run, row = audited_run(audit_problem, audit_config, spec)
        poisoned, poisoned_row = audited_run(audit_problem, audit_config, spec, poison=True)
        row.update(seed=seed, method=spec["name"], status=run["status"], poisoned_matches=poisoned_row["matches"],
                   trajectory_identical=same_trajectory(run, poisoned))
        audit_rows.append(row)
matvec_audit = pd.DataFrame(audit_rows).set_index(["seed", "method"])
short = dict(audit_config, max_passes=12.0)
faults = {fault: audited_run(audit_problem, short, SPEC_BY_NAME[name], fault=fault)[1]["matches"]
          for fault, name in (("column", "E-Random"), ("sparse", "B-GS(Burg)"))}
with pd.option_context("display.width", 250, "display.float_format", "{:.2f}".format):
    display(matvec_audit[["status", "charged", "uncharged", "independent_total", "ledger_total", "matches",
                          "poisoned_matches", "trajectory_identical", "setup_passes", "selection_passes", "inner_passes",
                          "offset_passes", "cache_passes", "stopping_passes", "uncharged_setup", "adhoc",
                          "safe_constants_formed", "majorization_violations"]])
assert matvec_audit.matches.all() and matvec_audit.poisoned_matches.all() and matvec_audit.trajectory_identical.all()
assert (matvec_audit.adhoc == 0).all() and (matvec_audit.majorization_violations == 0).all() and not any(faults.values()), faults
validation_report["matvec_audit"] = dict(
    seeds=list(AUDIT_SEEDS), passes_per_run=AUDIT_PASSES, runs=len(matvec_audit), all_match=True, trajectories_identical=True,
    injected_faults_detected=sorted(faults), seconds=round(time.perf_counter()-audit_started, 1))
print(f"Independent MatVec audit passed: {len(matvec_audit)} runs recounted exactly, uncharged work has no influence, "
      f"both injected faults detected ({time.perf_counter()-audit_started:.0f} s).")

## Why block size and geometry matter here

A safe step (a step with the global constant, no backtracking) is as good as the ratio $r$ between its constant and the
curvature it has to dominate: along one coordinate it closes about a fraction $1/r$ of the remaining distance to the minimiser
(linear rate $1-1/r$), and far from the minimiser its length is scaled down by the same factor. In this version the steps are
backtracked, but the same constants define the Burg gap, the Gauss–Southwell scores and the reference $\Delta_k$ of the
$\varepsilon_k$ rule. The table compares each safe constant with the local diagonal curvature of the likelihood,
$h_j(x)=\sum_iy_iA_{ij}^2/(Ax+b)_i^2$:

* Burg: $L_j/(x_j^2h_j(x))$ (the Burg model has curvature $L_j/x_j^2$);
* Euclidean coordinate block: $W_j/h_j(x)$, with the box bound $W_j$ for the offset frozen at $x$;
* Euclidean full vector: the row-sum box bound over $h_j(x)$.

Rows: spikes and background pixels at the start $x^0$, and spikes at the ground truth $x^\star$. Entries are medians over seeds
of the per-seed median over the pixels in the row (uncharged diagnostic).

In [ ]:
looseness = looseness_table(CONFIG_PSF)
with pd.option_context("display.float_format", "{:.3g}".format):
    display(looseness)

The two geometries are loose in opposite places.

* **Burg** constants are tight at the spikes near the solution (ratio 1.01: each spike dominates its rows, and $L_j$ is the
  count on those rows) and moderately loose at the start (2.4). They are extremely loose at background pixels ($4.5\cdot10^5$
  at $x^0$): $L_j$ counts every photon on the column's support, while the pixel's own Burg curvature $x_j^2h_j$ is tiny. The
  same ratio governs the Burg gap: a coordinate contributes about $(x_jg_j)^2/(2L_j)$, so background pixels, and in particular
  coordinates sitting at $x_{\min}$, barely register in the gap and in the Burg Gauss–Southwell score even when their gradient
  is large.
* **Euclidean** box bounds are tight at $x^0$ (1.0 at background pixels, 8.5 at the spikes) but are evaluated at $x_{\min}$
  and do not follow the curvature down as a spike grows: at the solution they are $4\cdot10^6$ times the curvature at the
  spikes.

With standard backtracking in both geometries (this version) the step weights adapt to the local curvature, so neither set of
constants limits the steps; the constants matter through the stationarity measures, the Gauss–Southwell scores and the
reference $\Delta_k$ of the $\varepsilon_k$ rule. What remains is the cost structure: a full-vector step costs two passes (a
forward and a transpose product) however little it moves each coordinate, and under the $\varepsilon_k$ rule a full-block
subproblem is a convex problem of the full size that must be solved to a relative accuracy; a coordinate step touches one
column of $A$, its subproblem is one-dimensional, and Gauss–Southwell with the Burg score $\Delta_j$ directs those steps to
the coordinates whose Burg model decrease is largest.

## Experiment

All eight methods are run on every seed of `CONFIG_PSF` (same data, same $x^0$, same MatVec ceiling, same accuracy schedule).
`assert_comparison` then re-checks every run: box feasibility, cached intensities and objectives against direct recomputation,
the budget, monotone objectives, zero majorisation/target-change/GS violations in the audit counters, inner budgets respected,
and — for runs that report `stationarity_reached` — that the recomputed final value of the method's own measure is within
its tolerance. A run takes a few seconds in Python (see the wall-clock column of the summary); `workers` spreads the runs over processes.

In [ ]:
experiment_config = copy.deepcopy(CONFIG_PSF)
experiment_started = time.perf_counter()
comparison = run_comparison(experiment_config, workers=experiment_config.get("workers", 1))
experiment_seconds = time.perf_counter()-experiment_started
experiment_checks = assert_comparison(comparison)
summary = comparison_summary(comparison)
print(f"{len(comparison['runs'])} runs in {experiment_seconds:.1f} s; checks: {experiment_checks}")

### Final state of every run

`matvec_passes` is the cost at which the run stopped. `burg_gap` and `euclidean_gap` are recomputed at the final iterate from a
fresh intensity; `objective_gap` is $F-F^\star$ with the per-seed empirical reference.

In [ ]:
with pd.option_context("display.max_rows", 200, "display.width", 250, "display.float_format", "{:.4g}".format):
    display(summary)

### Per-method summary over seeds

Medians for costs and gaps; means ± population standard deviations for recovery.

In [ ]:
method_results = method_summary(summary, comparison["methods"])
with pd.option_context("display.width", 250, "display.float_format", "{:.3g}".format):
    display(method_results)

### Inner accuracy: certified and true $\varepsilon_k$

Every accepted block solve $k$ ends with its certified error bound $U_k\ge Q_k(x^{k+1}_{I_k})-\min Q_k$ (column
`eps_certified`, the quantity the stopping decision used), its target $\rho\max\{\Delta_k,Q_k(x^k)-Q_k(x^{k+1})\}$, the
reference $\Delta_k$ (the block's Burg gap contribution at $x^k$), the achieved decrease, and, for reporting only, the true
error $Q_k(x^{k+1}_{I_k})-\min Q_k$ against the independent reference minimiser (column `eps`). The table pools all accepted
solves of a method over the seeds: how the solves ended (`inner_tolerance` = stopped by the rule, i.e. certified;
`block_stationary` = the block sits at a bound with its gradient pointing out, error 0 and no step; `below resolution` = a
step below floating-point resolution, or one that changed $Q_k$ by less than its resolution, uncertified; `step cap` = the
`max_inner_budget` cap), the number of inner steps, the certified $\varepsilon_k$ over the solves that took a step (median,
90th percentile, maximum), the true $\varepsilon_k$ (median, maximum), $\sum_k\varepsilon_k$ of the true errors over a run
(the quantity in the bound of Theorem 4.1; median over seeds), the share of solves certified within their target or exact
(no step needed), the median ratio of the certified $\varepsilon_k$ to $\Delta_k$ over the solves stopped by the rule (at
most $\rho$ when $\Delta_k$ is the larger reference), the median ratio of the certified to the true error (how loose the
certificate is at the stopping point, i.e. how much accuracy the certified rule buys beyond what it asks for), the largest
dual certificate of the reference minimisers themselves (how far the references are from optimal), and the uncharged
reference work per run in MatVec passes (ledger category `reference`; part of no cost in this notebook). A solve that ends
below resolution has reached the block minimiser to working precision; its true $\varepsilon_k$ may still exceed
$\rho\Delta_k$ when $\Delta_k$ itself is tiny (a coordinate at $x_{\min}$). `solves` counts accepted outer iterations;
`zero-step` the solves that returned the anchor (block-stationary). FISTA solves no DC subproblem and is not listed.

In [ ]:
def inner_accuracy_table(comparison):
    cfg, rows = comparison["config"], []
    for method in comparison["methods"]:
        if SPEC_BY_NAME[method]["selection"] == "fista":
            continue
        frames = [run["inner"] for (seed, name), run in comparison["runs"].items() if name == method]
        pooled = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        if pooled.empty:
            rows.append({"method": method, "solves": 0})
            continue
        numeric = [column for column in pooled.columns if column != "status"]
        pooled[numeric] = pooled[numeric].astype(float)         # (object dtype can arise from an empty per-run frame)
        statuses = pooled.status.value_counts(normalize=True)
        stepped = pooled[pooled.steps > 0]                      # solves that took a step (the others are exact: error 0)
        reached = pooled[pooled.status == "inner_tolerance"]
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(reached.eps_reference > 0, reached.eps_certified/reached.eps_reference, np.nan)
            tightness = np.where(stepped.eps > 0, stepped.eps_certified/stepped.eps, np.nan)
        rows.append({
            "method": method, "solves": len(pooled),
            "zero-step": f"{(pooled.steps == 0).mean():.0%}",
            "stopped by eps_k": f"{statuses.get('inner_tolerance', 0.0):.1%}",
            "block stationary": f"{statuses.get('block_stationary', 0.0):.1%}",
            "below resolution": f"{statuses.get('step_below_resolution', 0.0)+statuses.get('change_below_resolution', 0.0):.1%}",
            "step cap": f"{statuses.get('budget_used', 0.0)+statuses.get('backtracking_limit', 0.0):.1%}",
            "steps (median)": float(pooled.steps.median()), "steps (max)": int(pooled.steps.max()),
            "certified eps_k (median)": float(stepped.eps_certified.median()),
            "certified eps_k (p90)": float(stepped.eps_certified.quantile(0.9)),
            "certified eps_k (max)": float(stepped.eps_certified.max()),
            "true eps_k (median)": float(stepped.eps.median()), "true eps_k (max)": float(stepped.eps.max()),
            "sum true eps_k per run (median)": float(np.median([frame.eps[frame.steps > 0].sum() for frame in frames])),
            "certified within target or exact": f"{((pooled.eps_certified <= pooled.eps_target) | (pooled.steps == 0)).mean():.1%}",
            "certified eps_k/Delta_k (median, rule stops)": float(np.nanmedian(ratio)) if np.isfinite(ratio).any() else np.nan,
            "certified/true eps_k (median)": float(np.nanmedian(tightness)) if np.isfinite(tightness).any() else np.nan,
            "reference certificate (max)": float(stepped.reference_certificate.max()),
            "reference passes (uncharged, median per run)": float(np.median([run["work"].counts["reference"]/run["work"].nnz
                                                                for (seed, name), run in comparison["runs"].items() if name == method])),
        })
    return pd.DataFrame(rows)


inner_accuracy = inner_accuracy_table(comparison)
print(f"eps_k rule: inner_relative_tolerance={experiment_config['inner_relative_tolerance']:g}, "
      f"inner_absolute_tolerance={experiment_config['inner_absolute_tolerance']:g}, "
      f"max_inner_budget={experiment_config['max_inner_budget']}")
with pd.option_context("display.width", 320, "display.max_columns", 30, "display.float_format", "{:.3g}".format):
    display(inner_accuracy)

### Headline: coordinate blocks and Burg geometry

The four methods that separate the two effects. Entries: median MatVec passes to the target over the seeds that reach it
(seeds reached in brackets). Ratios (first method's cost over the second's) are medians over seeds of the per-seed cost ratio;
when the first method misses the target on a seed, the budget is used as its cost, so the ratio is a lower bound (marked ≥);
seeds on which both methods miss are left out.
A ratio below 1 means the first method is faster; n/a means the second method misses the target on every seed or where the
first one reaches it.

In [ ]:
HEADLINE_TARGETS = [("rel. error <= 20%", "relative_x_error", 0.20, False),
                    ("rel. error <= 5%", "relative_x_error", 0.05, False),
                    ("rel. error <= 2%", "relative_x_error", 0.02, False),
                    ("rel. error <= 1%", "relative_x_error", 0.01, False),
                    ("Burg gap <= 1e-1", "burg_gap", 1e-1, False),
                    ("top-s F1 = 1", "topk_support_f1", 1.0, True)]
headline = headline_table(comparison, HEADLINE_TARGETS)
with pd.option_context("display.width", 250, "display.max_colwidth", 40):
    display(headline)
headline_figure = plot_headline(comparison)
import io
from IPython.display import Image
buffer = io.BytesIO()
headline_figure.savefig(buffer, format="png", dpi=130, bbox_inches="tight")
display(Image(data=buffer.getvalue()))
plt.close(headline_figure)

### Cost to reach common targets

Entry `r/N @ p`: `r` of `N` seeds reached the target, median MatVec passes over those seeds. The measures come from the
uncharged reporting diagnostics (every `log_every_passes`) and the charged stopping checks, so every method is measured on the
same Burg-gap, Euclidean-gap and recovery targets regardless of its own stopping rule.

In [ ]:
TARGETS = [("Burg gap <= 1e-1", "burg_gap", 1e-1, False),
           ("Burg gap <= 1e-2", "burg_gap", 1e-2, False),
           ("Burg gap <= 1e-6", "burg_gap", 1e-6, False),
           ("Euclidean gap <= 1", "euclidean_gap", 1.0, False),
           ("Euclidean gap <= 1e-6", "euclidean_gap", 1e-6, False),
           ("rel. error <= 20%", "relative_x_error", 0.20, False),
           ("rel. error <= 5%", "relative_x_error", 0.05, False),
           ("rel. error <= 2%", "relative_x_error", 0.02, False),
           ("rel. error <= 1%", "relative_x_error", 0.01, False),
           ("top-s F1 = 1", "topk_support_f1", 1.0, True),
           ("precision = 1", "support_precision", 1.0, True)]
target_table = target_cost_table(comparison, TARGETS)
display(target_table)

### Equal-work comparison

Each run is sampled at fixed MatVec levels (last accepted state at or below the level; a run that already stopped keeps its final
state). Gaps are best-so-far values (median over seeds); recovery metrics are means over seeds.

In [ ]:
WORK_LEVELS = [25, 50, 100, 200, 500, 1000, 2000]
equal_work = {}
for label, metric, statistic in (("Burg gap (median, best so far)", "burg_gap", "median"),
                                 ("Euclidean gap (median, best so far)", "euclidean_gap", "median"),
                                 ("relative error (mean)", "relative_x_error", "mean"),
                                 ("top-s support F1 (mean)", "topk_support_f1", "mean"),
                                 ("thresholded support precision (mean)", "support_precision", "mean")):
    equal_work[metric] = equal_work_table(comparison, WORK_LEVELS, metric, statistic)
    print(label)
    with pd.option_context("display.float_format", "{:.3g}".format):
        display(equal_work[metric])

### Rankings

Ranks (1 = best) by the MatVec cost to reach each target: more seeds reached first, then fewer median passes; ties share the
best rank. The final-state and equal-work tables show where each method gets to.

In [ ]:
RANK_TARGETS = [("Burg gap <= 1e-1", "burg_gap", 1e-1, False),
                ("rel. error <= 5%", "relative_x_error", 0.05, False),
                ("rel. error <= 2%", "relative_x_error", 0.02, False),
                ("top-s F1 = 1", "topk_support_f1", 1.0, True)]
rankings = ranking_table(comparison, RANK_TARGETS, comparison["methods"])
with pd.option_context("display.width", 250):
    display(rankings)

### Where the passes go, and two accounting conventions

Median charged passes per ledger category at the end of each run. The last two columns give the median cost to reach
5% relative error under the ledger and under the alternative convention in which coordinate steps form
$s_{\rm rows}+A_{:j}(z-x_j)$ directly, so no offset is needed (which favours the coordinate methods).

In [ ]:
CATEGORIES = ("setup", "selection", "inner", "offset", "cache", "stopping")
breakdown = pd.DataFrame([dict(method=method, seed=seed, **{c: run["work"].counts[c]/run["work"].nnz for c in CATEGORIES})
                          for (seed, method), run in comparison["runs"].items()])
breakdown = breakdown.groupby("method")[list(CATEGORIES)].median().reindex(comparison["methods"])


def convention_costs(comparison, dropped, metric="relative_x_error", threshold=0.05):
    results = comparison["results"].copy()
    results["matvec_passes"] = results["matvec_passes"]-sum((results[c+"_passes"] for c in dropped), 0.0)
    return target_costs(dict(comparison, results=results), metric, threshold).groupby("method").passes.median()


for label, dropped in (("5% error: ledger", ()), ("5% error: no offset", ("offset",))):
    breakdown[label] = convention_costs(comparison, dropped).reindex(comparison["methods"])
with pd.option_context("display.width", 250, "display.float_format", "{:.1f}".format):
    display(breakdown)

In [ ]:
def findings_markdown(comparison, summary, target_table):
    methods = comparison["methods"]
    support_cost = target_costs(comparison, "topk_support_f1", 1.0, larger_is_better=True)

    def fastest(costs):
        stats = costs.groupby("method").agg(reached=("reached", "sum"), passes=("passes", "median"))
        stats = stats.reindex(methods)
        if stats.reached.max() == 0:
            return None, 0, np.nan, stats
        full = stats[stats.reached == stats.reached.max()]
        best = full.passes.idxmin()
        return best, int(stats.loc[best, "reached"]), float(stats.loc[best, "passes"]), stats

    def phrase(best, reached, passes, label):
        seeds = len(comparison["config"]["seeds"])
        return (f"no method reaches {label} within the budget" if best is None else
                f"fastest to {label}: **{best}** ({reached}/{seeds} seeds, median {passes:.0f} passes)")

    lines = []
    for threshold in (1e-1, 1e-2):
        best, reached, passes, stats = fastest(target_costs(comparison, "burg_gap", threshold))
        order = stats.sort_values(["reached", "passes"], ascending=[False, True])
        lines.append(f"* **Burg stationarity:** {phrase(best, reached, passes, f'Burg gap ≤ {threshold:g}')}. Order: " +
                     ", ".join(f"{m} ({int(r.reached)} seeds, {r.passes:.0f})" if r.reached else f"{m} (not reached)"
                               for m, r in order.iterrows()) + ".")
    best, reached, passes, stats = fastest(support_cost)
    finals = summary.groupby("method").topk_support_f1.mean().reindex(methods)
    lines.append(f"* **Support recovery:** final mean top-s F1 = " +
                 ", ".join(f"{m} {v:.2f}" for m, v in finals.items()) +
                 f"; {phrase(best, reached, passes, 'exact top-s support')}.")
    errors = summary.groupby("method").relative_x_error.mean().reindex(methods)
    for threshold in (0.05, 0.01):
        best, reached, passes, stats = fastest(target_costs(comparison, "relative_x_error", threshold))
        lines.append((f"* **Solution recovery:** final mean relative error = " +
                      ", ".join(f"{m} {v:.4f}" for m, v in errors.items()) + "; " if threshold == 0.05 else
                      "* **Solution recovery:** ") + f"{phrase(best, reached, passes, f'{threshold:.0%} error')}.")
    stops = summary.groupby("method").matvec_passes.median().reindex(methods)
    lines.append("* **Cost at the method's own stopping rule (median passes):** " +
                 ", ".join(f"{m} {v:.0f}" for m, v in stops.items()) + ".")
    return "\n".join(lines)

display(Markdown("#### Findings generated from this run\n\n"+findings_markdown(comparison, summary, target_table)))

### Figures

In [ ]:
figures = plot_comparison(comparison)
for figure in figures:
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=130, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close(figure)

## Raw Results and Cost Ledger

Exports include the raw histories (before best-so-far minima), the thinned inner audit with its full online counters, unfinished
candidates, all summary tables, the executed configuration with the validation report, the reconstructions and the MatVec and
vector-operation ledgers. Use `plot_comparison(comparison, axis="wall_clock_time")` for the wall-clock view.

In [ ]:
import json
from pathlib import Path
output_directory = Path.cwd()/"bcdc_block_geometry_results"
output_directory.mkdir(exist_ok=True)
comparison["results"].to_csv(output_directory/"raw_histories.csv", index=False)
comparison["audit"].to_csv(output_directory/"inner_audit_thinned.csv", index=False)
comparison["audit_counts"].to_csv(output_directory/"inner_audit_counts.csv", index=False)
comparison["pending"].to_csv(output_directory/"unfinished_candidates.csv", index=False)
summary.to_csv(output_directory/"final_states.csv", index=False)
method_results.to_csv(output_directory/"method_summary.csv", index=False)
inner_accuracy.to_csv(output_directory/"inner_accuracy.csv", index=False)
pd.concat([run["inner"].assign(seed=seed, method=method) for (seed, method), run in comparison["runs"].items()
           if run.get("inner") is not None], ignore_index=True).to_csv(output_directory/"inner_eps_records.csv", index=False)
rankings.to_csv(output_directory/"rankings.csv", index=False)
breakdown.to_csv(output_directory/"cost_breakdown.csv")
target_table.to_csv(output_directory/"target_costs.csv")
for metric, table in equal_work.items():
    table.to_csv(output_directory/f"equal_work_{metric}.csv")
ledger = []
for (seed, method), run in comparison["runs"].items():
    ledger.append(dict(seed=seed, method=method, **{k+"_contributions": v for k, v in run["work"].counts.items()},
        **{"ops_"+key: value for key, value in run["work"].operations.items()}, **run["work"].events))
pd.DataFrame(ledger).fillna(0).to_csv(output_directory/"cost_ledger.csv", index=False)
metadata = dict(executed_configuration=experiment_config, methods=METHOD_SPECS, validation=validation_report,
                experiment_checks=experiment_checks, experiment_seconds=experiment_seconds)
(output_directory/"configuration_and_checks.json").write_text(json.dumps(metadata, indent=2, default=str))
np.savez_compressed(output_directory/"reconstructions.npz",
    **{f"seed_{seed}_{method}": run["final_x"] for (seed, method), run in comparison["runs"].items()},
    **{f"seed_{seed}_truth": p["x_star"] for seed, p in comparison["problems"].items()})
for index, figure in enumerate([plot_headline(comparison)]+plot_comparison(comparison)):
    figure.savefig(output_directory/f"figure_{index}.png", dpi=150, bbox_inches="tight")
    plt.close(figure)
headline.to_csv(output_directory/"headline.csv")
looseness.to_csv(output_directory/"safe_constant_looseness.csv")
print("Results written to:", output_directory.relative_to(Path.cwd()))

### Optional: robustness over more seeds

Set `RUN_ROBUSTNESS = True` to rerun the whole comparison on thirty fresh seeds (155–184, disjoint from the fifty seeds of the
paper) and print the headline, cost-to-target and ranking tables. The configuration above is not changed; the paper uses the
fifty-seed run only.

In [ ]:
RUN_ROBUSTNESS = False
if RUN_ROBUSTNESS:
    robust_config = dict(copy.deepcopy(CONFIG_PSF), seeds=list(range(155, 185)))
    robust = run_comparison(robust_config, verbose=False, workers=robust_config.get("workers", 1))
    print(assert_comparison(robust))
    display(headline_table(robust, HEADLINE_TARGETS))
    display(target_cost_table(robust, TARGETS))
    with pd.option_context("display.width", 250):
        display(ranking_table(robust, RANK_TARGETS, robust["methods"]))

## Interpretation and Numerical Limits

*(The numbers in this cell are those of the previous run of the notebook and are to be refreshed from the tables above
after the fifty-seed run of the certified $\varepsilon_k$ rule.)*

Numbers are medians over the fifty seeds unless ranges or seed counts are given; the headline and cost-to-target tables have
the costs, the inner-accuracy table the $\varepsilon_k$ values. All seven DCA methods run the same protocol: backtracked inner
steps in their own geometry, and every block subproblem solved to the certified relative accuracy $\rho=0.1$ of the
$\varepsilon_k$ rule.

**1. Coordinate blocks beat the full vector in the Burg geometry.** B-GS(Burg) reaches 20% and 5% relative error after 15
passes on every seed, the exact support (top-$s$ F1 = 1) after 15 and precision 1 after 9, Burg gap $\le0.1$ after 26 and
$\le0.01$ after 31, and its stopping tolerance after 48 (37–48). B-Full reaches 20% error after 27 passes (49/50 seeds), 5%
after 27 (41/50) and the exact support after 24, then stops improving: it completes exactly two outer iterations on every
seed and spends the rest of the budget on the third. Under the $\varepsilon_k$ rule a full-block subproblem is a convex
problem of the full size, as hard as the one FISTA solves, and the second one costs 100–230 inner steps at two passes each
(the inner-accuracy table: median 3 steps for the first solve, up to 228); the unfinished third candidate has 2.5% error, like
the coordinate methods, but is never accepted, so B-Full ends at 3.4% (1.9–21%) against 2.5% for every coordinate method.
Its Burg gap stays at 13 (below 0.1 on one seed, after 700 passes; the tolerance on none), so its stopping cost is the
budget, 2000 passes against 48 for B-GS(Burg): a ratio of at least 42 for the stopping tolerance and 1.9–2.0 for the recovery
targets (lower bounds where B-Full misses the target).

**2. The Burg geometry beats the Euclidean one for both block sizes.** With Gauss–Southwell coordinates, E-GS(PG) needs 26
passes for 20% error (B-GS(Burg): 15), 37 for 5% (15), 53 for Burg gap $\le0.1$ (26) and 81 for its own tolerance (48), and
B-GS(Burg) is cheaper on all fifty seeds; both end at 2.5% error. With full vectors, E-Full completes two outer iterations and
ends at 33% error (29–39%; its unfinished third candidate is at 19%), so B-Full is at least 74 times cheaper on every seed
(a lower bound: E-Full's cost is taken as the budget). The looseness table explains the direction: at the solution the Burg
constant is within 1% of the local Burg curvature at the spikes, so the Burg gap and the Burg Gauss–Southwell score see the
spikes correctly, while the Euclidean picture of the same point is $4\cdot10^6$ times too pessimistic.

**3. Random coordinates are close behind, because the solves are cheap.** B-Random and E-Random reach 20% error after 26
passes, 5% after 31–37, Burg gap $\le0.1$ after 42 and their tolerances after 59, ending at the same 2.5% error. The
Gauss–Southwell methods need 6–8 times fewer outer iterations (B-GS(Burg) 2006, B-Random 15 700, E-Random 17 060) but pay
for the batch: with coordinate solves of one or two steps (median 1 for B-GS(Burg), 2 for E-GS(PG)), scoring 16 columns per
iteration is 31 of B-GS(Burg)'s 48 passes and 51 of E-GS(PG)'s 81, against 7 and 16 passes of inner work. B-GS(Burg) is
cheaper than E-Random on every seed and than B-Random on 32 of 50 (their stopping costs overlap at 48 passes). The
Burg-matched score matters most early: B-GS(Burg) is at 2.6% error after 25 passes when the random methods are at 30% and
B-GS(PG), which orders the same batches by the projected gradient, at 20%.

**4. The subproblems are solved as Algorithm 1 requires, by certificate.** Every coordinate solve ends by the rule
(certified error bound $\le\varepsilon_k$), at a bound with the gradient pointing out (error 0), or at floating-point
resolution (uncertified; the share is in the inner-accuracy table); none hits the step cap. For the certified solves
$Q_k(x^{k+1})-\min Q_k\le\varepsilon_k$ holds outright, so the decrease of every accepted iteration is at least
$0.9\,\Delta_k$ and Theorems 4.1–4.2 apply with $n/0.9$ and no $\sum_k\varepsilon_k$ term; the true errors measured
afterwards against the reporting-only reference minimisers (medians, $\sum_k\varepsilon_k$ per run against
$F(x^0)-F^\star$) are in the inner-accuracy table. The ratio of the certified to the true error says what the certified
rule costs: the derivative bracket is nearly exact for the Burg coordinate solves of B-GS(Burg), looser for the Euclidean
and the random coordinate solves (which therefore take more inner steps than a reference-based rule would), and within a
small factor for the full blocks. No reference solver takes part in any method, so no cost in this notebook depends on one;
the reference work appears only in the uncharged ledger category `reference`.

**5. FISTA**, the convex reference with $\mu=0$, finds the exact support after 12 passes but exhausts the 2000-pass budget at
14% error (10–20%) without reaching its tolerance: the amplitude bias of the convex $\ell_1$ model that the trimmed DC term
removes (every coordinate DCA method ends at the 2.5% noise floor).

**Limits.**

* One instance, fifty seeds. The instance was chosen so that both effects show with Euclidean coordinate methods that make
  real progress; the Poisson noise floor ($\approx0.88\sqrt{s/N}$, 2.5–2.7% here) sets the attainable error, so the 2% target
  is reached on 7 seeds by every converged method and 1% by none.
* The Burg gap is insensitive to a coordinate at $x_{\min}$: its contribution is about $(x_jg_j)^2/(2L_j)$, so a true spike
  started or driven to $x_{\min}$ has a negligible Burg score and Burg gap contribution however negative its gradient. With a
  start of variance 0.1 instead of $0.1^2$, about one pixel per seed is clipped to $x_{\min}$; on seed 129 it is the brightest
  spike, B-GS(Burg) never selects it, declares Burg stationarity at 59% error (Euclidean gap 256) and B-Full cannot finish its
  first subproblem. The interior start used here avoids the issue, but a Burg-gap stopping test should be read with this blind
  spot in mind (the Euclidean gap is reported for every method for that reason).
* The full-block methods are the ones sensitive to the inner protocol. Under the safe-step protocol of Appendix A.4 (global
  constants, $t_k=30+\lfloor k/N\rfloor$ steps per subproblem, Table 2 of the paper, 30 seeds) B-Full finds the exact support
  after 64 passes and reaches 5% error after 126, E-Full finds the support after 66 passes and never reaches 5% error, while
  B-GS(Burg) needs 26 passes for the support and for 5% error (15 and 15 under the protocol of this notebook).
* Costs are MatVec-pass equivalents; vector operations are not charged. In wall-clock time a Python
  coordinate run takes 1.8 s (B-GS(Burg)) to 7 s (random methods) against 0.5–0.6 s for the full-vector methods, which do
  two large products per step; the ranking by wall-clock in this implementation is therefore different.
* $F^\star$ in the tables is the best accepted objective found on the seed, not a certified optimum; the objective gap and the
  Euclidean gap are kept in the tables but not plotted.

In [ ]:
# Two cells to append at the end of the notebook (after the export cell): a markdown cell and a code cell.
# Lines starting with '# %% [markdown]' / '# %%' mark the cell boundaries (Jupytext percent format).

# %% [markdown]
# ## TikZ Figures for the Paper
#
# Every run of the notebook writes the two paper figures as pgfplots sources to `tikz_figures/`: the Burg gap (best so
# far) and the relative error, each against MatVec-pass equivalents on log–log axes,
# median over seeds with the central 68% band (16th–84th percentile), exactly as in the figures above. For every method and
# metric a table `tikz_data/<metric>__<method>.csv` (columns `passes, median, lo, hi`) is written on a geometric grid of
# passes; the gaps are taken best-so-far, only accepted states are used, and exact zeros are drawn at `TIKZ_FLOOR`. The
# style file `bcdc_tikz_styles.tex` encodes the block rule by colour, the geometry by line style (solid Burg, dashed
# Euclidean, dotted FISTA) and draws the two methods of interest thick; its macro `\bcdcmethods{<metric>}` lists exactly the
# methods of this run, so the cell works unchanged for any subset (for example only `B-GS(Burg)` and `B-GS(PG)`). The figures
# are compiled when `pdflatex` is on the path. In the paper, `\input{bcdc_tikz_styles}` in the preamble and copy the `axis`
# environment of a figure file with `width=\columnwidth`.

# %%
import re
import shutil
import subprocess
from pathlib import Path

TIKZ_DIRECTORY = Path.cwd()/"tikz_figures_noref"
TIKZ_GRID_POINTS = 200          # geometric grid of MatVec passes per curve
TIKZ_FLOOR = 1e-12              # exact zeros (e.g. the run that attains F*) are drawn at this value on the log axis
TIKZ_COMPILE = True             # run pdflatex on the figures when it is available
# metric -> (y label, best-so-far?, y minimum shown or None, configuration key of a tolerance line or None)
TIKZ_METRICS = {
    "burg_gap": (r"Burg gap $\mathrm{Gap}_\Psi(x^k)$ (best so far)", True, None, "tol_burg"),
    "relative_x_error": (r"Relative error $\|x^k-x^\star\|_2/\|x^\star\|_2$", False, None, None),
}
TIKZ_COLORS = {"random": "2A78D6", "gs_batch:pg": "EB6834", "gs_batch:burg": "1BAF7A", "full": "EDA100", "fista": "52514E"}


def tikz_slug(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


def tikz_tables(comparison, directory, grid_points=TIKZ_GRID_POINTS, floor=TIKZ_FLOOR):
    """pgfplots tables (passes, median, lo, hi) per method and metric, with the resampling of plot_comparison."""
    data = comparison["results"].query("accepted_state")
    start = max(float(data.groupby(["seed", "method"]).matvec_passes.min().max()), 1e-3)
    grid = np.geomspace(start, max(float(data.matvec_passes.max()), start*1.0001), grid_points)
    directory.mkdir(parents=True, exist_ok=True)
    written = []
    for method in comparison["methods"]:
        group = data[data.method == method]
        for metric, (_, best_so_far, _, _) in TIKZ_METRICS.items():
            curves = []
            for _, run in group.groupby("seed", sort=False):
                points = run[np.isfinite(run[metric])].sort_values("matvec_passes", kind="stable")
                if points.empty:
                    continue
                cost, value = points.matvec_passes.to_numpy(float), points[metric].to_numpy(float)
                if best_so_far:
                    value = np.minimum.accumulate(value)
                index = np.searchsorted(cost, grid, side="right")-1
                row = value[np.maximum(index, 0)]
                row[index < 0] = np.nan                                  # before this seed's first logged state
                curves.append(row)
            if not curves:
                continue
            curves = np.maximum(np.vstack(curves), floor)
            defined = np.isfinite(curves).any(axis=0)
            table = pd.DataFrame(dict(passes=grid[defined], median=np.nanmedian(curves[:, defined], axis=0),
                                      lo=np.nanquantile(curves[:, defined], 0.16, axis=0),
                                      hi=np.nanquantile(curves[:, defined], 0.84, axis=0)))
            path = directory/f"{metric}__{tikz_slug(method)}.csv"
            table.to_csv(path, index=False, float_format="%.6g")
            written.append(path.name)
    return written


def tikz_style_of(spec):
    key = spec["selection"] if spec["selection"] != "gs_batch" else "gs_batch:"+spec["score"]
    dash = "dotted" if spec["selection"] == "fista" else "solid" if spec["geometry"] == "burg" else "densely dashed"
    width = "1.6pt" if spec["name"] in HIGHLIGHT_METHODS else "0.9pt" if spec["selection"] == "fista" else "0.7pt"
    return TIKZ_COLORS[key], f"{dash}, line width={width}"


def tikz_sources(comparison, directory):
    """The style file (with a \\bcdcmethods macro for the methods of this run) and one standalone figure per metric."""
    cfg, methods = comparison["config"], comparison["methods"]
    colours = "\n".join(f"\\definecolor{{bcdc{key}}}{{HTML}}{{{value}}}"
                        for key, value in zip(("Random", "GSPG", "GSBurg", "Full", "Fista"), TIKZ_COLORS.values()))
    colour_name = dict(zip(TIKZ_COLORS.values(), ("bcdcRandom", "bcdcGSPG", "bcdcGSBurg", "bcdcFull", "bcdcFista")))
    entries = []
    for method in methods:
        colour, line = tikz_style_of(SPEC_BY_NAME[method])
        entries.append(f"  \\bcdcmethod{{#1}}{{{tikz_slug(method)}}}{{{colour_name[colour]}}}{{{colour_name[colour]}, {line}}}{{{method}}}")
    columns = min(4, len(methods))
    style = rf"""% Shared pgfplots styles for the BCDC figures, written by the notebook for the methods of this run.
% \input this file in the preamble; set \bcdcdata to the folder with the tables (default: tikz_data).
\usepackage{{pgfplots}}
\usepgfplotslibrary{{fillbetween}}
\pgfplotsset{{compat=1.18}}
\providecommand{{\bcdcdata}}{{tikz_data}}
% colour = block selection rule, line style = inner geometry (solid Burg, dashed Euclidean, dotted FISTA)
{colours}
\pgfplotsset{{
  bcdc axis/.style={{
    width=8.4cm, height=6cm,
    log basis x=10, log basis y=10,
    xlabel={{MatVec-pass equivalents}},
    grid=both, major grid style={{black!10}}, minor grid style={{black!4}},
    axis line style={{black!40}}, tick label style={{font=\scriptsize}}, label style={{font=\small}},
    legend cell align=left,
    legend style={{font=\scriptsize, draw=none, fill=none, at={{(0.5,-0.3)}}, anchor=north, legend columns={columns},
                  /tikz/every even column/.append style={{column sep=4pt}}}},
    every axis plot/.append style={{line join=round, line cap=round}},
  }},
}}
% \bcdcmethod{{metric}}{{slug}}{{colour}}{{line style}}{{legend text}}: 68% band and median line of one method
\newcommand{{\bcdcmethod}}[5]{{%
  \addplot[name path=#2lo, draw=none, forget plot] table[col sep=comma, x=passes, y=lo]{{\bcdcdata/#1__#2.csv}};
  \addplot[name path=#2hi, draw=none, forget plot] table[col sep=comma, x=passes, y=hi]{{\bcdcdata/#1__#2.csv}};
  \addplot[fill=#3, fill opacity=0.12, draw=none, forget plot] fill between[of=#2lo and #2hi];
  \addplot[#4] table[col sep=comma, x=passes, y=median]{{\bcdcdata/#1__#2.csv}};
  \addlegendentry{{#5}}}}
% the methods of this run, in the notebook's order
\newcommand{{\bcdcmethods}}[1]{{%
{chr(10).join(entries)}}}
% horizontal tolerance line across the axis (does not change the limits): \bcdctolerance{{value}}
\newcommand{{\bcdctolerance}}[1]{{%
  \draw[black!55, line width=0.5pt] ({{rel axis cs:0,0}} |- {{axis cs:1,#1}}) -- ({{rel axis cs:1,0}} |- {{axis cs:1,#1}});
  \node[anchor=south west, font=\tiny, text=black!55, inner sep=1pt] at ({{rel axis cs:0,0}} |- {{axis cs:1,#1}}) {{tolerance}};}}
"""
    (directory/"bcdc_tikz_styles.tex").write_text(style)
    files = []
    for metric, (label, _, ymin, tolerance_key) in TIKZ_METRICS.items():
        limits = f", ymin={ymin:g}" if ymin is not None else ""
        tolerance = f"\\bcdctolerance{{{cfg[tolerance_key]:g}}}\n" if tolerance_key else ""
        source = (f"\\documentclass[tikz]{{standalone}}\n\\input{{bcdc_tikz_styles}}\n\\renewcommand{{\\bcdcdata}}{{tikz_data}}\n"
                  f"\\begin{{document}}\n\\begin{{tikzpicture}}\n"
                  f"\\begin{{axis}}[bcdc axis, xmode=log, ymode=log, ylabel={{{label}}}{limits}]\n"
                  f"\\bcdcmethods{{{metric}}}\n{tolerance}\\end{{axis}}\n\\end{{tikzpicture}}\n\\end{{document}}\n")
        path = directory/f"fig_{metric}.tex"
        path.write_text(source)
        files.append(path)
    return files


def tikz_compile(files, directory):
    """Compile with pdflatex when available; return the PDFs that were produced."""
    if shutil.which("pdflatex") is None:
        print("pdflatex not found: the .tex files were written but not compiled.")
        return []
    produced = []
    for path in files:
        result = subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", path.name],
                                cwd=directory, capture_output=True, text=True)
        pdf = path.with_suffix(".pdf")
        if result.returncode == 0 and pdf.exists():
            produced.append(pdf)
        else:
            tail = [line for line in result.stdout.splitlines() if line.startswith("!")][:3]
            print(f"pdflatex failed for {path.name}: {' | '.join(tail) or 'see the .log file'}")
        for extension in (".aux", ".log"):
            path.with_suffix(extension).unlink(missing_ok=True)
    return produced


tikz_written = tikz_tables(comparison, TIKZ_DIRECTORY/"tikz_data")
tikz_files = tikz_sources(comparison, TIKZ_DIRECTORY)
tikz_pdfs = tikz_compile(tikz_files, TIKZ_DIRECTORY) if TIKZ_COMPILE else []
print(f"{len(tikz_written)} pgfplots tables, {len(tikz_files)} TikZ figures"
      f"{f', {len(tikz_pdfs)} compiled PDFs' if tikz_pdfs else ''} in {TIKZ_DIRECTORY.relative_to(Path.cwd())}/")
if tikz_pdfs and shutil.which("pdftoppm"):
    from IPython.display import Image
    for pdf in tikz_pdfs:
        png = subprocess.run(["pdftoppm", "-png", "-r", "110", "-singlefile", pdf.name, pdf.stem], cwd=TIKZ_DIRECTORY)
        if png.returncode == 0:
            display(Image(filename=str(pdf.with_suffix(".png"))))

In [ ]:
def support_table_latex(comparison, path="tikz_figures_noref/support_table.tex"):
    seeds = len(comparison["config"]["seeds"]); rows = []
    final = comparison_summary(comparison)
    for method in comparison["methods"]:
        f1 = target_costs(comparison, "topk_support_f1", 1.0, larger_is_better=True).query("method == @method")
        pr = target_costs(comparison, "support_precision", 1.0, larger_is_better=True).query("method == @method")
        g = final[final.method == method]
        cell = lambda c: f"{c.passes.median():.0f} ({int(c.reached.sum())}/{seeds})" if c.reached.any() else f"-- (0/{seeds})"
        rows.append(f"{method} & {cell(f1)} & {cell(pr)} & {g.topk_support_f1.mean():.3f} & {g.support_precision.mean():.3f} \\\\")
    table = ("\\begin{tabular}{lcccc}\\toprule\nMethod & passes to top-$s$ F1 $=1$ & passes to precision $=1$ & "
             "final top-$s$ F1 & final precision \\\\\\midrule\n" + "\n".join(rows) + "\n\\bottomrule\\end{tabular}\n")
    Path(path).write_text(table); print(table)
support_table_latex(comparison)